# Notebook 03 — Monthly & Annual LST Temporal Analysis

I extract MODIS land surface temperature (LST) for the three zones of every study city, 2006–2025,
and test its long-term trends — per city, per landscape cluster and for the nation. I then set the
trends against what the workshop participants reported.

- **Version** `NB03_v5_SMOD` (needs NB01 `NB01_v5_SMOD`) · **outputs** `Results/03_temporal_analysis/`
- NB02's `Results/02_stakeholder_analysis/Data/workshop_attendees_clean.csv` feeds §10.

---

## Analysis Hierarchy — City → Cluster → National

I analyse at three levels:

```
Level 1 — CITY-REGION  (per-city three-zone extraction)
├─ Urban Core (UC)              SMOD classes 30 + 23 + 22 (high-density urban)
├─ Background Reference Area (BRA) 10 km buffer around UC (background reference)
└─ City-Region (CR)             Full extent — spatial context P5–P95

Level 2 — CLUSTER  (Cluster25 physiographic landscape typology)
├─ Aggregates cities sharing the same landscape character class
└─ Separates humid-coastal from Sahel from plateau signatures

Level 3 — NATIONAL  (Nigerian urban composite)
├─ Unweighted mean across all 56 study cities
└─ Represents the average Nigerian urban thermal signal

Output locations
  City results   →  Results/03_temporal_analysis/{city_slug}/
  Cluster stats  →  Results/03_temporal_analysis/Cluster25_Composites/
  National chart →  Results/03_temporal_analysis/National_Composite/
  Cross-city     →  Results/03_temporal_analysis/heatmaps/
```

<a id="nb03-toc"></a>
## Table of Contents

| Section | Description |
|---------|-------------|
| [§1 — Setup & Imports](#nb03-s01) | Python packages, paths, CRS constants |
| [§2 — GEE Initialisation](#nb03-s02) | Earth Engine authentication |
| [§3 — Load Zone Geometries](#nb03-s03) | UC / BRA / CR from GEE assets or local GeoJSONs |
| [§4 — MODIS LST Extraction](#nb03-s04) | Three-zone daily median LST via GEE; smart cache by quarter |
| [§5 — Monthly & Annual Aggregation](#nb03-s05) | Daily to monthly to annual; validity flags |
| &nbsp;&nbsp; [§5b — City Count Assertion](#nb03-s05b) | Confirms city count matches NB01 cluster lookup |
| **[§6 — City-Level Analysis](#nb03-s06)** | **Statistical pipeline + SUHII + charts + reports** |
| &nbsp;&nbsp; [§6a — Statistical Analysis](#nb03-s06a) | Ljung-Box → MK / Hamed-Rao → Sen slope → OLS R² |
| &nbsp;&nbsp; [§6b — Delta / SUHII](#nb03-s06b) | UC − BRA delta; full independent trend pipeline on SUHII |
| &nbsp;&nbsp; [§6c — Per-City Temporal Charts](#nb03-s06c) | Annual + monthly panels per city (SVG + PDF) |
| &nbsp;&nbsp; [§6d — Per-City CSV Export](#nb03-s06d) | Monthly and annual CSVs per city |
| &nbsp;&nbsp; [§6e — City Summary Reports](#nb03-s06e) | Wide CSV + gradient HTML + per-city Markdown |
| &nbsp;&nbsp; [§6f — City Butterfly Violin](#nb03-s06f) | Spread of each city's annual UC LST, day and night |
| &nbsp;&nbsp; [§6g — City Comparison Table](#nb03-s06g) | Transposed HTML: cities as columns, metrics as rows |
| **[§7 — Cluster-Level Analysis](#nb03-s07)** | **PRIMARY analysis level for publication** |
| &nbsp;&nbsp; [§7a — Cluster Pipeline](#nb03-s07a) | Composite → SUHII → MK/Sen/OLS → chart (per-cluster loop) |
| &nbsp;&nbsp; [§7a2 — Cluster Violin Charts](#nb03-s07a2) | Spread of the cities' UC LST within each cluster |
| &nbsp;&nbsp; [§7a2b — Cluster Butterfly Violins](#nb03-s07a2b) | Day against night, per cluster |
| &nbsp;&nbsp; [§7b — Cluster CSV Export](#nb03-s07b) | Annual, Monthly, SUHII, Stats CSVs |
| &nbsp;&nbsp; [§7c — Cluster Summary Reports](#nb03-s07c) | `Cluster25_LST_Wide_Summary.csv` + gradient HTML |
| &nbsp;&nbsp; [§7d — Publication Composite Grids](#nb03-s07d) | 4×4 grids of the cluster trends, climatology and SUHII |
| **[§8 — National-Level Analysis](#nb03-s08)** | **Unweighted mean across 56 cities** |
| &nbsp;&nbsp; [§8a — Composite Series](#nb03-s08a) | Annual + monthly UC / BRA / CR composite from all 56 cities |
| &nbsp;&nbsp; [§8b — Trend Statistics & SUHII](#nb03-s08b) | Ljung-Box → MK / Hamed-Rao → Sen slope → OLS R²; SUHII pipeline |
| &nbsp;&nbsp; [§8c — National Composite Chart](#nb03-s08c) | Temporal trend chart (same format as §6c) |
| &nbsp;&nbsp; [§8d — Export](#nb03-s08d) | Append to Wide CSV; Markdown report |
| &nbsp;&nbsp; [§8e — National Cluster Violin](#nb03-s08e) | Distribution of city annual LST across all 16 clusters |
| **[§9 — Cross-City Comparison Heatmaps](#nb03-s09)** | **City → Cluster → National hierarchy** |
| &nbsp;&nbsp; [§9a — City-Level Heatmaps](#nb03-s09a) | 56-city standard + 21-row publication (national composite in middle) |
| &nbsp;&nbsp; [§9b — Cluster-Level Heatmaps](#nb03-s09b) | One row per Cluster25 composite + national composite at bottom |
| **[§10 — Stakeholder Validation](#nb03-s10)** | **City → Cluster → National** |
| &nbsp;&nbsp; [§10a — City-Level Validation](#nb03-s10a) | Per-city Q61 / Q58 agreement with the UC trend |
| &nbsp;&nbsp; [§10b — Cluster-Level Validation](#nb03-s10b) | Agreement counted per participant, by cluster |
| &nbsp;&nbsp; [§10c — Comprehensive Validation Table](#nb03-s10c) | City, cluster and national rows in one table |
| &nbsp;&nbsp; [§10d — Perception Panels](#nb03-s10d) | Warming, diurnal and peak-season answers as bar and donut panels |
| **[§11 — Composite Summary Tables](#nb03-s11)** | **Hottest periods, SUHII intensity, master cross-notebook table** |
| &nbsp;&nbsp; [§11a — Hottest Periods](#nb03-s11) | Per city + cluster: years/months above 90th-pct UC LST |
| &nbsp;&nbsp; [§11b — SUHII Intensity](#nb03-s11b) | Per city + cluster: SUHII max / mean / min / std + trend |
| &nbsp;&nbsp; [§11c — Master Composite Table](#nb03-s11c) | Grouped by Cluster25; single-city clusters merged; national at end |
| [§12 — Summary & Next Steps](#nb03-s12) | Full outputs tree; statistical method quick reference |

---
## Data Sources & Scale

| Source | Native resolution | Temporal resolution | Role |
|--------|-----------------|--------------------|----|
| MODIS MYD11A1 (Aqua) | 1 km | Daily (~13:30 day / ~01:30 night local) | All trend analysis |

---

## Three-Zone Architecture

I run every extraction and test on three parallel series per city and period (day, night):

| Zone | Definition | Role |
|------|-----------|------|
| Urban Core (UC) | SMOD classes 30 + 23 + 22, water-free | Primary urban thermal signal |
| Background Reference Area (BRA) | 10 km buffer − UC − water − peri-urban | Background reference baseline |
| City-Region (CR) | Full BRA bounding box − water | Spatial context envelope (P5–P95 band) |
I load the zones from the per-city Earth Engine assets of NB01 §10, or from the local GeoJSONs when
an asset is missing.

---

## Statistical Protocol — Role of Each Method

Each method answers its own question and none replaces another: **Mann–Kendall** tells whether a
significant monotonic trend exists and its direction; **Sen's slope** tells how fast LST changes
(°C yr⁻¹ or °C decade⁻¹); **OLS R²** tells how well a straight line fits.

| Step | Method | Output used | Purpose |
|------|--------|------------|---------|
| 1 | **Ljung-Box test** | Selects MK variant | Pre-test for autocorrelation — not a trend test |
| 2a | **Standard Mann-Kendall (MK)** | p-value + direction ↑↓ | Monotonic trend test — no autocorrelation |
| 2b | **Hamed-Rao corrected MK** | p-value [H&R] + direction ↑↓ | MK with adjusted variance for autocorrelated series |
| 2c | **Seasonal Mann-Kendall** | p-value + direction ↑↓ | Trend test for monthly series; handles seasonality |
| 3 | **Sen's slope** (Theil-Sen) | °C yr⁻¹ and °C decade⁻¹ | Robust rate of change — median of all pairwise slopes |
| 4 | **OLS regression** | R² only (no OLS slope) | Linear fit quality; OLS slope is NOT the reported rate |

---

## Chart Design (per-city and national composite)

Each chart has an annual trend panel (left, 2006–2025) and a monthly climatology panel (right), day
and night on the same axes, SUHII (UC − BRA) on its own right-hand axis, the day statistics top-left
and the night statistics bottom-left, Sen's slope as a thin dashed line, the city-region P5–P95 range
as a shaded band, and the peak and trough marked.

---
<a id="nb03-s01"></a>
## Section 1 — Setup & Imports

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
import sys, os, json, warnings, hashlib, math
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

# ── Project paths ─────────────────────────────────────────────────────────────
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

import config
from gee_utils import (initialize_gee, extract_three_zone_lst)
from temporal_analysis import run_trend_analysis
from visualization import (create_temporal_chart,
                            create_single_page_heatmap,
                            create_validation_sunburst_charts,
                            create_perception_chart_with_donuts,
                            create_cluster_perception_chart,
                            create_national_perception_donuts)

# GEE
import ee
from geojson import mapping

# ── Output paths ──────────────────────────────────────────────────────────────
RESULTS_03   = project_root / 'Results' / '03_temporal_analysis'
RESULTS_SUPP = project_root / 'Results' / 'supplementary'
RESULTS_03.mkdir(parents=True, exist_ok=True)
RESULTS_SUPP.mkdir(parents=True, exist_ok=True)

RESULTS_01     = project_root / 'Results' / '01_aoi_delineation'
SOURCE_VERSION = 'NB01_v5_SMOD'
STORAGE_CRS    = 'EPSG:4326'

# ── TEST MODE CONTROL ─────────────────────────────────────────────────────────
# Set TEST_MODE = True to run Lagos only (for debugging / first-run validation).
# Set TEST_MODE = False to process all cities with population > 300 k.
TEST_MODE = False

from aoi_utils import load_aoi_datasets

# Load cities via the same normalised loader as NB01
# (City_Name -> Urban Agglomeration renaming handled internally)
_cities_df, _, _ = load_aoi_datasets(
    str(project_root / config.AOI_DATA_PATHS['cities_csv']),
    str(project_root / config.AOI_DATA_PATHS['urban_geojson']),
    str(project_root / config.AOI_DATA_PATHS['boundary_geojson']),
)
_pop_yr = str(config.PROJECT_STUDY_YEAR)
_cities_df['_pop'] = pd.to_numeric(
    _cities_df[_pop_yr].astype(str).str.replace(',', ''), errors='coerce')

if TEST_MODE:
    CITIES_TO_PROCESS = ['Aba', 'Lagos']
    print('MODE : TEST (Aba and Lagos only)')
else:
    CITIES_TO_PROCESS = _cities_df[_cities_df['_pop'] > 300]['Urban Agglomeration'].tolist()
    print(f'MODE : FULL ({len(CITIES_TO_PROCESS)} cities > 300 k)')

print(f'Source version : {SOURCE_VERSION}')
print(f'Output dir     : {RESULTS_03}')

# ── Skip toggles (from config.SKIP_EXISTING) ──────────────────────────────────
# IMPORTANT: When TEST_MODE = True, all skips are disabled to run full Lagos test
# When TEST_MODE = False, respects config.SKIP_EXISTING for efficient re-runs
SKIP_EXISTING = config.SKIP_EXISTING.copy()

if TEST_MODE:
    print("\n TEST_MODE = True: Disabling ALL skip toggles for full Lagos test\n")
    for key in SKIP_EXISTING:
        SKIP_EXISTING[key] = False
else:
    print("\n FULL MODE: Using skip toggles from config.SKIP_EXISTING\n")

print('Skip toggles status:')
for _k, _v in SKIP_EXISTING.items():
    status = "SKIP" if _v else "RUN"
    print(f'  {_k:32s}: {status}')

---
<a id="nb03-s02"></a>
## Section 2 — GEE Initialisation

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── GEE initialisation ─────────────────────────────────────────────────────────────────
GEE_AVAILABLE = False
try:
    initialize_gee()
    GEE_AVAILABLE = True
    print('GEE : initialised')
except Exception as e:
    print(f'GEE : NOT available ({e})')

# ── Thread pool size for parallel GEE extraction ────────────────────────
# Threads share the main-process memory so ee.Geometry objects pass directly.
# GEE Python client is thread-safe (gRPC). Keep ≤8 to respect GEE rate limits.
N_WORKERS = min(config.PROCESSING_OPTIONS.get('dask_chunksize', 4), 8)
print(f'GEE extraction will use {N_WORKERS} parallel threads')


---
<a id="nb03-s03"></a>
## Section 3 — Load Zone Geometries

I load each city's UC, BRA and city-region geometry from the Earth Engine assets of NB01 §10, or
from the local `_v5` GeoJSONs when an asset is missing, and hash each city's centroid for the cache.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
def _geom_hash(geom_wkt):
    return hashlib.sha256(geom_wkt.encode()).hexdigest()[:16]

collection_base = config.GEE_CONFIG.get('collection_name', 'Nigeria_AOI')
project_id      = config.GEE_CONFIG['project_id']

zone_geoms  = {}
load_errors = []

_meta_path = RESULTS_01 / 'all_cities_zone_metadata.json'
if _meta_path.exists():
    _all_meta  = json.loads(_meta_path.read_text(encoding='utf-8'))
    _all_cities = (list(_all_meta.keys()) if isinstance(_all_meta, dict)
                   else [m['city_name'] for m in _all_meta])
else:
    _all_cities = [d.name.replace('_', ' ')
                   for d in RESULTS_01.iterdir()
                   if d.is_dir() and not d.name.startswith('.')]

# Filter to CITIES_TO_PROCESS (TEST_MODE or > 300 k)
cities_to_load = [c for c in _all_cities if c in CITIES_TO_PROCESS]
print(f'Cities in metadata : {len(_all_cities)}')
print(f'Cities to process  : {len(cities_to_load)}  '
      f'({"TEST" if TEST_MODE else "FULL"} mode)')

for city in tqdm(sorted(cities_to_load), desc='Loading zones'):
    city_slug = city.replace(' ', '_')
    city_dir  = RESULTS_01 / city_slug

    uc_ee = bra_ee = cr_ee = None
    if GEE_AVAILABLE:
        try:
            asset_path = (f'projects/{project_id}/assets/'
                          f'{collection_base}/{city_slug}')
            fc     = ee.FeatureCollection(asset_path)
            uc_ee  = fc.filter(ee.Filter.eq('zone', 'UrbanCore')).first().geometry()
            bra_ee = fc.filter(ee.Filter.eq('zone', 'BRA')).first().geometry()
            cr_ee  = fc.filter(ee.Filter.eq('zone', 'CityRegion')).first().geometry()
        except Exception:
            uc_ee = bra_ee = cr_ee = None

    uc_gdf = bra_gdf = cr_gdf = None
    for zone, label in [('urban_core','uc'), ('bra','bra'), ('city_region','cr')]:
        fpath = city_dir / f'{city_slug}_{zone}_4326_v5.geojson'
        if fpath.exists():
            try:
                gdf = gpd.read_file(fpath)
                # Fix invalid geometries: buffer(0) + simplify for robustness
                gdf['geometry'] = gdf.geometry.buffer(0)
                invalid_mask = ~gdf.geometry.is_valid
                if invalid_mask.any():
                    gdf.loc[invalid_mask, 'geometry'] = gdf.loc[invalid_mask, 'geometry'].simplify(tolerance=1e-6)
                if label == 'uc':  uc_gdf  = gdf
                if label == 'bra': bra_gdf = gdf
                if label == 'cr':  cr_gdf  = gdf
            except Exception as e:
                print(f'    ⚠️  Error reading {zone} for {city}: {e}')
                continue

    if uc_ee is None and uc_gdf is not None and bra_gdf is not None and cr_gdf is not None and GEE_AVAILABLE:
        try:
            uc_ee  = ee.Geometry(mapping(uc_gdf.geometry.unary_union))
            bra_ee = ee.Geometry(mapping(bra_gdf.geometry.unary_union))
            cr_ee  = ee.Geometry(mapping(cr_gdf.geometry.unary_union))
        except Exception as e:
            print(f'    ⚠️  EE geometry conversion failed for {city}: {str(e)[:80]}...')
            uc_ee = bra_ee = cr_ee = None

    if uc_gdf is None:
        load_errors.append(city)
        continue

    cr_centroid_wkt = (cr_gdf.geometry.unary_union.centroid.wkt
                       if cr_gdf is not None else '')
    zone_geoms[city] = {
        'uc_ee': uc_ee, 'bra_ee': bra_ee, 'cr_ee': cr_ee,
        'uc_gdf': uc_gdf, 'bra_gdf': bra_gdf, 'cr_gdf': cr_gdf,
        'city_slug': city.replace(' ', '_'),
        'cache_hash': _geom_hash(cr_centroid_wkt),
    }

print(f'Zones loaded  : {len(zone_geoms)}')
if load_errors:
    print(f'Load errors   : {load_errors}')

---
<a id="nb03-s04"></a>
## Section 4 — MODIS LST Extraction (Three Zones)

I extract the daily median LST of the UC, BRA and city region of every city with server-side
Earth Engine `reduceRegion` calls, cached by quarter.

- **Data:** MODIS MYD11A1 (Aqua, ~13:30 day and ~01:30 night local time), with the strict and
  moderate QA levels of `config.LST_CONFIG`; zones from NB01 §10 (or the local GeoJSONs).
- **Cache:** I skip a city-quarter whose CSV exists and whose source version and centroid hash match.
  Deleting a city's `*_monthly_lst_threezone.csv`, or changing its geometry in NB01, forces a new
  extraction.
- **Without Earth Engine** (`GEE_AVAILABLE = False`) I use the CSVs already on disk; a city with none
  drops out of the analysis, and §5 warns about it.

**Output:** `{city_slug}_monthly_lst_threezone.csv` per city.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# extract_three_zone_lst imported from src/gee_utils.py
# Now extracts monthly composites (one record per year-month) instead of daily,
# reducing GEE compute and transfer by ~32x.

print(f'Extracting LST for {len(zone_geoms)} cities '
      f'({N_WORKERS} parallel threads)...')

all_monthly    = {}
extract_errors = []

def _extract_city(city_zones_tuple):
    """Worker function: extract one city. Threads share process so ee objects are live."""
    city, zones = city_zones_tuple
    try:
        result = extract_three_zone_lst(
            city_name      = city,
            zones          = zones,
            years          = config.TEMPORAL_CONFIG['years'],
            output_dir     = RESULTS_03,
            source_version = SOURCE_VERSION,
        )
        return city, result, None
    except Exception as e:
        return city, None, str(e)

with ThreadPoolExecutor(max_workers=N_WORKERS) as _pool:
    _futures = {
        _pool.submit(_extract_city, (city, zones)): city
        for city, zones in zone_geoms.items()
    }
    for _fut in tqdm(as_completed(_futures), total=len(_futures),
                     desc='GEE extraction'):
        _city, _result, _err = _fut.result()
        if _err:
            print(f'  ERROR {_city}: {_err}')
            extract_errors.append(_city)
        elif _result is not None:
            all_monthly[_city] = _result
        else:
            print(f'  [SKIP] {_city} — extraction returned no data')
            extract_errors.append(_city)

print(f'Extraction complete : {len(all_monthly)} cities')
if extract_errors:
    print(f'Errors / skipped    : {extract_errors}')


---
<a id="nb03-s05"></a>
## Section 5 — Monthly & Annual Aggregation

I aggregate the daily values to months (city × zone × period × year × month) and years
(city × zone × period × year), keep the counts of valid observations, and compute the city-region
P5–P95 range.

---

> ↑ [Back to Table of Contents](#nb03-toc)
#### Data Quality Flags Explained

I flag thin data but keep it, so a reader can decide whether to drop it:
- **a month** is `low_coverage` when it has fewer than `MIN_VALID_IMAGES` (5) usable images —
  usually cloud, orbit gaps or QA masking;
- **a year** is `low_coverage` when fewer than `MIN_VALID_MONTHS` (5) of its months pass that test,
  however good those months are.

In [ ]:
# ── Section 5: Aggregate monthly GEE composites to analysis DataFrames ──────
# GEE returns one record per year-month carrying:
#   • Spatial LST stats per zone×period  (mean, median, min, max, std, p05/p95)
#   • Image-level QA counts per period   (n_days/nights, n_strict, n_moderate)
#
# Monthly records: one row per city × zone × period × year-month.
# Annual records : aggregate 12 monthly values → median/mean/min/max/std,
#                  hottest/coolest month, and summed image-count columns.

MIN_VALID_MONTHS = 5   # flag year as low-coverage if < 5 months have LST data
MIN_VALID_IMAGES = 5   # flag month as low-coverage if < 5 usable images

monthly_records = []
annual_records  = []

_STAT_COLS = ['mean', 'median', 'min', 'max', 'std']

# Image-count columns are NOT zone-specific; they differ by period only.
# Map each period to its column names in the raw monthly CSV.
_IMG_COLS = {
    'day'  : ('n_days',   'n_day_strict',   'n_day_moderate'),
    'night': ('n_nights', 'n_night_strict', 'n_night_moderate'),
}

for city, df_mon_raw in tqdm(all_monthly.items(), desc='Aggregating'):
    city_slug = zone_geoms[city]['city_slug']

    for zone_prefix, zone_label, has_envelope in [
            ('uc',  'urban_core',  False),
            ('bra', 'bra',         False),
            ('cr',  'city_region', True),
    ]:
        for period in ['day', 'night']:
            pfx = f'{zone_prefix}_{period}'     # e.g. 'uc_day'
            col_med = f'{pfx}_median'
            col_n   = f'{pfx}_n'
            col_p05 = f'{pfx}_p05' if has_envelope else None
            col_p95 = f'{pfx}_p95' if has_envelope else None

            if col_med not in df_mon_raw.columns:
                continue

            # Period-specific image-count column names
            col_obs, col_strict, col_moderate = _IMG_COLS[period]

            extra_cols = [f'{pfx}_{s}' for s in _STAT_COLS
                          if f'{pfx}_{s}' in df_mon_raw.columns]
            env_cols   = ([col_p05, col_p95]
                          if has_envelope and col_p05 in df_mon_raw.columns else [])
            img_cols   = [c for c in ['n_images', col_obs, col_strict, col_moderate]
                          if c in df_mon_raw.columns]
            keep_cols  = ['year', 'month', col_n] + extra_cols + env_cols + img_cols
            sub = df_mon_raw[[c for c in keep_cols
                               if c in df_mon_raw.columns]].copy()
            sub = sub.dropna(subset=[col_med])

            # ── Monthly records ───────────────────────────────────────────────
            for _, row in sub.iterrows():
                n_obs      = int(row.get(col_obs,      0) or 0)
                n_strict   = int(row.get(col_strict,   0) or 0)
                n_moderate = int(row.get(col_moderate, 0) or 0)
                n_images   = int(row.get('n_images',   0) or 0)
                rec = {
                    'city'         : city,
                    'city_slug'    : city_slug,
                    'zone'         : zone_label,
                    'period'       : period,
                    'year'         : int(row['year']),
                    'month'        : int(row['month']),
                    # LST statistics
                    'median_lst'   : round(float(row[col_med]), 4),
                    'mean_lst'     : round(float(row.get(f'{pfx}_mean',  row[col_med])), 4),
                    'min_lst'      : round(float(row.get(f'{pfx}_min',   row[col_med])), 4),
                    'max_lst'      : round(float(row.get(f'{pfx}_max',   row[col_med])), 4),
                    'std_lst'      : round(float(row.get(f'{pfx}_std',   0.0) or 0.0), 4),
                    # Image-count coverage (period-specific)
                    'n_images'     : n_images,
                    'n_obs'        : n_obs,        # total usable (strict + moderate)
                    'n_strict'     : n_strict,     # preferred quality
                    'n_moderate'   : n_moderate,   # supplement only
                    'pct_strict'   : round(n_strict / n_obs, 4) if n_obs > 0 else 0.0,
                    'pct_valid_obs': round(n_obs / n_images, 4) if n_images > 0 else 0.0,
                    'low_coverage' : n_obs < MIN_VALID_IMAGES,
                    # Per-pixel valid-observation count (spatial mean across zone pixels)
                    'n_valid_pixels': int(row.get(col_n, 0) or 0),
                }
                if has_envelope and col_p05 in df_mon_raw.columns:
                    rec['p05_lst'] = round(float(row.get(col_p05, row[col_med])), 4)
                    rec['p95_lst'] = round(float(row.get(col_p95, row[col_med])), 4)
                else:
                    rec['p05_lst'] = rec['mean_lst']
                    rec['p95_lst'] = rec['mean_lst']
                monthly_records.append(rec)

            # ── Annual records ────────────────────────────────────────────────
            for yr, grp in sub.groupby('year'):
                med_vals  = grp[col_med].dropna().values
                mean_vals = grp.get(f'{pfx}_mean',  grp[col_med]).dropna().values
                min_vals  = grp.get(f'{pfx}_min',   grp[col_med]).dropna().values
                max_vals  = grp.get(f'{pfx}_max',   grp[col_med]).dropna().values
                std_vals  = grp.get(f'{pfx}_std',   pd.Series(dtype=float)).dropna().values
                n_valid_months = len(med_vals)
                if n_valid_months == 0:
                    continue

                # Summed image counts across all months of the year
                n_obs_total      = int(grp[col_obs].sum())      if col_obs      in grp.columns else 0
                n_strict_total   = int(grp[col_strict].sum())   if col_strict   in grp.columns else 0
                n_moderate_total = int(grp[col_moderate].sum()) if col_moderate in grp.columns else 0
                n_images_total   = int(grp['n_images'].sum())   if 'n_images'   in grp.columns else 0

                # Hottest / coolest month identification
                months_arr = grp.loc[grp[col_med].notna(), 'month'].values
                hot_idx    = int(np.argmax(mean_vals)) if len(mean_vals) else 0
                cool_idx   = int(np.argmin(mean_vals)) if len(mean_vals) else 0

                annual_records.append({
                    'city'             : city,
                    'city_slug'        : city_slug,
                    'zone'             : zone_label,
                    'period'           : period,
                    'year'             : yr,
                    # LST statistics (aggregated from 12 monthly values)
                    'median_lst'       : round(float(np.median(med_vals)),          4),
                    'mean_lst'         : round(float(np.mean(mean_vals)),           4),
                    'min_lst'          : round(float(np.min(min_vals)),             4),
                    'max_lst'          : round(float(np.max(max_vals)),             4),
                    'std_lst'          : round(float(np.std(med_vals)),             4),
                    'p05_lst'          : round(float(np.percentile(med_vals,  5)),  4),
                    'p95_lst'          : round(float(np.percentile(med_vals, 95)),  4),
                    'mean_spatial_std' : round(float(np.mean(std_vals)) if len(std_vals) else 0.0, 4),
                    # Image-count coverage (summed across 12 months)
                    'n_images_total'   : n_images_total,
                    'n_obs_total'      : n_obs_total,
                    'n_strict_total'   : n_strict_total,
                    'n_moderate_total' : n_moderate_total,
                    'pct_strict'       : round(n_strict_total / n_obs_total, 4) if n_obs_total > 0 else 0.0,
                    'pct_valid_obs'    : round(n_obs_total / n_images_total, 4) if n_images_total > 0 else 0.0,
                    # Month-level coverage
                    'n_valid_months'   : n_valid_months,
                    'pct_valid_months' : round(n_valid_months / 12, 4),
                    'low_coverage'     : n_valid_months < MIN_VALID_MONTHS,
                    # Temporal peak identification
                    'hottest_month'    : int(months_arr[hot_idx])  if len(months_arr) > hot_idx  else None,
                    'coolest_month'    : int(months_arr[cool_idx]) if len(months_arr) > cool_idx else None,
                })

df_monthly = pd.DataFrame(monthly_records)
df_annual  = pd.DataFrame(annual_records)

print(f'Monthly records : {len(df_monthly):,}')
print(f'Annual records  : {len(df_annual):,}')
print(f'Cities covered  : {df_annual["city"].nunique() if len(df_annual) else 0}')

# Sanity: hottest month distribution for UC day
if len(df_annual):
    uc_day_hot = (df_annual[(df_annual.zone == 'urban_core') & (df_annual.period == 'day')]
                  .groupby('hottest_month').size())
    print('\nHottest month distribution (UC day across all city-years):')
    print(uc_day_hot.to_string())
    # QA quality overview
    print(f'\nMean pct_strict (UC day): {df_annual[(df_annual.zone=="urban_core") & (df_annual.period=="day")]["pct_strict"].mean():.1%}')


---
<a id="nb03-s05b"></a>
### §5b — City Count Assertion

I check that the cities loaded match NB01's cluster lookup: 56 study cities, all of at least 300,000
inhabitants.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §5b — City count assertion ────────────────────────────────────────────────
# Verifies that df_annual / df_monthly contain only the target study cities
# produced by NB01.  Fails loudly if the counts diverge.

_n_ann_cities = df_annual['city'].nunique()
_n_mon_cities = df_monthly['city'].nunique()

_cl_lookup_path = RESULTS_03.parent / '01_aoi_delineation' / 'city_cluster25_lookup.csv'
if _cl_lookup_path.exists():
    _cl_check = pd.read_csv(_cl_lookup_path)
    _expected_n = len(_cl_check)
    print(f'Cities in annual data    : {_n_ann_cities}')
    print(f'Cities in monthly data   : {_n_mon_cities}')
    print(f'Expected (cluster lookup): {_expected_n}  (NB01 study cities)')
    assert _n_ann_cities <= _expected_n, (
        f'Annual data has MORE cities ({_n_ann_cities}) than the cluster lookup ({_expected_n}). '
        'Check CITY_SELECTION_MODE and re-run NB01.'
    )
    if _n_ann_cities < _expected_n:
        _missing = set(_cl_check['city_name']) - set(df_annual['city'].unique())
        print(f'  ⚠  {_expected_n - _n_ann_cities} cities in lookup but not yet in extracted data:')
        for _m in sorted(_missing):
            print(f'       {_m}')
    else:
        print('  ✓  City count matches cluster lookup — filter OK')
else:
    print(f'  ⚠  Cluster lookup not found at {_cl_lookup_path}')
    print(f'     Cannot verify city count. Cities in annual data: {_n_ann_cities}')

print(f'\nConfig selection mode: {config.CITY_SELECTION_MODE}')
print(f'Years in annual data : {sorted(df_annual["year"].unique())[:3]} ... '
      f'{sorted(df_annual["year"].unique())[-2:]}  '
      f'({df_annual["year"].nunique()} years total)')

---
<a id="nb03-s06"></a>
## Section 6 — City-Level Analysis

I analyse each city on its own:

| Sub-section | Content |
|-------------|---------|
| [§6a — Statistical Analysis](#nb03-s06a) | Ljung-Box → MK / Hamed-Rao → Sen's slope → OLS R² |
| [§6b — Delta / SUHII](#nb03-s06b) | UC − BRA year-wise delta; full independent trend pipeline |
| [§6c — Per-City Temporal Charts](#nb03-s06c) | Annual + monthly panels per city (SVG + PDF) |
| [§6d — Per-City CSV Export](#nb03-s06d) | Monthly and annual CSVs per city |
| [§6e — City Summary Reports](#nb03-s06e) | Wide CSV + gradient HTML + per-city Markdown |
| [§6f — City Butterfly Violin](#nb03-s06f) | Spread of each city's annual UC LST, day and night |
| [§6g — City Comparison Table](#nb03-s06g) | Transposed HTML table: cities as columns |

---

> ↑ [Back to Table of Contents](#nb03-toc)

---
<a id="nb03-s06a"></a>
### §6a — Statistical Analysis

I apply four methods in a fixed order; each answers a different question (see the protocol table
at the top).

### Layer 1 — Ljung-Box autocorrelation pre-test

$$LB = n(n+2)\sum_{k=1}^{h}\frac{\hat{\rho}_k^2}{n-k} \sim \chi^2(h), \quad H_0: \text{residuals are i.i.d.}$$
with $n$ years, $h = \min(10, n/5)$ lags and $\hat{\rho}_k$ the autocorrelation at lag $k$. I apply
it to the detrended annual series. When $p < 0.05$ the series is autocorrelated and I use the
Hamed–Rao correction in the Mann–Kendall test. It only chooses the test; it says nothing about a trend.

### Layer 2 — Trend significance test (variant auto-selected per series)

| Condition | Test applied | Reference |
|-----------|-------------|-----------|
| Annual + LB p ≥ 0.05 (no autocorrelation) | Standard Mann-Kendall | Mann (1945); Kendall (1975) |
| Annual + LB p < 0.05 (autocorrelation detected) | Hamed-Rao variance-corrected MK | Hamed & Rao (1998) |
| Monthly (always, regardless of LB result) | Seasonal Mann-Kendall ± H&R | Hirsch et al. (1982) |
$$S = \sum_{i=1}^{n-1} \sum_{j=i+1}^{n} \text{sgn}(x_j - x_i)$$
summing the signs of all later-minus-earlier differences. I report the direction (`increasing`,
`decreasing`, `no trend`) and the p-value, with `*` p < 0.05, `**` p < 0.01, `***` p < 0.001, and
`[H&R]` where the Hamed–Rao correction applied. Mann–Kendall gives no rate.

### Layer 3 — Sen's slope (Theil-Sen median slope estimator)

$$\hat{\beta} = \mathrm{median}\left\{\frac{x_j - x_k}{j - k} : j > k\right\}$$
the median of all pairwise slopes. I report it per year (`sen_slope_per_year`) and per decade
(`sen_slope_per_decade`). It is my warming rate throughout: as a median, it resists outliers and
assumes no normal residuals, whereas a single extreme year can pull an OLS slope.

### Layer 4 — OLS regression (fit quality only)

$$R^2 = 1 - \frac{\sum_{t=1}^{n} (x_t - \hat{x}_t)^2}{\sum_{t=1}^{n} (x_t - \bar{x})^2}$$
I fit a straight line to each annual series and report only its R², the share of year-to-year
variance a linear trend explains; I never use the OLS slope as a rate.

### Monthly amplitude

$$A = \bar{x}_{\text{hottest month}} - \bar{x}_{\text{coolest month}}$$
the difference between the multi-year mean LST of the hottest and the coolest calendar month.

### Common Misinterpretations ⚠️

| ❌ Wrong reading | ✅ Correct reading |
|-----------------|-------------------|
| "MK p < 0.05 means the warming rate is large" | MK p < 0.05 means a monotonic trend **exists**; magnitude comes from Sen's slope only. The trend could be +0.01 °C/yr or +1 °C/yr — MK does not say. |
| "OLS slope is the warming rate because R² = 0.85" | OLS R² = 0.85 describes **fit quality** only. Sen's slope is the rate (robust to outliers). OLS slope is not reported. |
| "MK p > 0.05 means no climate signal" | The trend is **not statistically significant** given the noise level. The city may still be warming — just not detectably over 20 years with current inter-annual variability. |
| "Sen slope + MK together give the warming trend" | Sen slope gives **rate**; MK gives **significance**. Neither gives the other. Report both together. |

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# run_trend_analysis() imported from src/temporal_analysis.py

# ── Annual statistics ─────────────────────────────────────────────────────────
annual_stats_rows = []
for (city, zone, period), grp in tqdm(
        df_annual.groupby(['city', 'zone', 'period']),
        desc='Annual stats'):
    grp_s = grp.sort_values('year').set_index('year')['median_lst']
    res   = run_trend_analysis(grp_s, mode='annual')
    hi    = grp_s.idxmax() if not grp_s.dropna().empty else None
    lo    = grp_s.idxmin() if not grp_s.dropna().empty else None
    annual_stats_rows.append({
        'city': city, 'zone': zone, 'period': period,
        'highest_year':     hi,
        'highest_year_lst': round(float(grp_s.max()), 4) if hi else None,
        'lowest_year':      lo,
        'lowest_year_lst':  round(float(grp_s.min()), 4) if lo else None,
        **res
    })

df_annual_stats = pd.DataFrame(annual_stats_rows)

# ── Monthly statistics ────────────────────────────────────────────────────────
monthly_stats_rows = []
for (city, zone, period), grp in tqdm(
        df_monthly.groupby(['city', 'zone', 'period']),
        desc='Monthly stats'):
    ordered = grp.sort_values(['year', 'month'])
    idx     = pd.to_datetime(
        ordered.apply(lambda r: f'{int(r["year"])}-{int(r["month"]):02d}', axis=1))
    grp_s   = ordered.set_index(idx)['median_lst']
    res     = run_trend_analysis(grp_s, mode='monthly')

    clim    = grp.groupby('month')['median_lst'].mean()
    amp     = round(float(clim.max() - clim.min()), 4) if len(clim) == 12 else None
    hi_mo   = int(clim.idxmax()) if not clim.empty else None
    lo_mo   = int(clim.idxmin()) if not clim.empty else None

    monthly_stats_rows.append({
        'city': city, 'zone': zone, 'period': period,
        'monthly_amplitude':   amp,
        'highest_month':       hi_mo,
        'highest_month_lst':   round(float(clim.max()), 4) if hi_mo else None,
        'lowest_month':        lo_mo,
        'lowest_month_lst':    round(float(clim.min()), 4) if lo_mo else None,
        **res
    })

df_monthly_stats = pd.DataFrame(monthly_stats_rows)
print(f'Annual stats rows  : {len(df_annual_stats)}')
print(f'Monthly stats rows : {len(df_monthly_stats)}')


---
<a id="nb03-s06b"></a>
### §6b — Delta (UC − BRA) Computation and SUHII

I compute the difference between the urban-core and the BRA median LST, year by year and month by
month:

$$\Delta_{LST} = LST_{UC} - LST_{BRA}$$

This is the surface urban heat island intensity (SUHII). Subtracting the background removes the
shared regional climate and isolates the city's own thermal excess: positive values mean the core is
warmer than its surroundings, negative values cooler. I run the whole §6a pipeline directly on this
series, so the SUHII trend shows how the contrast itself evolves — something the separate UC and BRA
trends cannot show.

| SUHII value | Interpretation |
|-------------|----------------|
| **> 0 °C** | Urban core is **warmer** than surrounding area — classic urban heat island |
| **= 0 °C** | No thermal anomaly between urban and rural zones |
| **< 0 °C** | Urban core is **cooler** — urban cool island (possible in vegetated or coastal cities) |
For example, a day SUHII of +2.5 °C means the core is 2.5 °C warmer than its background, and a SUHII
Sen slope of +0.05 °C yr⁻¹ means the contrast grows by 0.5 °C per decade.

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
def _compute_delta(df: pd.DataFrame, time_key: list) -> pd.DataFrame:
    """Pivot UC and BRA medians, compute Delta = UC - BRA."""
    uc  = df[df['zone'] == 'urban_core'].set_index(time_key + ['city', 'period'])
    bra = df[df['zone'] == 'bra'       ].set_index(time_key + ['city', 'period'])
    delta = (uc['median_lst'] - bra['median_lst']).reset_index()
    delta.columns = time_key + ['city', 'period', 'delta_lst']
    delta['zone'] = 'delta_uc_bra'
    return delta

delta_annual  = _compute_delta(df_annual,  ['year'])
delta_monthly = _compute_delta(df_monthly, ['year', 'month'])

# Run trend analysis on Delta series
delta_annual_stats_rows = []
for (city, period), grp in delta_annual.groupby(['city', 'period']):
    grp_s = grp.sort_values('year').set_index('year')['delta_lst'].dropna()
    res = run_trend_analysis(grp_s, mode='annual')
    hi = grp_s.idxmax() if not grp_s.empty else None
    lo = grp_s.idxmin() if not grp_s.empty else None
    delta_annual_stats_rows.append({
        'city': city, 'period': period, 'zone': 'delta_uc_bra',
        'highest_year': hi, 'highest_year_lst': round(float(grp_s.max()),4) if hi else None,
        'lowest_year':  lo, 'lowest_year_lst':  round(float(grp_s.min()),4) if lo else None,
        **res
    })

delta_monthly_stats_rows = []
for (city, period), grp in delta_monthly.groupby(['city', 'period']):
    grp_s = grp.sort_values(['year','month']).set_index(
        pd.to_datetime(grp.sort_values(['year','month'])
                       .apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1)))
    res = run_trend_analysis(grp_s['delta_lst'], mode='monthly')
    clim = grp.groupby('month')['delta_lst'].mean()
    amp  = round(float(clim.max()-clim.min()),4) if len(clim)==12 else None
    delta_monthly_stats_rows.append({
        'city': city, 'period': period, 'zone': 'delta_uc_bra',
        'monthly_amplitude': amp, **res
    })

df_delta_annual_stats  = pd.DataFrame(delta_annual_stats_rows)
df_delta_monthly_stats = pd.DataFrame(delta_monthly_stats_rows)

print(f'Delta annual stats  : {len(df_delta_annual_stats)}')
print(f'Delta monthly stats : {len(df_delta_monthly_stats)}')

---
<a id="nb03-s06c"></a>
### §6c — Per-City Temporal Charts

I draw one chart per city in the design described at the top.

### Reading the statistics boxes

The top-left box gives the day trends and the bottom-left box the night trends, one row per zone
(UC, BRA, SUHII):

| Symbol | Source method | Meaning |
|--------|--------------|---------|
| ↑ | Mann-Kendall | Statistically significant **increasing** trend |
| ↓ | Mann-Kendall | Statistically significant **decreasing** trend |
| `*` / `**` / `***` | Mann-Kendall p-value | p < 0.05 / p < 0.01 / p < 0.001 |
| `ns` | Mann-Kendall | No significant trend (p ≥ 0.05) |
| `+x.xx °C/dec` | **Sen's slope** | Warming (or cooling) **rate** per decade |
| `[H&R]` | Hamed-Rao correction | MK variance adjusted for autocorrelation — p-value is conservative |
| Dashed line on chart | **Sen's slope** | Visual overlay of the trend rate |
The arrow and the stars come from Mann–Kendall (is there a trend?); the °C per decade comes from
Sen's slope (how fast?). They are independent: a trend can be significant but slow, or fast but noisy.

### Colour scheme

| Element | Day-time | Night-time | Style |
|---------|----------|------------|-------|
| UC median LST | Orange `#E07B00` | Near-black `#1C1C1C` | Solid lw = 1.8 |
| BRA median LST | Dark gold `#B8860B` | Mid-grey `#6B6B6B` | Solid lw = 1.8 |
| SUHII (UC − BRA) | Red `#B22222` | Purple `#9932CC` | Solid lw = 2.8 + cosmetic glow halo |
| Sen's slope overlay | Same as zone colour | Same as zone colour | Thin dashed lw = 1.0 |
| City Region (range) | Peach fill | Grey fill | P5–P95 band α = 0.45 |
The SUHII axis is scaled to the data midpoint ± 3.5 half-ranges, so the SUHII line does not fill the
panel. I smooth the annual series with a Gaussian filter (σ = 0.8) and the monthly climatology with a
cubic spline; hollow circles mark only the raw peak and trough.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# create_temporal_chart() imported from src/visualization.py
# Colour and marker constants kept here for reference / override if needed.

COLOURS = {
    'uc_day':    '#E07B00',
    'uc_night':  '#4A6FA5',
    'bra_day':   '#F5C842',
    'bra_night': '#87CEEB',
    'delta_day':    '#B22222',
    'delta_night':  '#2E8B57',
    'cr_day_fill':   '#FFDAB9',
    'cr_night_fill': '#B0C4DE',
}
MARKERS    = {'uc': 'o', 'bra': 's', 'delta': '^'}
MONTH_ABBR = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

import importlib, visualization
importlib.reload(visualization)
from visualization import create_temporal_chart
print('Chart function ready (src/visualization.py) — reloaded.')


In [ ]:
created = skipped = 0

print('Generating per-city temporal trend charts...')
for city in tqdm(sorted(all_monthly.keys()), desc='Charts'):
    city_slug = zone_geoms[city]['city_slug']
    fig_dir   = RESULTS_03 / city_slug / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path  = fig_dir / f'{city_slug}_temporal_trend.svg'
    _ann = df_annual [df_annual ['city'] == city]
    _mon = df_monthly[df_monthly['city'] == city]
    _da  = delta_annual [delta_annual ['city'] == city]
    _dm  = delta_monthly[delta_monthly['city'] == city]
    _as  = df_annual_stats [df_annual_stats ['city'] == city]
    _ms  = df_monthly_stats[df_monthly_stats['city'] == city]
    _das = df_delta_annual_stats [df_delta_annual_stats ['city'] == city]
    _dms = df_delta_monthly_stats[df_delta_monthly_stats['city'] == city]
    try:
        fig = create_temporal_chart(
            city=city,
            df_ann=_ann, df_mon=_mon,
            df_del_ann=_da, df_del_mon=_dm,
            ann_stats=_as, mon_stats=_ms,
            del_ann_stats=_das, del_mon_stats=_dms,
            save_path=fig_path,
            dpi=config.FIGURE_DPI,
        )
        plt.close(fig)
        created += 1
    except Exception as e:
        print(f'\n  WARNING {city}: {e}')

print(f'Done — {created} charts generated.')


---
<a id="nb03-s06d"></a>
### §6d — Per-City CSV Export

I save every per-city and combined CSV to `Results/03_temporal_analysis/`.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── Per-city CSVs ─────────────────────────────────────────────────────────────
for city in sorted(all_monthly.keys()):
    city_slug = zone_geoms[city]['city_slug']
    city_out  = RESULTS_03 / city_slug
    city_out.mkdir(parents=True, exist_ok=True)

    _ann = df_annual [df_annual ['city'] == city]
    _mon = df_monthly[df_monthly['city'] == city]
    _as  = df_annual_stats [df_annual_stats ['city'] == city]
    _ms  = df_monthly_stats[df_monthly_stats['city'] == city]

    if not _ann.empty:
        _ann.merge(_as, on=['city','zone','period'], how='left').to_csv(
            city_out / f'{city_slug}_annual_lst.csv', index=False)
    if not _mon.empty:
        _mon.merge(_ms, on=['city','zone','period'], how='left').to_csv(
            city_out / f'{city_slug}_monthly_lst.csv', index=False)

# ── Aggregated CSVs ───────────────────────────────────────────────────────────
df_all_annual = df_annual.merge(
    df_annual_stats, on=['city','zone','period'], how='left')
df_all_monthly = df_monthly.merge(
    df_monthly_stats, on=['city','zone','period'], how='left')

df_all_annual.to_csv( RESULTS_03 / 'all_cities_temporal_annual.csv',  index=False)
df_all_monthly.to_csv(RESULTS_03 / 'all_cities_temporal_monthly.csv', index=False)

# Delta CSVs
delta_annual.merge(df_delta_annual_stats, on=['city','period','zone'], how='left'
    ).to_csv(RESULTS_03 / 'all_cities_delta_annual.csv', index=False)
delta_monthly.merge(df_delta_monthly_stats, on=['city','period','zone'], how='left'
    ).to_csv(RESULTS_03 / 'all_cities_delta_monthly.csv', index=False)

print('Exported:')
for f in sorted(RESULTS_03.glob('all_cities_*.csv')):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} kB)')

---
<a id="nb03-s06e"></a>
### §6e — City Summary Reports

I write three summaries straight from the data in memory:

| Output | Format | Contents |
|--------|--------|-----------|
| `Stats/City_LST_Wide_Summary.csv`  | Wide CSV     | One row per city — UC/BRA/Delta × day/night key stats |
| `Stats/City_LST_Wide_Summary.html` | Styled HTML  | Gradient-coloured for co-author/stakeholder review |
| `Stats/{city_slug}/{city_slug}_lst_summary.md` | Markdown | Per-city manuscript seed (all 3 zones) |
A warming rate reads `+0.123 °C/yr (R²=0.85*)`: Sen's slope, OLS R² and the Mann–Kendall star.

#### Data Dictionary: `City_LST_Wide_Summary.csv`

| Column prefix | Type | Units | Meaning |
|---------------|------|-------|---------|
| `UC_Day_MeanLST` | float | °C | Urban core daytime mean LST (average across all years) |
| `UC_Day_SenSlope` | float | °C/yr | Rate of change — Sen's slope (robust to outliers) |
| `UC_Day_OLS_R2` | float | 0–1 | Linear fit quality (1 = perfect straight line) |
| `UC_Day_MKTrend` | str | — | Mann-Kendall direction: `increasing` / `decreasing` / `no trend` |
| `UC_Day_MKp` | float | 0–1 | MK significance (< 0.05 = statistically significant trend) |
| `UC_Day_HRcorrected` | bool | — | `True` if Hamed-Rao autocorrelation correction was applied |
| `BRA_Day_*` | — | — | Same metrics for Background Reference Area |
| `SUHII_Day_*` | — | — | Same metrics for Urban Heat Island (UC − BRA delta series) |
| `*_Night_*` | — | — | Night-time equivalents of all above |
In the HTML, `*` p < 0.05, `**` p < 0.01, `***` p < 0.001 and `[H&R]` marks the Hamed–Rao correction;
the `National_Composite` row is the unweighted mean of the 56 cities.

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
import numpy as _np

STATS_DIR = RESULTS_03 / 'Stats'
STATS_DIR.mkdir(parents=True, exist_ok=True)

_MONTH_NAMES = {
    1:'January', 2:'February', 3:'March',    4:'April',
    5:'May',     6:'June',     7:'July',      8:'August',
    9:'September',10:'October',11:'November',12:'December',
}

def _fmt_rate(sr):
    if sr is None: return 'N/A'
    s  = sr.get('sen_slope_per_year',   float('nan'))
    r2 = sr.get('ols_r_squared',        float('nan'))
    p  = sr.get('mk_p',                 1.0)
    try:
        if _np.isnan(float(s)): return 'N/A'
    except (TypeError, ValueError): return 'N/A'
    sig    = '*' if float(p) < 0.05 else ''
    try:
        r2_str = f' (R²={float(r2):.2f}{sig})' if not _np.isnan(float(r2)) else sig
    except (TypeError, ValueError):
        r2_str = sig
    return f'{float(s):+.3f} °C/yr{r2_str}'

def _safe(val, fmt='.2f'):
    try:
        v = float(val)
        return format(v, fmt) if not _np.isnan(v) else '—'
    except (TypeError, ValueError):
        return '—'

# ── Build wide summary ────────────────────────────────────────────────────────
print('Building City_LST_Wide_Summary...')
records = []

for city in tqdm(sorted(df_annual['city'].unique()), desc='Wide summary'):
    row = {'City': city}

    for zone_label, zone_col, ann_src in [
            ('UC',    'urban_core',   df_annual_stats),
            ('BRA',   'bra',          df_annual_stats),
            ('SUHII', 'delta_uc_bra', df_delta_annual_stats),
    ]:
        for period in ['day', 'night']:
            pfx = f'{zone_label}_{period.title()}'

            # Annual trend stats row
            if 'zone' in ann_src.columns:
                _sr = ann_src[
                    (ann_src['city']   == city) &
                    (ann_src['zone']   == zone_col) &
                    (ann_src['period'] == period)]
            else:
                _sr = ann_src[
                    (ann_src['city']   == city) &
                    (ann_src['period'] == period)]
            sr = _sr.iloc[0].to_dict() if not _sr.empty else None

            # Raw annual data rows for this city×zone×period
            if zone_col == 'delta_uc_bra':
                _sub_ann = delta_annual[
                    (delta_annual['city']   == city) &
                    (delta_annual['period'] == period)]
                _lst_col = 'delta_lst'
            else:
                _sub_ann = df_annual[
                    (df_annual['city']   == city) &
                    (df_annual['zone']   == zone_col) &
                    (df_annual['period'] == period)]
                _lst_col = 'median_lst'

            _vals = _sub_ann[_lst_col].dropna()
            nan = float('nan')

            # Trend stats
            row[f'{pfx}_MeanLST']     = round(float(_vals.mean()), 3) if not _vals.empty else nan
            row[f'{pfx}_SenSlope']    = sr.get('sen_slope_per_year',   nan) if sr else nan
            row[f'{pfx}_SlopeDec']    = sr.get('sen_slope_per_decade', nan) if sr else nan
            row[f'{pfx}_OLS_R2']      = sr.get('ols_r_squared',        nan) if sr else nan
            row[f'{pfx}_OLS_p']       = sr.get('ols_p_value',          nan) if sr else nan
            row[f'{pfx}_MKTrend']     = sr.get('mk_trend', '')              if sr else ''
            row[f'{pfx}_MKp']         = sr.get('mk_p',     nan)             if sr else nan
            row[f'{pfx}_HRcorrected'] = bool(sr.get('autocorr_corrected', False)) if sr else False
            row[f'{pfx}_WarmingRate'] = _fmt_rate(sr)
            row[f'{pfx}_HighestYear'] = sr.get('highest_year',     nan) if sr else nan
            row[f'{pfx}_LowestYear']  = sr.get('lowest_year',      nan) if sr else nan

            # Descriptive LST range (non-delta only)
            if zone_col != 'delta_uc_bra' and not _sub_ann.empty:
                row[f'{pfx}_MeanLST_Mean']   = round(float(_sub_ann['mean_lst'].mean()),   3) if 'mean_lst'   in _sub_ann else nan
                row[f'{pfx}_MeanLST_Min']    = round(float(_sub_ann['min_lst'].min()),     3) if 'min_lst'    in _sub_ann else nan
                row[f'{pfx}_MeanLST_Max']    = round(float(_sub_ann['max_lst'].max()),     3) if 'max_lst'    in _sub_ann else nan
                row[f'{pfx}_MeanLST_P05']    = round(float(_sub_ann['p05_lst'].mean()),    3) if 'p05_lst'    in _sub_ann else nan
                row[f'{pfx}_MeanLST_P95']    = round(float(_sub_ann['p95_lst'].mean()),    3) if 'p95_lst'    in _sub_ann else nan
                # Coverage / data quality
                row[f'{pfx}_AvgObsPerYear']  = round(float(_sub_ann['n_obs_total'].mean()), 1) if 'n_obs_total'   in _sub_ann else nan
                row[f'{pfx}_AvgPctStrict']   = round(float(_sub_ann['pct_strict'].mean()),  4) if 'pct_strict'    in _sub_ann else nan
                row[f'{pfx}_AvgValidMonths'] = round(float(_sub_ann['n_valid_months'].mean()), 2) if 'n_valid_months' in _sub_ann else nan
                row[f'{pfx}_PctValidObs']    = round(float(_sub_ann['pct_valid_obs'].mean()),   4) if 'pct_valid_obs' in _sub_ann else nan
                n_low = (_sub_ann['low_coverage'] == True).sum() if 'low_coverage' in _sub_ann else nan
                row[f'{pfx}_LowCovYears']    = int(n_low) if not _np.isnan(float(n_low)) else nan
            else:
                for _sfx in ['_MeanLST_Mean','_MeanLST_Min','_MeanLST_Max','_MeanLST_P05',
                             '_MeanLST_P95','_AvgObsPerYear','_AvgPctStrict',
                             '_AvgValidMonths','_PctValidObs','_LowCovYears']:
                    row[f'{pfx}{_sfx}'] = nan

            # Monthly climatology stats (UC and BRA only)
            if zone_col != 'delta_uc_bra':
                _mr = df_monthly_stats[
                    (df_monthly_stats['city']   == city) &
                    (df_monthly_stats['zone']   == zone_col) &
                    (df_monthly_stats['period'] == period)]
                mr = _mr.iloc[0].to_dict() if not _mr.empty else None
            else:
                mr = None

            if mr:
                hi_mo = mr.get('highest_month')
                lo_mo = mr.get('lowest_month')
                amp   = mr.get('monthly_amplitude')
                try:
                    row[f'{pfx}_HottestMonth'] = (
                        _MONTH_NAMES.get(int(hi_mo), str(hi_mo))
                        if hi_mo and not _np.isnan(float(hi_mo)) else 'N/A')
                except (TypeError, ValueError):
                    row[f'{pfx}_HottestMonth'] = 'N/A'
                try:
                    row[f'{pfx}_CoolestMonth'] = (
                        _MONTH_NAMES.get(int(lo_mo), str(lo_mo))
                        if lo_mo and not _np.isnan(float(lo_mo)) else 'N/A')
                except (TypeError, ValueError):
                    row[f'{pfx}_CoolestMonth'] = 'N/A'
                try:
                    row[f'{pfx}_SeasonalAmp'] = (
                        round(float(amp), 3) if amp and not _np.isnan(float(amp)) else nan)
                except (TypeError, ValueError):
                    row[f'{pfx}_SeasonalAmp'] = nan
            else:
                row[f'{pfx}_HottestMonth'] = 'N/A'
                row[f'{pfx}_CoolestMonth'] = 'N/A'
                row[f'{pfx}_SeasonalAmp']  = nan

    records.append(row)

wide_df = pd.DataFrame(records)

wide_csv = STATS_DIR / 'City_LST_Wide_Summary.csv'
wide_df.to_csv(wide_csv, index=False)
print(f'  ✓ {wide_csv.name}  ({len(wide_df)} cities × {len(wide_df.columns)-1} metrics)')

# ── Styled HTML ──────────────────────────────────────────────────────────────
_num_mean  = [c for c in wide_df.columns if c.endswith('_MeanLST')]
_num_slope = [c for c in wide_df.columns if c.endswith('_SenSlope')]
_num_r2    = [c for c in wide_df.columns if c.endswith('_OLS_R2')]
_num_amp   = [c for c in wide_df.columns if c.endswith('_SeasonalAmp')]
_num_cov   = [c for c in wide_df.columns if c.endswith('_AvgPctStrict')]
_all_num   = _num_mean + _num_slope + _num_r2 + _num_amp
_fmt_map   = {c: '{:.3f}' for c in _all_num if c in wide_df.columns}

try:
    _styled = (
        wide_df.style
        .set_caption('City LST Wide Summary — UC / BRA / CR / SUHII × Day / Night')
        .format(_fmt_map, na_rep='—')
        .background_gradient(subset=_num_mean,  cmap='YlOrRd',   axis=0)
        .background_gradient(subset=_num_slope, cmap='RdYlGn_r', axis=0)
        .background_gradient(subset=_num_r2,    cmap='Blues',    axis=0)
        .background_gradient(subset=[c for c in _num_cov if c in wide_df.columns],
                             cmap='Greens', axis=0)
        .set_table_styles([
            {'selector': 'th',
             'props': 'background:#2c3e50;color:white;padding:6px;'
                      'font-size:11px;text-align:center;white-space:nowrap'},
            {'selector': 'td',
             'props': 'padding:4px 6px;font-size:10px;text-align:center'},
            {'selector': 'tr:nth-child(even)',
             'props': 'background:#f7f7f7'},
            {'selector': 'caption',
             'props': 'font-size:14px;font-weight:bold;padding:8px;text-align:left'},
        ])
    )
    html_path = STATS_DIR / 'City_LST_Wide_Summary.html'
    with open(html_path, 'w', encoding='utf-8') as _fh:
        _fh.write(_styled.to_html())
    print(f'  ✓ {html_path.name}')
except Exception as _e:
    print(f'  ⚠ HTML styling failed ({_e}); saving plain HTML')
    wide_df.to_html(STATS_DIR / 'City_LST_Wide_Summary.html', index=False)

# ── Per-city Markdown summaries ───────────────────────────────────────────────
print()
_written = 0
_years   = config.TEMPORAL_CONFIG['years']
_yr_rng  = f'{min(_years)}–{max(_years)}'

_ZONE_LABELS = [
    ('Urban Core (UC)',                 'UC'),
    ('Background Reference Area (BRA)', 'BRA'),
    ('SUHII — Surface Urban Heat Island Intensity (UC−BRA)', 'SUHII'),
]

for _, row in tqdm(wide_df.iterrows(), total=len(wide_df), desc='Markdown summaries'):
    city      = row['City']
    city_slug = city.replace(' ', '_')
    _cdir     = STATS_DIR / city_slug
    _cdir.mkdir(parents=True, exist_ok=True)

    lines = [
        f'# {city} — LST Temporal Analysis Summary', '',
        f'**Generated:** {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M UTC")}  ',
        f'**Source version:** `{SOURCE_VERSION}`  ',
        f'**Period:** MODIS MYD11A1, {_yr_rng}',
        '', '---', '',
    ]

    for zone_name, zk in _ZONE_LABELS:
        lines += [f'## {zone_name}', '']

        for period_label, pk in [
            ('Day-time (≈13:30 local)',   'Day'),
            ('Night-time (≈01:30 local)', 'Night'),
        ]:
            pfx      = f'{zk}_{pk}'
            mean_v   = row.get(f'{pfx}_MeanLST')
            rate_str = str(row.get(f'{pfx}_WarmingRate', 'N/A'))
            trend    = str(row.get(f'{pfx}_MKTrend', ''))
            p_val    = row.get(f'{pfx}_MKp')
            hr_flag  = bool(row.get(f'{pfx}_HRcorrected', False))
            hot_mo   = str(row.get(f'{pfx}_HottestMonth', 'N/A'))
            cool_mo  = str(row.get(f'{pfx}_CoolestMonth', 'N/A'))
            amp_v    = row.get(f'{pfx}_SeasonalAmp')
            slope_dc = row.get(f'{pfx}_SlopeDec')
            mn_min   = row.get(f'{pfx}_MeanLST_Min')
            mn_max   = row.get(f'{pfx}_MeanLST_Max')
            avg_obs  = row.get(f'{pfx}_AvgObsPerYear')
            pct_str  = row.get(f'{pfx}_AvgPctStrict')
            low_yrs  = row.get(f'{pfx}_LowCovYears')

            sym     = {'increasing':'↑','decreasing':'↓'}.get(trend, '—')
            p_str   = _safe(p_val, '.4f')
            hr_note = '  *(Hamed-Rao autocorrelation correction)*' if hr_flag else ''

            lines += [
                f'### {period_label}', '',
                '| Metric | Value |',
                '|--------|-------|',
                f'| Mean annual LST       | {_safe(mean_v)} °C |',
                f'| Annual range          | {_safe(mn_min)} – {_safe(mn_max)} °C |',
                f'| Warming rate          | {rate_str} |',
                f'| Decadal rate          | {_safe(slope_dc, "+.3f")} °C/decade |',
                f'| MK trend              | {sym} {trend} (p = {p_str}){hr_note} |',
            ]
            if zk != 'SUHII':
                lines += [
                    f'| Hottest month         | {hot_mo} |',
                    f'| Coolest month         | {cool_mo} |',
                    f'| Seasonal amplitude    | {_safe(amp_v)} °C |',
                    f'| Avg valid images/yr   | {_safe(avg_obs, ".0f")} |',
                    f'| Avg strict-quality %  | {_safe(pct_str, ".1%")} |',
                    f'| Low-coverage years    | {_safe(low_yrs, ".0f")} |',
                ]
            lines += ['']

    lines += [
        '---', '',
        '## Statistical Notes',
        '- **Warming rate**: Sen slope (°C yr⁻¹) with OLS R². `*` = Mann-Kendall p < 0.05.',
        '- **Autocorrelation**: Ljung-Box p < 0.05 triggers Hamed-Rao variance correction.',
        '- **Monthly series**: Seasonal Mann-Kendall (period = 12).',
        '- **Seasonal amplitude**: max − min of 12-month climatological mean (UC and BRA only).',
        '- **Strict quality**: MODIS QC bits 0-1 = 00 (best). Moderate = 01 (supplement).',
        '- **SUHII > 0**: urban heat island (UC warmer than BRA). **SUHII < 0**: urban cool island.',
        '',
    ]

    out_md = _cdir / f'{city_slug}_lst_summary.md'
    out_md.write_text('\n'.join(lines), encoding='utf-8')
    _written += 1

print(f'  ✓ {_written} Markdown summaries → Stats/{{city_slug}}/{{city_slug}}_lst_summary.md')
print(f'\nEnhanced exports complete → {STATS_DIR}')


---
<a id="nb03-s06f"></a>
### §6f — City Butterfly Violin

I show, for every city (warmest at the top), the spread of its annual urban-core LST over the study
years: day on the right in orange, night on the left in grey. The thick tick marks the median; red
and blue dots mark the hottest and coolest years.

**Output:** `publication/City_Butterfly_Violin.svg`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §6f — City Butterfly Violin ──────────────────────────────────────────────
# Butterfly half-violin: all study cities × all years, urban_core zone.
# Mirrors create_annual_lst_violin_chart() in src/visualization.py but uses
# df_annual in memory (avoids re-reading CSV and column-name mismatch).

import numpy as np
from scipy.stats import gaussian_kde as _gkde
import matplotlib.pyplot as plt
import matplotlib.ticker  as _mticker
import matplotlib.lines   as _mlines
import matplotlib.patches as _mpatch

PUB_DIR = RESULTS_03 / 'publication'
PUB_DIR.mkdir(parents=True, exist_ok=True)

_uc = df_annual[df_annual['zone'] == 'urban_core'].copy()

# Sort cities by mean daytime UC LST (descending → warmest at top)
_day_means = (
    _uc[_uc['period'] == 'day']
    .groupby('city')['median_lst'].mean()
    .sort_values(ascending=False)
)
_sorted_cities = _day_means.index.tolist()
_n             = len(_sorted_cities)
_city_ypos     = {c: i + 1 for i, c in enumerate(_sorted_cities)}

_X_MIN, _X_MAX = 10, 46
_DAY_COL   = '#FF7F0E'
_NIGHT_COL = '#555555'
_HALF_W    = 0.38
_KDE_PTS   = 300

fig, ax = plt.subplots(figsize=(13.0, max(5.0, _n * 0.42)))


def _draw_hv(data, y0, side, color):
    arr = np.asarray(data, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) < 2:
        return
    xlo = max(arr.min() - 0.5, _X_MIN)
    xhi = min(arr.max() + 0.5, _X_MAX)
    xg  = np.linspace(xlo, xhi, _KDE_PTS)
    try:
        sigma = np.std(arr)
        bw    = max(0.3, sigma * 0.35)
        kde   = _gkde(arr, bw_method=bw / (sigma if sigma > 0 else 1))
        dens  = kde(xg)
    except Exception:
        return
    dn = dens / dens.max() * _HALF_W if dens.max() > 0 else dens
    if side == 'right':
        yl, yh = np.full_like(xg, y0), y0 + dn
    else:
        yl, yh = y0 - dn, np.full_like(xg, y0)
    ax.fill_between(xg, yl, yh, color=color, alpha=0.48, linewidth=0)
    ax.plot(xg, yh, color=color, linewidth=0.9, alpha=0.85)
    ax.plot(xg, yl, color=color, linewidth=0.9, alpha=0.85)
    med  = float(np.median(arr))
    medn = float(np.interp(med, xg, dn))
    if side == 'right':
        ax.plot([med, med], [y0, y0 + medn],
                color=color, linewidth=2.5, solid_capstyle='round', zorder=5)
    else:
        ax.plot([med, med], [y0 - medn, y0],
                color=color, linewidth=2.5, solid_capstyle='round', zorder=5)


_has_hot = _has_cool = False
for _city in _sorted_cities:
    _y0 = _city_ypos[_city]
    for _per, _col, _side in [('day', _DAY_COL, 'right'), ('night', _NIGHT_COL, 'left')]:
        _vals = (_uc[(_uc['period'] == _per) & (_uc['city'] == _city)]
                 ['median_lst'].dropna().values)
        _draw_hv(_vals, _y0, _side, _col)
        if len(_vals) > 0:
            _dy = 0.10 if _per == 'day' else -0.10
            ax.scatter(_vals.max(), _y0 + _dy, color='#d73027', s=48, zorder=7,
                       linewidths=0,
                       label='Hottest year' if not _has_hot else '')
            ax.scatter(_vals.min(), _y0 + _dy, color='#4575b4', s=48, zorder=7,
                       linewidths=0,
                       label='Coolest year' if not _has_cool else '')
            _has_hot = _has_cool = True

ax.set_xlim(_X_MIN, _X_MAX)
ax.set_ylim(_n + 0.7, 0.3)
ax.set_xlabel('Annual Median UC LST (°C)', fontsize=12)
ax.xaxis.set_major_locator(_mticker.MultipleLocator(4))
ax.xaxis.set_minor_locator(_mticker.MultipleLocator(2))
ax.grid(axis='x', which='major', linestyle='--', alpha=0.45)
ax.grid(axis='x', which='minor', linestyle=':',  alpha=0.25)
ax.set_yticks(list(range(1, _n + 1)))
ax.set_yticklabels(_sorted_cities, fontsize=max(6, 10 - _n // 12))
ax.tick_params(axis='y', length=0)
ax.spines[['top', 'right']].set_visible(False)
ax.axvline(27.0, color='#888888', lw=0.7, ls=':', alpha=0.55)

_leg_handles = [
    _mpatch.Patch(color=_DAY_COL,   alpha=0.7, label='Daytime UC LST'),
    _mpatch.Patch(color=_NIGHT_COL, alpha=0.7, label='Night-time UC LST'),
    _mlines.Line2D([], [], color='#d73027', marker='o', ms=5, ls='',
                   label='Hottest year'),
    _mlines.Line2D([], [], color='#4575b4', marker='o', ms=5, ls='',
                   label='Coolest year'),
]
ax.legend(handles=_leg_handles, fontsize=8, framealpha=0.85, loc='lower right')
ax.set_title('Study Cities — Annual Urban Core LST Distribution (All Years)',
             fontsize=11, fontweight='bold', pad=8)

_out = PUB_DIR / 'City_Butterfly_Violin.svg'
fig.savefig(_out, bbox_inches='tight', dpi=config.FIGURE_DPI)
plt.close(fig)
print(f'City butterfly violin saved: {_out.name}  ({_n} cities)')


---
<a id="nb03-s06g"></a>
### §6g — City Comparison Table

I build an HTML table for scanning the cities side by side — the one output with the cities as
columns, meant for on-screen scrolling only; every CSV and printed table keeps cities as rows. It has
eight blocks (UC, BRA, CR and SUHII × day and night), each with the annual statistics (mean, spread,
hottest and coolest year, Sen slope and Mann–Kendall p, OLS R²) and, except for SUHII, the monthly
climatology (amplitude, hottest and coolest month). Zone colours match the charts; the national
composite column is highlighted.

**Output:** `Results/03_temporal_analysis/Stats/City_Comparison_Table.html`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
import numpy as _np
import pandas as _pd
from pathlib import Path
from IPython.display import display, HTML

# ── Zone style config ─────────────────────────────────────────────────────────
_ZONE_CFG = {
    "urban_core":    dict(label="Urban Core (UC)",
                          head_bg="#E07B00", head_fg="#FFFFFF",
                          row_bg="#FFF4E0"),
    "bra":           dict(label="Background Reference Area (BRA)",
                          head_bg="#B8860B", head_fg="#FFFFFF",
                          row_bg="#FFFFF0"),
    "city_region":   dict(label="City-Region (CR)",
                          head_bg="#CC8844", head_fg="#FFFFFF",
                          row_bg="#FFF4EC"),
    "delta_uc_bra":  dict(label="SUHII — UC minus BRA",
                          head_bg="#7B3333", head_fg="#FFFFFF",
                          row_bg="#FFF0F0"),
}
_PERIOD_LABEL = {"day": "Day-time", "night": "Night-time"}
_MN = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
       7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}

STATS_DIR = RESULTS_03 / "Stats"
STATS_DIR.mkdir(parents=True, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def _safe(v, fmt=".2f", sfx=""):
    try:
        f = float(v)
        return "-" if _np.isnan(f) else format(f, fmt) + sfx
    except:
        return "-"

def _mo_name(v):
    try: return _MN.get(int(float(v)), str(int(float(v))))
    except: return "-"

def _trend_str(row_dict):
    """Return compact trend string: arrow, Sen slope, MK p."""
    try:
        tr  = str(row_dict.get("mk_trend", ""))
        sym = {"increasing": "\u2191", "decreasing": "\u2193"}.get(tr, "\u2014")
        p   = float(row_dict.get("mk_p", _np.nan))
        hr  = " [H&R]" if row_dict.get("autocorr_corrected", False) else ""
        sig = ("***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns")))
        bd  = row_dict.get("sen_slope_per_decade", _np.nan)
        bd_s = _safe(bd, "+.2f", "\u00b0C/dec") if not _np.isnan(float(bd if bd is not None else _np.nan)) else "n/a"
        return f"{sym} {bd_s} p={p:.3f}{sig}{hr}"
    except:
        return "-"

# ── Load data ─────────────────────────────────────────────────────────────────
_ann_raw = _pd.read_csv(RESULTS_03 / "all_cities_temporal_annual.csv")
_mon_raw = _pd.read_csv(RESULTS_03 / "all_cities_temporal_monthly.csv")
_del_ann = _pd.read_csv(RESULTS_03 / "all_cities_delta_annual.csv")

# Annual stats per city/zone/period (first row per group = trend stats)
_TC = ["mk_trend","mk_p","autocorr_corrected","sen_slope_per_year",
       "sen_slope_per_decade","ols_r_squared","ols_p_value",
       "highest_year","highest_year_lst","lowest_year","lowest_year_lst"]

def _build_ann_stats(df, val_col, group_extra=None):
    grp = ["city","zone","period"] if not group_extra else group_extra
    agg = (df.groupby(grp)[val_col]
             .agg(mean="mean", std="std", min="min", max="max")
             .reset_index())
    agg["range"] = agg["max"] - agg["min"]
    tc = [c for c in _TC if c in df.columns]
    trend = df.groupby(grp)[tc].first().reset_index()
    return agg.merge(trend, on=grp)

_ann_stats = _build_ann_stats(_ann_raw, "median_lst")

# Delta / SUHII annual stats
_del_stats = _build_ann_stats(_del_ann, "delta_lst",
                              group_extra=["city","zone","period"]
                              if "zone" in _del_ann.columns
                              else ["city","period"])
if "zone" not in _del_stats.columns:
    _del_stats["zone"] = "delta_uc_bra"
_del_stats = _del_stats.rename(columns={"mean":"mean","std":"std","min":"min","max":"max","range":"range"})

# Monthly climatology per city/zone/period
_mon_cli = (_mon_raw.groupby(["city","zone","period","month"])["median_lst"]
            .mean().reset_index())
_mon_cli_pv = _mon_cli.pivot_table(index=["city","zone","period"], columns="month",
                                    values="median_lst").reset_index()
_mon_cli_pv.columns = [str(c) if isinstance(c, int) else c for c in _mon_cli_pv.columns]

# Hottest / coolest month per city/zone/period
_hot_mo = (_mon_cli.loc[_mon_cli.groupby(["city","zone","period"])["median_lst"].idxmax()]
           [["city","zone","period","month","median_lst"]]
           .rename(columns={"month":"hot_month","median_lst":"hot_month_lst"}))
_cool_mo = (_mon_cli.loc[_mon_cli.groupby(["city","zone","period"])["median_lst"].idxmin()]
            [["city","zone","period","month","median_lst"]]
            .rename(columns={"month":"cool_month","median_lst":"cool_month_lst"}))

# Seasonal amplitude per city/zone/period
_seas_amp = (_mon_cli.groupby(["city","zone","period"])["median_lst"]
             .agg(lambda x: x.max() - x.min()).reset_index()
             .rename(columns={"median_lst":"seas_amp"}))

# ── City order: alphabetical, with National Composite last ───────────────────
_all_cities = sorted([c for c in _ann_raw["city"].unique() if "Nigerian" not in c])
_comp_city  = next((c for c in _ann_raw["city"].unique() if "Nigerian" in c), None)
_city_order = _all_cities + ([_comp_city] if _comp_city else [])

# ── Parameter rows definition ─────────────────────────────────────────────────
_ANN_PARAMS = [
    ("Mean LST (°C)",       lambda r: _safe(r.get("mean"), ".2f")),
    ("Std dev (°C)",        lambda r: _safe(r.get("std"),  ".2f")),
    ("Min LST (°C)",        lambda r: _safe(r.get("min"),  ".2f")),
    ("Max LST (°C)",        lambda r: _safe(r.get("max"),  ".2f")),
    ("Range (°C)",          lambda r: _safe(r.get("range"),".2f")),
    ("Hottest year",        lambda r: (str(int(float(r["highest_year"])))
                                       if r.get("highest_year") is not None
                                       and not _np.isnan(float(r["highest_year"] if r["highest_year"] is not None else _np.nan))
                                       else "-")),
    ("Coolest year",        lambda r: (str(int(float(r["lowest_year"])))
                                       if r.get("lowest_year") is not None
                                       and not _np.isnan(float(r["lowest_year"] if r["lowest_year"] is not None else _np.nan))
                                       else "-")),
    ("Trend (Sen+MK)",      lambda r: _trend_str(r)),
    ("OLS R²",              lambda r: _safe(r.get("ols_r_squared"), ".3f")),
]

_MON_PARAMS = [
    ("Seas. amplitude (°C)", lambda r: _safe(r.get("seas_amp"), ".2f")),
    ("Hottest month",        lambda r: _mo_name(r.get("hot_month"))),
    ("Coolest month",        lambda r: _mo_name(r.get("cool_month"))),
    ("Hottest mo. LST (°C)", lambda r: _safe(r.get("hot_month_lst"), ".2f")),
    ("Coolest mo. LST (°C)", lambda r: _safe(r.get("cool_month_lst"), ".2f")),
]

# ── CSS ───────────────────────────────────────────────────────────────────────
_CSS = """
<style>
.cct { border-collapse:collapse; font-size:11px; font-family:Arial,sans-serif;
       width:100%; margin-bottom:24px; }
.cct th, .cct td { border:1px solid #ccc; padding:3px 5px; white-space:nowrap; }
.cct th.city-hdr { writing-mode:vertical-rl; transform:rotate(180deg);
                   height:100px; text-align:left; vertical-align:bottom; padding:4px 2px; }
.cct td.param-hdr { font-weight:bold; background:#F5F5F5; min-width:160px; }
.cct td.block-hdr { font-weight:bold; text-align:center; font-style:italic; }
.cct .comp { background:#D6EAF8 !important; font-style:italic; }
.footnote { font-size:10px; font-family:Arial,sans-serif; margin-top:6px;
            color:#333; border-top:1px solid #ccc; padding-top:4px; }
.legend-row { display:flex; gap:14px; flex-wrap:wrap; margin-bottom:10px; }
.leg-item { display:flex; align-items:center; gap:5px; font-size:11px; }
.leg-swatch { width:14px; height:14px; border-radius:2px; }
</style>
"""

# ── Legend ────────────────────────────────────────────────────────────────────
_LEGEND_HTML = '<div class="legend-row">'
for zk, zcfg in _ZONE_CFG.items():
    _LEGEND_HTML += (f'<div class="leg-item">'
                     f'<div class="leg-swatch" style="background:{zcfg["head_bg"]}"></div>'
                     f'{zcfg["label"]}</div>')
_LEGEND_HTML += "</div>"

# ── Footnote ─────────────────────────────────────────────────────────────────
_FOOTNOTE = """
<div class='footnote'>
<b>Notes:</b>
Sen's slope (°C/decade) with Mann–Kendall (MK) significance: *** p&lt;0.001 &nbsp;** p&lt;0.01 &nbsp;* p&lt;0.05 &nbsp;ns not significant.<br>
[H&amp;R] = Hamed &amp; Rao (1998) autocorrelation correction applied to MK variance when Ljung–Box test detects significant serial correlation.<br>
Sen's slope is the dashed line in trend charts. OLS R² indicates fraction of variance explained by linear regression.<br>
Seasonal amplitude = max monthly LST − min monthly LST (climatological mean). SUHII = Urban Core − Background Reference Area (year-wise median LST).<br>
Shaded blue columns = National composite (unweighted mean across all 56 city-region annual series).
</div>
"""

# ── Build one HTML table for a given zone × period ───────────────────────────
def _build_table(zone_key, period_key):
    zcfg = _ZONE_CFG[zone_key]
    is_delta = zone_key == "delta_uc_bra"
    hbg, hfg, rbg = zcfg["head_bg"], zcfg["head_fg"], zcfg["row_bg"]

    # Lookup dicts: city -> row dict
    if is_delta:
        _src = _del_stats
        def _ann_row(city):
            r = _src[(_src["city"] == city) & (_src["period"] == period_key)]
            return r.iloc[0].to_dict() if len(r) else {}
    else:
        _src = _ann_stats
        def _ann_row(city):
            r = _src[(_src["city"] == city) & (_src["zone"] == zone_key) & (_src["period"] == period_key)]
            return r.iloc[0].to_dict() if len(r) else {}

    def _mon_row(city):
        if is_delta:
            return {}
        r_amp  = _seas_amp[(_seas_amp["city"]==city) & (_seas_amp["zone"]==zone_key) & (_seas_amp["period"]==period_key)]
        r_hot  = _hot_mo  [(_hot_mo  ["city"]==city) & (_hot_mo  ["zone"]==zone_key) & (_hot_mo  ["period"]==period_key)]
        r_cool = _cool_mo [(_cool_mo ["city"]==city) & (_cool_mo ["zone"]==zone_key) & (_cool_mo ["period"]==period_key)]
        d = {}
        if len(r_amp):  d.update(r_amp.iloc[0].to_dict())
        if len(r_hot):  d.update(r_hot.iloc[0].to_dict())
        if len(r_cool): d.update(r_cool.iloc[0].to_dict())
        return d

    # Header row
    html = [f"<table class='cct'>"]
    html.append("<thead>")
    html.append(f"<tr><th style='background:{hbg};color:{hfg}' colspan='{len(_city_order)+1}'>"
                f"{zcfg['label']} — {_PERIOD_LABEL[period_key]}</th></tr>")
    html.append(f"<tr><th style='background:{hbg};color:{hfg}'>Parameter</th>")
    for city in _city_order:
        is_comp = (city == _comp_city)
        extra = " comp" if is_comp else ""
        short = city.replace("Nigerian Cities across 56 City-Regions", "National Composite")
        html.append(f"<th class='city-hdr{extra}' style='background:{hbg};color:{hfg}'>{short}</th>")
    html.append("</tr></thead><tbody>")

    # Annual block header
    html.append(f"<tr><td class='block-hdr' colspan='{len(_city_order)+1}' "
                f"style='background:{hbg}22'>Annual statistics (2006–2025)</td></tr>")

    # Annual param rows
    for param_name, param_fn in _ANN_PARAMS:
        html.append(f"<tr style='background:{rbg}'>"
                    f"<td class='param-hdr'>{param_name}</td>")
        for city in _city_order:
            is_comp = (city == _comp_city)
            extra = " comp" if is_comp else ""
            rd = _ann_row(city)
            val = param_fn(rd) if rd else "-"
            html.append(f"<td class='{extra}' style='text-align:center'>{val}</td>")
        html.append("</tr>")

    # Monthly block (skip for SUHII)
    if not is_delta:
        html.append(f"<tr><td class='block-hdr' colspan='{len(_city_order)+1}' "
                    f"style='background:{hbg}22'>Monthly climatology</td></tr>")
        for param_name, param_fn in _MON_PARAMS:
            html.append(f"<tr style='background:{rbg}'>"
                        f"<td class='param-hdr'>{param_name}</td>")
            for city in _city_order:
                is_comp = (city == _comp_city)
                extra = " comp" if is_comp else ""
                rd = _mon_row(city)
                val = param_fn(rd) if rd else "-"
                html.append(f"<td class='{extra}' style='text-align:center'>{val}</td>")
            html.append("</tr>")

    html.append("</tbody></table>")
    return "\n".join(html)

# ── Assemble full document ───────────────────────────────────────────────────
_ZONE_PERIOD_ORDER = [
    ("urban_core",   "day"),
    ("urban_core",   "night"),
    ("bra",          "day"),
    ("bra",          "night"),
    ("city_region",  "day"),
    ("city_region",  "night"),
    ("delta_uc_bra", "day"),
    ("delta_uc_bra", "night"),
]

_html_parts = [_CSS,
               "<h2>City Comparison Table — LST Temporal Analysis (56 Nigerian City-Regions)</h2>",
               _LEGEND_HTML]
for _zk, _pk in _ZONE_PERIOD_ORDER:
    _html_parts.append(_build_table(_zk, _pk))
_html_parts.append(_FOOTNOTE)

_full_html = "\n".join(_html_parts)
_out_path  = STATS_DIR / "City_Comparison_Table.html"
_out_path.write_text(_full_html, encoding="utf-8")
print(f"City_Comparison_Table saved: {_out_path}")

# Preview in notebook
display(HTML(_CSS + _LEGEND_HTML + _build_table("urban_core", "day") + _FOOTNOTE))


In [ ]:
print('=== TREND SUMMARY ===')
for zone in ['urban_core', 'bra', 'delta_uc_bra']:
    _s = df_all_annual if zone != 'delta_uc_bra' else delta_annual.merge(
        df_delta_annual_stats, on=['city','period','zone'], how='left')
    _s = _s[_s['zone'] == zone] if 'zone' in _s.columns else _s
    if _s.empty or 'mk_trend' not in _s.columns:
        continue
    print(f'\n{zone.upper()}')
    for period in ['day', 'night']:
        sub = _s[_s['period'] == period]
        n_inc = (sub['mk_trend'] == 'increasing').sum()
        n_dec = (sub['mk_trend'] == 'decreasing').sum()
        n_ns  = (sub['mk_trend'] == 'no trend'  ).sum()
        n_hr  = sub.get('autocorr_corrected', pd.Series(dtype=bool)).sum()
        print(f'  {period.upper():5s}: ↑{n_inc}  ↓{n_dec}  —{n_ns}  [H&R applied: {n_hr}]')

print(f'\nCities with charts : {created + skipped}')
print(f'Source version     : {SOURCE_VERSION}')



---
<a id="nb03-s07"></a>
## Section 7 — Cluster-Level Analysis

This is my main level of analysis. For each Cluster25 class with study cities, I average the city
series into one composite (each city counts equally, whatever its size), compute its SUHII, run the
§6a pipeline and draw its chart.

**Inputs:** the city series of §5; NB01's `city_cluster25_lookup.csv`; `config.CLUSTER25_COLORS`.
**Outputs:** one chart per cluster in `Cluster25_Composites/Cluster25_{NN}/figures/`, and the
cluster tables of §7b and §7c.

| Sub-section | Content |
|-------------|---------|
| [§7a — Cluster Pipeline](#nb03-s07a) | Composite → SUHII → MK/Sen/OLS statistics → chart (per-cluster loop) |
| [§7a2 — Violin Charts](#nb03-s07a2) | Distribution of city LSTs within each cluster (annual + monthly) |
| [§7b — Cluster CSV Export](#nb03-s07b) | Six CSVs: Annual, Monthly, SUHII series + stats |
| [§7c — Cluster Summary Reports](#nb03-s07c) | `Cluster25_LST_Wide_Summary.csv` + gradient HTML |
| [§7d — Publication Composites](#nb03-s07d) | 4×4 composite grids for annual trend, monthly climatology, SUHII |

---

> ↑ [Back to Table of Contents](#nb03-toc)

---
<a id="nb03-s07a"></a>
### §7a — Cluster Pipeline: Composite → SUHII → Statistics → Chart

For each Cluster25 class with at least two study cities I run:

| Step | Operation | Output variable |
|------|-----------|----------------|
| 1 | Composite UC/BRA/CR annual + monthly series | `df_cl_ann`, `df_cl_mon` |
| 2 | SUHII (UC − BRA) | `delta_cl_ann`, `delta_cl_mon` |
| 3 | Ljung-Box → MK / Hamed-Rao → Sen slope → OLS R² | `df_cl_st_ann`, `df_cl_st_da`, … |
| 4 | Temporal trend chart (same format as §6c) | `Cluster25_{NN}/figures/*.svg` |
| 5 | Accumulate into master DataFrames (for §7b / §7c) | `df_all_cl_ann_stats`, … |
**Output:** `Results/03_temporal_analysis/Cluster25_Composites/`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── Load cluster lookup ───────────────────────────────────────────────────────
_cl_path = RESULTS_03.parent / '01_aoi_delineation' / 'city_cluster25_lookup.csv'
if not _cl_path.exists():
    raise FileNotFoundError(
        f'Cluster lookup not found: {_cl_path}\n'
        'Run NB01 Section 8 first to generate city_cluster25_lookup.csv.'
    )
_cl_df = pd.read_csv(_cl_path)
_city_to_cluster  = dict(zip(_cl_df['city_name'], _cl_df['Cluster25'].astype(int)))
_cluster_to_cities = {}
for _ci, _cid in _city_to_cluster.items():
    _cluster_to_cities.setdefault(int(_cid), []).append(_ci)

_cl_meta = (_cl_df.groupby('Cluster25')
            .agg(Climate_Class=('Climate_Class','first'),
                 Landcover_Class=('Landcover_Class','first'),
                 land_code25=('land_code25','first'),
                 description=('description','first'))
            .reset_index())

CLUSTER_DIR = RESULTS_03 / 'Cluster25_Composites'
CLUSTER_DIR.mkdir(parents=True, exist_ok=True)

# ── Accumulators: filled per cluster, concatenated after loop (used in §7b, §7c) ─
_acc_cl_ann_data    = []   # composite annual series
_acc_cl_mon_data    = []   # composite monthly series
_acc_delta_ann_data = []   # SUHII annual series
_acc_delta_mon_data = []   # SUHII monthly series
_acc_cl_st_ann      = []   # annual stats (MK / Sen / OLS)
_acc_cl_st_mon      = []   # monthly stats
_acc_cl_st_da       = []   # delta annual stats
_acc_cl_st_dm       = []   # delta monthly stats

_study_cities = set(df_annual['city'].unique())
print(f'Cluster25 groups: {len(_cluster_to_cities)} total  |  study cities: {len(_study_cities)}')
for _cid in sorted(_cluster_to_cities):
    _av = [c for c in _cluster_to_cities[_cid] if c in _study_cities]
    _flag = '[composite]' if len(_av) >= 2 else '[single city — shown as-is]'
    print(f'  C{_cid:2d}: {len(_av):2d} cities {_flag}')

_cl_summary_rows = []

for _cid in sorted(_cluster_to_cities.keys()):
    _avail = [c for c in _cluster_to_cities[_cid] if c in _study_cities]
    _n     = len(_avail)
    _mt    = _cl_meta[_cl_meta['Cluster25'] == _cid].iloc[0]
    _cl_class = _mt['Climate_Class']
    _lc_class = _mt['Landcover_Class']
    _slug     = f'Cluster25_{_cid:02d}'
    _n_label  = f'{_n} {"city" if _n == 1 else "cities"}'
    _title    = f'Cluster {_cid} — {_cl_class} / {_lc_class} ({_n_label})'

    print(f'\n{"-"*60}')
    print(f'{_title}')

    if _n == 0:
        print('  No study cities — skipping.')
        continue

    _cl_summary_rows.append({
        'Cluster25': _cid, 'Climate_Class': _cl_class,
        'Landcover_Class': _lc_class, 'n_cities': _n,
        'cities': ', '.join(sorted(_avail)),
        'composite_computed': _n >= 2,
    })

    if _n == 1:
        print(f'  Single city: {_avail[0]} — computing full pipeline '
              f'(result is identical to the city-level output in §6).')
        # NOTE: single-city cluster → the composite = that city alone.
        # Included in all cluster-level outputs for completeness.

    # ── Composite annual series ───────────────────────────────────────────────
    _ca_rows = []
    for _zc in ['urban_core', 'bra', 'city_region']:
        for _per in ['day', 'night']:
            _s = df_annual[(df_annual['city'].isin(_avail)) &
                           (df_annual['zone'] == _zc) &
                           (df_annual['period'] == _per)]
            if _s.empty:
                continue
            _agg = {'median_lst': ('median_lst','mean'),
                    'lst_std':    ('median_lst','std'),
                    'n_cities':   ('median_lst','count')}
            for _col, _fn in [('mean_lst','mean'),('min_lst','min'),('max_lst','max'),
                               ('p05_lst','mean'),('p95_lst','mean')]:
                if _col in _s.columns:
                    _agg[_col] = (_col, _fn)
            _g = _s.groupby('year').agg(**_agg).reset_index()
            _g['city'] = _slug; _g['zone'] = _zc; _g['period'] = _per
            _ca_rows.append(_g)
    df_cl_ann = pd.concat(_ca_rows, ignore_index=True) if _ca_rows else pd.DataFrame()

    # ── Composite monthly series ──────────────────────────────────────────────
    _cm_rows = []
    for _zc in ['urban_core', 'bra', 'city_region']:
        for _per in ['day', 'night']:
            _s = df_monthly[(df_monthly['city'].isin(_avail)) &
                            (df_monthly['zone'] == _zc) &
                            (df_monthly['period'] == _per)]
            if _s.empty:
                continue
            _agg = {'median_lst': ('median_lst','mean'),
                    'lst_std':    ('median_lst','std'),
                    'n_cities':   ('median_lst','count')}
            for _col, _fn in [('mean_lst','mean'),('min_lst','min'),('max_lst','max'),
                               ('p05_lst','mean'),('p95_lst','mean')]:
                if _col in _s.columns:
                    _agg[_col] = (_col, _fn)
            _g = _s.groupby(['year','month']).agg(**_agg).reset_index()
            _g['city'] = _slug; _g['zone'] = _zc; _g['period'] = _per
            _cm_rows.append(_g)
    df_cl_mon = pd.concat(_cm_rows, ignore_index=True) if _cm_rows else pd.DataFrame()

    # ── Delta (SUHII) ─────────────────────────────────────────────────────────
    delta_cl_ann = _compute_delta(df_cl_ann, ['year'])
    delta_cl_mon = _compute_delta(df_cl_mon, ['year','month'])

    # ── Trend statistics ──────────────────────────────────────────────────────
    _st_ann, _st_mon, _st_da, _st_dm = [], [], [], []
    for _zc in ['urban_core','bra','city_region']:
        for _per in ['day','night']:
            _s  = df_cl_ann[(df_cl_ann['zone']==_zc) & (df_cl_ann['period']==_per)]
            _gs = _s.sort_values('year').set_index('year')['median_lst'].dropna()
            if len(_gs) < 5: continue
            _r  = run_trend_analysis(_gs, mode='annual')
            _st_ann.append({'city':_slug,'zone':_zc,'period':_per,
                            'highest_year':_gs.idxmax(),'highest_year_lst':round(float(_gs.max()),4),
                            'lowest_year': _gs.idxmin(),'lowest_year_lst': round(float(_gs.min()),4),
                            **_r})
    for _zc in ['urban_core','bra','city_region']:
        for _per in ['day','night']:
            _s   = df_cl_mon[(df_cl_mon['zone']==_zc) & (df_cl_mon['period']==_per)]
            _ord = _s.sort_values(['year','month'])
            _idx = pd.to_datetime(_ord.apply(lambda r: f'{int(r["year"])}-{int(r["month"]):02d}',axis=1))
            _gs  = _ord.set_index(_idx)['median_lst'].dropna()
            if len(_gs) < 12: continue
            _r    = run_trend_analysis(_gs, mode='monthly')
            _clim = _s.groupby('month')['median_lst'].mean()
            _amp  = round(float(_clim.max()-_clim.min()),4) if len(_clim)==12 else None
            _st_mon.append({'city':_slug,'zone':_zc,'period':_per,'monthly_amplitude':_amp,
                            'highest_month':int(_clim.idxmax()) if not _clim.empty else None,
                            'lowest_month': int(_clim.idxmin()) if not _clim.empty else None,
                            **_r})
    for _per in ['day','night']:
        _s  = delta_cl_ann[delta_cl_ann['period']==_per]
        _gs = _s.sort_values('year').set_index('year')['delta_lst'].dropna()
        if len(_gs) < 5: continue
        _r  = run_trend_analysis(_gs, mode='annual')
        _st_da.append({'city':_slug,'zone':'delta_uc_bra','period':_per,
                       'highest_year':_gs.idxmax(),'highest_year_lst':round(float(_gs.max()),4),
                       'lowest_year': _gs.idxmin(),'lowest_year_lst': round(float(_gs.min()),4),
                       **_r})
    for _per in ['day','night']:
        _s   = delta_cl_mon[delta_cl_mon['period']==_per]
        _ord = _s.sort_values(['year','month'])
        _idx = pd.to_datetime(_ord.apply(lambda r: f'{int(r["year"])}-{int(r["month"]):02d}',axis=1))
        _gs  = _ord.set_index(_idx)['delta_lst'].dropna()
        if len(_gs) < 12: continue
        _r    = run_trend_analysis(_gs, mode='monthly')
        _clim = _s.groupby('month')['delta_lst'].mean()
        _amp  = round(float(_clim.max()-_clim.min()),4) if len(_clim)==12 else None
        _st_dm.append({'city':_slug,'zone':'delta_uc_bra','period':_per,'monthly_amplitude':_amp,**_r})

    df_cl_st_ann = pd.DataFrame(_st_ann)
    df_cl_st_mon = pd.DataFrame(_st_mon)
    df_cl_st_da  = pd.DataFrame(_st_da)
    df_cl_st_dm  = pd.DataFrame(_st_dm)
    print(f'  Stats: annual={len(df_cl_st_ann)} monthly={len(df_cl_st_mon)} delta_ann={len(df_cl_st_da)}')

    # ── Temporal trend chart ──────────────────────────────────────────────────
    _fig_dir  = CLUSTER_DIR / _slug / 'figures'
    _fig_dir.mkdir(parents=True, exist_ok=True)
    _fig_path = _fig_dir / f'{_slug}_temporal_trend.svg'
    try:
        _cl_fig = create_temporal_chart(
            city=_title, df_ann=df_cl_ann, df_mon=df_cl_mon,
            df_del_ann=delta_cl_ann, df_del_mon=delta_cl_mon,
            ann_stats=df_cl_st_ann, mon_stats=df_cl_st_mon,
            del_ann_stats=df_cl_st_da, del_mon_stats=df_cl_st_dm,
            save_path=_fig_path, dpi=config.FIGURE_DPI,
        )
        plt.close(_cl_fig)
        print(f'  Chart: {_fig_path.name}')
    except Exception as _e:
        print(f'  Chart failed: {_e}')

    # ── Accumulate per-cluster results for §7b / §7c export ──────────────────
    _acc_cl_ann_data.append(df_cl_ann)
    _acc_cl_mon_data.append(df_cl_mon)
    _acc_delta_ann_data.append(delta_cl_ann)
    _acc_delta_mon_data.append(delta_cl_mon)
    if not df_cl_st_ann.empty: _acc_cl_st_ann.append(df_cl_st_ann)
    if not df_cl_st_mon.empty: _acc_cl_st_mon.append(df_cl_st_mon)
    if not df_cl_st_da.empty:  _acc_cl_st_da.append(df_cl_st_da)
    if not df_cl_st_dm.empty:  _acc_cl_st_dm.append(df_cl_st_dm)

# ── Final summary ─────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('Section 9e complete')
print('='*60)
_summ_df = (pd.DataFrame(_cl_summary_rows)
            .sort_values('n_cities', ascending=False)
            .reset_index(drop=True))
for _, _r in _summ_df.iterrows():
    _s = 'composite' if _r['composite_computed'] else 'single city'
    print(f'  C{int(_r["Cluster25"]):2d} ({int(_r["n_cities"]):2d} cities) [{_s}] '
          f'— {_r["Climate_Class"]} / {_r["Landcover_Class"]}')
print(f'\nOutputs: {CLUSTER_DIR}')

# ── Master cluster DataFrames (available for §7b CSV export and §7c reports) ──
_c = lambda lst: pd.concat(lst, ignore_index=True) if lst else pd.DataFrame()
df_all_cl_annual       = _c(_acc_cl_ann_data)
df_all_cl_monthly      = _c(_acc_cl_mon_data)
df_all_cl_delta_annual = _c(_acc_delta_ann_data)
df_all_cl_delta_monthly= _c(_acc_delta_mon_data)
df_all_cl_ann_stats    = _c(_acc_cl_st_ann)
df_all_cl_mon_stats    = _c(_acc_cl_st_mon)
df_all_cl_delta_stats  = _c(_acc_cl_st_da)
df_all_cl_delta_mstats = _c(_acc_cl_st_dm)
_n_comp = int((_summ_df['composite_computed']).sum()) if '_summ_df' in dir() and not _summ_df.empty else 0
print(f'Master DataFrames ready: {len(df_all_cl_ann_stats)} stat rows, {_n_comp} cluster composites')


---
<a id="nb03-s07a2"></a>
### §7a2 — Cluster Violin Charts

I show the spread of the cities' annual and monthly urban-core LST within each Cluster25 class:

| Figure | Layout | Each panel | Output |
|--------|--------|-----------|--------|
| Annual violin day | 2×8 grid | Violins per year — city spread in that cluster/year | `Cluster25_violin_annual_day.svg` |
| Annual violin night | 2×8 grid | Same for night | `Cluster25_violin_annual_night.svg` |
| Monthly violin day | 2×8 grid | Violins per calendar month | `Cluster25_violin_monthly_day.svg` |
| Monthly violin night | 2×8 grid | Same for night | `Cluster25_violin_monthly_night.svg` |
All in `Results/03_temporal_analysis/publication/violins/`.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §7a2 — Cluster Violin Charts (multi-city clusters only, 5×2 A4 grid) ────
# Distribution of city-level annual / monthly UC LST within each cluster.
# Only clusters with ≥ 3 study cities are plotted (6 of 16 clusters).
# Annual : one violin per year (spread = inter-city range that year).
# Monthly: one violin per calendar month (each = all city×year combos).

import numpy as np

PUB_DIR        = RESULTS_03 / 'publication'
PUB_VIOLIN_DIR = PUB_DIR / 'violins'
PUB_VIOLIN_DIR.mkdir(parents=True, exist_ok=True)

_MONTH_ABBR = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

_study_cluster_ids = sorted([
    cid for cid, cities in _cluster_to_cities.items()
    if any(c in _study_cities for c in cities)
])

# Multi-city filter — single-city clusters excluded from plots
_multi_cids = [
    cid for cid in _study_cluster_ids
    if len([c for c in _cluster_to_cities.get(cid, []) if c in _study_cities]) >= 3
]
_NVC_R, _NVC_C = 3, 2   # ceil(6/2)=3 rows × 2 cols  →  A4 portrait
print(f'Study clusters : {len(_study_cluster_ids)}  (all)  '
      f'| Multi-city: {len(_multi_cids)}  {_multi_cids}')


def _draw_violin(ax, data_list, positions, color, widths=0.75, alpha=0.72):
    for pos, vals in zip(positions, data_list):
        arr = np.asarray(vals, dtype=float)
        arr = arr[~np.isnan(arr)]
        if len(arr) >= 3 and np.std(arr) > 0:
            try:
                parts = ax.violinplot([arr], positions=[pos],
                                      widths=widths,
                                      showmeans=True, showmedians=False,
                                      showextrema=True)
                for pc in parts['bodies']:
                    pc.set_facecolor(color); pc.set_alpha(alpha)
                for key in ['cmeans', 'cmins', 'cmaxes', 'cbars']:
                    if key in parts:
                        parts[key].set_color(color); parts[key].set_linewidth(0.7)
            except Exception:
                ax.scatter([pos] * len(arr), arr, c=color, s=8, zorder=5, alpha=0.8)
        elif len(arr) > 0:
            ax.scatter([pos] * len(arr), arr, c=color, s=8, zorder=5, alpha=0.8)


def _style_ax(ax, title, xticks, xlabels, ylabel=False):
    ax.set_title(title, fontsize=7, fontweight='bold', pad=3)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=5, rotation=45, ha='right')
    ax.tick_params(axis='y', labelsize=5)
    if ylabel:
        ax.set_ylabel('UC LST (°C)', fontsize=6)
    ax.grid(axis='y', alpha=0.25, linewidth=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# ── Annual violins: 5×2 grid (A4 portrait) ───────────────────────────────────
for period in ['day', 'night']:
    _df_uc = (df_annual[(df_annual['zone'] == 'urban_core') &
                        (df_annual['period'] == period)].copy())
    _df_uc['cluster'] = _df_uc['city'].map(_city_to_cluster)
    _years   = sorted(_df_uc['year'].dropna().unique())
    _yr_pos  = list(range(1, len(_years) + 1))
    _yr_ticks  = _yr_pos[::4]
    _yr_labels = [str(int(y)) for y in _years[::4]]

    fig, axes = plt.subplots(_NVC_R, _NVC_C, figsize=(8.27, 11.69), sharey=False)
    fig.suptitle(
        f'Annual UC LST Distribution by Cluster — {period.capitalize()}time  '
        f'({int(_years[0]) if _years else "?"}–{int(_years[-1]) if _years else "?"})',
        fontsize=9, fontweight='bold', y=0.998)

    for _i, _cid in enumerate(_multi_cids):
        ax = axes[_i // _NVC_C, _i % _NVC_C]
        _cl_color  = config.CLUSTER25_COLORS.get(_cid, '#888888')
        _cl_cities = [c for c in _cluster_to_cities.get(_cid, []) if c in _study_cities]
        _cl_data   = _df_uc[_df_uc['cluster'] == _cid]
        _n_cities  = len(_cl_cities)

        _yr_vals = [_cl_data[_cl_data['year'] == yr]['median_lst'].values
                    for yr in _years]
        _draw_violin(ax, _yr_vals, _yr_pos, _cl_color, widths=0.65)
        _all_v  = _cl_data['median_lst'].dropna().values
        _sd_v   = float(np.std(_all_v)) if len(_all_v) >= 2 else float('nan')
        _sd_lbl = f'  σ={_sd_v:.1f}°C' if not np.isnan(_sd_v) else ''
        _style_ax(ax,
                  title=f'C{_cid}  (n={_n_cities}{_sd_lbl})',
                  xticks=_yr_ticks,
                  xlabels=_yr_labels,
                  ylabel=(_i % _NVC_C == 0))
        ax.set_xlim(0.2, len(_years) + 0.8)

    for _j in range(len(_multi_cids), _NVC_R * _NVC_C):
        axes[_j // _NVC_C, _j % _NVC_C].set_visible(False)

    plt.tight_layout()
    _out = PUB_VIOLIN_DIR / f'Cluster25_violin_annual_{period}.svg'
    fig.savefig(_out, bbox_inches='tight', dpi=config.FIGURE_DPI)
    plt.close(fig)
    print(f'  ✓  {_out.name}  ({len(_multi_cids)} clusters, {_NVC_R}×{_NVC_C})')


# ── Monthly violins: 5×2 grid ─────────────────────────────────────────────────
for period in ['day', 'night']:
    _df_uc = (df_monthly[(df_monthly['zone'] == 'urban_core') &
                         (df_monthly['period'] == period)].copy())
    _df_uc['cluster'] = _df_uc['city'].map(_city_to_cluster)

    fig, axes = plt.subplots(_NVC_R, _NVC_C, figsize=(8.27, 11.69), sharey=False)
    fig.suptitle(
        f'Monthly UC LST Distribution by Cluster — {period.capitalize()}time',
        fontsize=9, fontweight='bold', y=0.998)

    for _i, _cid in enumerate(_multi_cids):
        ax = axes[_i // _NVC_C, _i % _NVC_C]
        _cl_color  = config.CLUSTER25_COLORS.get(_cid, '#888888')
        _cl_cities = [c for c in _cluster_to_cities.get(_cid, []) if c in _study_cities]
        _cl_data   = _df_uc[_df_uc['cluster'] == _cid]
        _mo_vals   = [_cl_data[_cl_data['month'] == mo]['median_lst'].values
                      for mo in range(1, 13)]
        _draw_violin(ax, _mo_vals, list(range(1, 13)), _cl_color, widths=0.7)
        _all_v_m  = _cl_data['median_lst'].dropna().values
        _sd_v_m   = float(np.std(_all_v_m)) if len(_all_v_m) >= 2 else float('nan')
        _sd_lbl_m = f'  σ={_sd_v_m:.1f}°C' if not np.isnan(_sd_v_m) else ''
        _style_ax(ax,
                  title=f'C{_cid}  (n={len(_cl_cities)}{_sd_lbl_m})',
                  xticks=list(range(1, 13)),
                  xlabels=_MONTH_ABBR,
                  ylabel=(_i % _NVC_C == 0))
        ax.set_xlim(0.2, 12.8)

    for _j in range(len(_multi_cids), _NVC_R * _NVC_C):
        axes[_j // _NVC_C, _j % _NVC_C].set_visible(False)

    plt.tight_layout()
    _out = PUB_VIOLIN_DIR / f'Cluster25_violin_monthly_{period}.svg'
    fig.savefig(_out, bbox_inches='tight', dpi=config.FIGURE_DPI)
    plt.close(fig)
    print(f'  ✓  {_out.name}  ({len(_multi_cids)} clusters, {_NVC_R}×{_NVC_C})')

print(f'\nViolin charts saved → {PUB_VIOLIN_DIR}')

---
<a id="nb03-s07a2b"></a>
### §7a2b — Cluster Butterfly Violin Charts

I draw a butterfly violin per cluster, day on the right (orange) against night on the left (blue):
one per year for the annual charts and one per calendar month for the monthly charts. I show only the
six clusters with at least three study cities (a 3 × 2 A4 grid); σ_d and σ_n are the standard
deviations pooled over all years and cities.

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §7a2b — Cluster Butterfly Violin Charts (≥3 cities, 3×2 A4) ─────────────
# Kept alongside §7a2 symmetric violins; team picks best format for publication.
import numpy as _npb
from scipy.stats import gaussian_kde as _gkde_b
import matplotlib.pyplot as _pltb
import matplotlib.patches as _mpb

PUB_VIOLIN_DIR = RESULTS_03 / 'publication' / 'violins'
PUB_VIOLIN_DIR.mkdir(parents=True, exist_ok=True)

_MONTH_ABBR_B = ['Jan','Feb','Mar','Apr','May','Jun',
                 'Jul','Aug','Sep','Oct','Nov','Dec']

_multi_cids_b = [
    cid for cid in _study_cluster_ids
    if len([c for c in _cluster_to_cities.get(cid, []) if c in _study_cities]) >= 3
]
_NBR_R, _NBR_C = 3, 2    # ceil(6/2)=3 rows × 2 cols
_DAY_B  = '#FF7F0E'
_NGT_B  = '#4575b4'


def _draw_half_b(ax, arr, xpos, side, color, half_w=0.38):
    arr = _npb.asarray(arr, float); arr = arr[~_npb.isnan(arr)]
    if len(arr) < 2:
        if len(arr) == 1:
            off = half_w / 2 if side == 'right' else -half_w / 2
            ax.scatter(xpos + off, arr[0], color=color, s=20, zorder=7, lw=0)
        return
    yg = _npb.linspace(arr.min() - 0.5, arr.max() + 0.5, 250)
    try:
        sig = _npb.std(arr); bw = max(0.3, sig * 0.35)
        kde = _gkde_b(arr, bw_method=bw / (sig if sig > 0 else 1))
        dn  = kde(yg); dn = dn / dn.max() * half_w if dn.max() > 0 else dn
    except Exception:
        return
    if side == 'right':
        ax.fill_betweenx(yg, xpos, xpos + dn, color=color, alpha=0.45, lw=0)
        ax.plot(xpos + dn, yg, color=color, lw=0.8, alpha=0.85)
    else:
        ax.fill_betweenx(yg, xpos - dn, xpos, color=color, alpha=0.45, lw=0)
        ax.plot(xpos - dn, yg, color=color, lw=0.8, alpha=0.85)
    med    = float(_npb.median(arr))
    med_dn = float(_npb.interp(med, yg, dn))
    if side == 'right':
        ax.plot([xpos, xpos + med_dn], [med, med], color=color, lw=2.0, zorder=6)
    else:
        ax.plot([xpos - med_dn, xpos], [med, med], color=color, lw=2.0, zorder=6)


def _style_b(ax, title, xticks, xlabels, ylabel=False):
    ax.set_title(title, fontsize=6.5, fontweight='bold', pad=3)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=5, rotation=45, ha='right')
    ax.tick_params(axis='y', labelsize=5)
    if ylabel:
        ax.set_ylabel('UC LST (°C)', fontsize=6)
    ax.grid(axis='y', alpha=0.25, lw=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


_leg_d_b = _mpb.Patch(facecolor=_DAY_B, alpha=0.7, label='Daytime')
_leg_n_b = _mpb.Patch(facecolor=_NGT_B, alpha=0.7, label='Nighttime')

# ── Annual butterfly ─────────────────────────────────────────────────────────
_OUT_B_ANN = PUB_VIOLIN_DIR / 'Cluster25_butterfly_annual.svg'
if _OUT_B_ANN.exists():
    print(f'§7a2b annual cache hit — {_OUT_B_ANN.name}')
else:
    _df_d_b = (df_annual[(df_annual['zone'] == 'urban_core') &
                          (df_annual['period'] == 'day')].copy())
    _df_n_b = (df_annual[(df_annual['zone'] == 'urban_core') &
                          (df_annual['period'] == 'night')].copy())
    _df_d_b['cluster'] = _df_d_b['city'].map(_city_to_cluster)
    _df_n_b['cluster'] = _df_n_b['city'].map(_city_to_cluster)
    _yrs_b   = sorted(_df_d_b['year'].dropna().unique())
    _yp_b    = list(range(1, len(_yrs_b) + 1))
    _yt_b    = _yp_b[::4]
    _yl_b    = [str(int(y)) for y in _yrs_b[::4]]

    fig_b, axes_b = _pltb.subplots(_NBR_R, _NBR_C, figsize=(8.27, 11.69), sharey=False)
    fig_b.suptitle(
        f'Annual UC LST — Cluster Butterfly  (Day right ▶ / Night left ◀)\n'
        f'{int(_yrs_b[0])}–{int(_yrs_b[-1])}  |  ≥3 cities per cluster',
        fontsize=8.5, fontweight='bold', y=0.999)

    for _i, _cid in enumerate(_multi_cids_b):
        ax = axes_b[_i // _NBR_C, _i % _NBR_C]
        _dc = _df_d_b[_df_d_b['cluster'] == _cid]
        _nc = _df_n_b[_df_n_b['cluster'] == _cid]
        _nc_cities = len([c for c in _cluster_to_cities.get(_cid, []) if c in _study_cities])
        _sd_d = float(_npb.std(_dc['median_lst'].dropna().values)) if len(_dc) >= 2 else float('nan')
        _sd_n = float(_npb.std(_nc['median_lst'].dropna().values)) if len(_nc) >= 2 else float('nan')
        _sd_s = (f'  σ_d={_sd_d:.1f}, σ_n={_sd_n:.1f}°C'
                 if not (_npb.isnan(_sd_d) or _npb.isnan(_sd_n)) else '')
        _ttl  = f'C{_cid} (n={_nc_cities}){_sd_s}'

        for _xp, _yr in zip(_yp_b, _yrs_b):
            _draw_half_b(ax, _dc[_dc['year'] == _yr]['median_lst'].values, _xp, 'right', _DAY_B)
            _draw_half_b(ax, _nc[_nc['year'] == _yr]['median_lst'].values, _xp, 'left',  _NGT_B)

        _style_b(ax, title=_ttl, xticks=_yt_b, xlabels=_yl_b,
                 ylabel=(_i % _NBR_C == 0))
        ax.set_xlim(0.2, len(_yrs_b) + 0.8)
        if _i == 0:
            ax.legend(handles=[_leg_d_b, _leg_n_b], fontsize=5,
                      loc='upper right', framealpha=0.6)

    for _j in range(len(_multi_cids_b), _NBR_R * _NBR_C):
        axes_b[_j // _NBR_C, _j % _NBR_C].set_visible(False)

    _pltb.tight_layout()
    fig_b.savefig(_OUT_B_ANN, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _pltb.close(fig_b)
    print(f'  ✓  {_OUT_B_ANN.name}  ({len(_multi_cids_b)} clusters, {_NBR_R}×{_NBR_C})')


# ── Monthly butterfly ────────────────────────────────────────────────────────
_OUT_B_MO = PUB_VIOLIN_DIR / 'Cluster25_butterfly_monthly.svg'
if _OUT_B_MO.exists():
    print(f'§7a2b monthly cache hit — {_OUT_B_MO.name}')
else:
    _df_d_mo = (df_monthly[(df_monthly['zone'] == 'urban_core') &
                            (df_monthly['period'] == 'day')].copy())
    _df_n_mo = (df_monthly[(df_monthly['zone'] == 'urban_core') &
                            (df_monthly['period'] == 'night')].copy())
    _df_d_mo['cluster'] = _df_d_mo['city'].map(_city_to_cluster)
    _df_n_mo['cluster'] = _df_n_mo['city'].map(_city_to_cluster)

    fig_bm, axes_bm = _pltb.subplots(_NBR_R, _NBR_C, figsize=(8.27, 11.69), sharey=False)
    fig_bm.suptitle(
        'Monthly UC LST — Cluster Butterfly  (Day right ▶ / Night left ◀)\n'
        '≥3 cities per cluster',
        fontsize=8.5, fontweight='bold', y=0.999)

    for _i, _cid in enumerate(_multi_cids_b):
        ax = axes_bm[_i // _NBR_C, _i % _NBR_C]
        _dc_m = _df_d_mo[_df_d_mo['cluster'] == _cid]
        _nc_m = _df_n_mo[_df_n_mo['cluster'] == _cid]
        _nc_c = len([c for c in _cluster_to_cities.get(_cid, []) if c in _study_cities])
        _sd_d_m = float(_npb.std(_dc_m['median_lst'].dropna().values)) if len(_dc_m) >= 2 else float('nan')
        _sd_n_m = float(_npb.std(_nc_m['median_lst'].dropna().values)) if len(_nc_m) >= 2 else float('nan')
        _sd_s_m = (f'  σ_d={_sd_d_m:.1f}, σ_n={_sd_n_m:.1f}°C'
                   if not (_npb.isnan(_sd_d_m) or _npb.isnan(_sd_n_m)) else '')
        _ttl_m  = f'C{_cid} (n={_nc_c}){_sd_s_m}'

        for _mo in range(1, 13):
            _draw_half_b(ax, _dc_m[_dc_m['month'] == _mo]['median_lst'].values, _mo, 'right', _DAY_B)
            _draw_half_b(ax, _nc_m[_nc_m['month'] == _mo]['median_lst'].values, _mo, 'left',  _NGT_B)

        _style_b(ax, title=_ttl_m, xticks=list(range(1, 13)), xlabels=_MONTH_ABBR_B,
                 ylabel=(_i % _NBR_C == 0))
        ax.set_xlim(0.2, 12.8)
        if _i == 0:
            ax.legend(handles=[_leg_d_b, _leg_n_b], fontsize=5,
                      loc='upper right', framealpha=0.6)

    for _j in range(len(_multi_cids_b), _NBR_R * _NBR_C):
        axes_bm[_j // _NBR_C, _j % _NBR_C].set_visible(False)

    _pltb.tight_layout()
    fig_bm.savefig(_OUT_B_MO, bbox_inches='tight', dpi=config.FIGURE_DPI)
    _pltb.close(fig_bm)
    print(f'  ✓  {_OUT_B_MO.name}  ({len(_multi_cids_b)} clusters, {_NBR_R}×{_NBR_C})')

print(f'\nButterfly violins → {PUB_VIOLIN_DIR}')


---
<a id="nb03-s07b"></a>
### §7b — Cluster CSV Export

I export the cluster tables of §7a, in the same columns as the city exports (§6d):

| File | Contents |
|------|---------|
| `Cluster25_LST_Annual.csv` | Annual median LST per cluster × zone × period |
| `Cluster25_LST_Monthly.csv` | Monthly median LST per cluster × zone × period |
| `Cluster25_SUHII_Annual.csv` | Annual SUHII (UC−BRA) per cluster |
| `Cluster25_Annual_Stats.csv` | MK trend, Sen slope, OLS R² — annual series |
| `Cluster25_Monthly_Stats.csv` | MK trend, Sen slope, OLS R² — monthly series |
| `Cluster25_SUHII_Stats.csv` | MK trend, Sen slope, OLS R² — SUHII series |
**Output directory:** `Results/03_temporal_analysis/Cluster25_Composites/stats/`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §7b — Cluster CSV Export ──────────────────────────────────────────────────
# Exports master cluster DataFrames as CSVs — mirrors city-level §6d structure.

CL_STAT_DIR = CLUSTER_DIR / 'stats'
CL_STAT_DIR.mkdir(parents=True, exist_ok=True)

_exports = [
    ('Cluster25_LST_Annual.csv',    df_all_cl_annual,       'composite annual series'),
    ('Cluster25_LST_Monthly.csv',   df_all_cl_monthly,      'composite monthly series'),
    ('Cluster25_SUHII_Annual.csv',  df_all_cl_delta_annual, 'SUHII annual series'),
    ('Cluster25_SUHII_Monthly.csv', df_all_cl_delta_monthly,'SUHII monthly series'),
    ('Cluster25_Annual_Stats.csv',  df_all_cl_ann_stats,    'MK/Sen/OLS annual stats'),
    ('Cluster25_Monthly_Stats.csv', df_all_cl_mon_stats,    'MK/Sen/OLS monthly stats'),
    ('Cluster25_SUHII_Stats.csv',   df_all_cl_delta_stats,  'SUHII annual stats'),
]
print('Cluster CSV exports:')
for fname, df, desc in _exports:
    if df is not None and not df.empty:
        out = CL_STAT_DIR / fname
        df.to_csv(out, index=False)
        _n_cl = df['city'].nunique() if 'city' in df.columns else '—'
        print(f'  ✓ {fname:45s}  {len(df):5d} rows  {_n_cl} clusters  [{desc}]')
    else:
        print(f'  — {fname:45s}  (empty — no composites computed)')
print(f'\nAll cluster CSVs → {CL_STAT_DIR}')


---
<a id="nb03-s07c"></a>
### §7c — Cluster Summary Reports

I write the cluster summary in the same columns as the city summary (§6e), so city and cluster
results compare directly. It is my cluster-level results table.

| Output | Format | Contents |
|--------|--------|---------|
| `Cluster25_LST_Wide_Summary.csv` | Wide CSV | One row per cluster — UC/BRA/SUHII × Day/Night key stats |
| `Cluster25_LST_Wide_Summary.html` | Gradient HTML | For co-author/stakeholder review |
---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §7c — Cluster Wide Summary Report (mirrors §6e City Wide Summary) ─────────
# Builds Cluster25_LST_Wide_Summary.csv + HTML with same column format as
# City_LST_Wide_Summary.csv for direct comparison.

import numpy as _np2

_ZONE_LABELS_CL = [('UC', 'urban_core'), ('BRA', 'bra'), ('SUHII', 'delta_uc_bra')]

def _cl_stat_val(stats_df, cluster_slug, zone, period, col):
    if stats_df is None or stats_df.empty:
        return float('nan')
    sub = stats_df[(stats_df['city'] == cluster_slug) &
                   (stats_df.get('zone', pd.Series()) == zone if 'zone' in stats_df.columns
                    else pd.Series([True]*len(stats_df))) &
                   (stats_df['period'] == period)]
    if sub.empty:
        return float('nan')
    return sub.iloc[0].get(col, float('nan'))

def _cl_mean_lst(series_df, cluster_slug, zone_or_key, period, val_col):
    if series_df is None or series_df.empty:
        return float('nan')
    if 'zone' in series_df.columns:
        sub = series_df[(series_df['city'] == cluster_slug) &
                        (series_df['zone'] == zone_or_key) &
                        (series_df['period'] == period)]
    else:
        sub = series_df[(series_df['city'] == cluster_slug) &
                        (series_df['period'] == period)]
    vals = sub[val_col].dropna() if val_col in sub.columns else pd.Series()
    return round(float(vals.mean()), 3) if not vals.empty else float('nan')

def _cl_hottest_month(mon_stats_df, cluster_slug, zone, period):
    sub = mon_stats_df[(mon_stats_df['city'] == cluster_slug) &
                       (mon_stats_df['zone'] == zone) &
                       (mon_stats_df['period'] == period)]           if (mon_stats_df is not None and not mon_stats_df.empty and
              'zone' in mon_stats_df.columns) else pd.DataFrame()
    if sub.empty:
        return 'N/A'
    hm = sub.iloc[0].get('highest_month')
    _MNAMES = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    try:
        return _MNAMES.get(int(hm), 'N/A')
    except:
        return 'N/A'

# All cluster slugs with composites
_cl_slugs = sorted(df_all_cl_ann_stats['city'].unique().tolist())             if not df_all_cl_ann_stats.empty else []

_cl_wide_rows = []
for _slug in _cl_slugs:
    _cid = int(_slug.split('_')[-1])
    _cities_in = sorted([c for c in _cluster_to_cities.get(_cid, [])
                          if c in set(df_annual['city'].unique())])
    row = {
        'Cluster': _slug,
        'Cluster_ID': _cid,
        'N_Cities': len(_cities_in),
        'Member_Cities': ', '.join(_cities_in),
    }
    for _zl, _zk in _ZONE_LABELS_CL:
        _stats_df = df_all_cl_ann_stats if _zk != 'delta_uc_bra' else df_all_cl_delta_stats
        _mon_df   = df_all_cl_mon_stats  if _zk != 'delta_uc_bra' else df_all_cl_delta_mstats
        _ser_df   = df_all_cl_annual     if _zk != 'delta_uc_bra' else df_all_cl_delta_annual
        _val_col  = 'median_lst'         if _zk != 'delta_uc_bra' else 'delta_lst'
        for _per in ['day', 'night']:
            _pfx = f'{_zl}_{_per.title()}'
            _ss  = _cl_stat_val(_stats_df, _slug, _zk, _per, 'sen_slope_per_year')
            _mk  = _cl_stat_val(_stats_df, _slug, _zk, _per, 'mk_trend')
            _mkp = _cl_stat_val(_stats_df, _slug, _zk, _per, 'mk_p')
            _r2  = _cl_stat_val(_stats_df, _slug, _zk, _per, 'ols_r_squared')
            _hr  = _cl_stat_val(_stats_df, _slug, _zk, _per, 'autocorr_corrected')
            _ml  = _cl_mean_lst(_ser_df,   _slug, _zk, _per, _val_col)
            _hm  = _cl_hottest_month(_mon_df, _slug, _zk, _per) if _zk != 'delta_uc_bra' else 'N/A'
            try:   _sd  = float(_ss) * 10 if not _np2.isnan(float(_ss)) else float('nan')
            except: _sd = float('nan')
            # Warming rate string
            try:
                _star = '***' if float(_mkp)<0.001 else ('**' if float(_mkp)<0.01
                        else ('*' if float(_mkp)<0.05 else ''))
                _hr_tag = '[H&R]' if str(_hr) == 'True' else ''
                _wr = f'{float(_ss):+.3f} °C/yr (R²={float(_r2):.2f}{_star}){_hr_tag}'
            except:
                _wr = 'N/A'
            row.update({
                f'{_pfx}_MeanLST':     round(float(_ml), 3) if not _np2.isnan(float(_ml)) else float('nan'),
                f'{_pfx}_SenSlope':    round(float(_ss), 4) if not _np2.isnan(float(_ss)) else float('nan'),
                f'{_pfx}_SlopeDec':    round(float(_sd), 3) if not _np2.isnan(float(_sd)) else float('nan'),
                f'{_pfx}_OLS_R2':      round(float(_r2), 3) if not _np2.isnan(float(_r2)) else float('nan'),
                f'{_pfx}_MKTrend':     str(_mk),
                f'{_pfx}_MKp':         round(float(_mkp), 4) if not _np2.isnan(float(_mkp)) else float('nan'),
                f'{_pfx}_HRcorrected': str(_hr) == 'True',
                f'{_pfx}_WarmingRate': _wr,
                f'{_pfx}_HottestMonth': _hm,
            })
    _cl_wide_rows.append(row)

cl_wide_df = pd.DataFrame(_cl_wide_rows)

# ── Export CSV ────────────────────────────────────────────────────────────────
_cl_wide_path = CL_STAT_DIR / 'Cluster25_LST_Wide_Summary.csv'
cl_wide_df.to_csv(_cl_wide_path, index=False)
print(f'✓ {_cl_wide_path.name}  ({len(cl_wide_df)} clusters)')
print(cl_wide_df[['Cluster','N_Cities','UC_Day_MKTrend','UC_Day_SenSlope','SUHII_Day_MKTrend']].to_string(index=False))

# ── Export styled HTML (gradient on SenSlope columns) ────────────────────────
_slope_cols = [c for c in cl_wide_df.columns if 'SenSlope' in c or 'OLS_R2' in c]
_num_cols   = cl_wide_df.select_dtypes(include='number').columns.tolist()
try:
    _styler = (cl_wide_df.style
               .background_gradient(cmap='RdYlBu_r', subset=_slope_cols,
                                    axis=0, vmin=-0.1, vmax=0.1)
               .format({c: '{:.3f}' for c in _num_cols if c in cl_wide_df.columns})
               .set_caption('Cluster25 LST Wide Summary — Cluster-Level Trend Statistics'))
    _html_path = CL_STAT_DIR / 'Cluster25_LST_Wide_Summary.html'
    _html_path.write_text(_styler.to_html(), encoding='utf-8')
    print(f'✓ {_html_path.name}')
except Exception as _e:
    print(f'HTML export skipped: {_e}')
print(f'\nCluster summary reports → {CL_STAT_DIR}')


---
<a id="nb03-s07d"></a>
### §7d — Publication Composite Grids (4×4)

I draw three A4 grids, one panel per Cluster25 class:

| Grid | Content | Output |
|------|---------|--------|
| Annual LST trend | UC annual LST + Sen slope line | `cluster_annual_trend_4x4_{day|night}.svg` |
| Monthly climatology | Multi-year monthly mean ± 1 SD band | `cluster_monthly_climatology_4x4_{day|night}.svg` |
| SUHII trend | Annual UC − BRA SUHII + Sen slope | `cluster_suhii_trend_4x4_{day|night}.svg` |
All in `Results/03_temporal_analysis/publication/`.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §7d — Publication 4×4 Composite Grids ───────────────────────────────────
# One panel per Cluster25, arranged 4×4 (A4 portrait).
# Each panel split: left = annual UC LST trend, right = monthly climatology.
# Day (solid) and night (dashed) on the same axes; SUHII on secondary y-axis.
# Sen slope dashed-line overlay with tiny in-text annotation.
# Produces one combined SVG: cluster_composite_4x4.svg

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import matplotlib.ticker  as _mticker
import matplotlib.lines   as _mlines

PUB_DIR = RESULTS_03 / 'publication'
PUB_DIR.mkdir(parents=True, exist_ok=True)

_MONTH_TINY  = ['J','F','M','A','M','J','J','A','S','O','N','D']
_FIG_W, _FIG_H = 8.27, 11.69
_SUHII_COL     = '#c0c0c0'
_INT_FMT       = _mticker.FuncFormatter(lambda x, _: f'{int(x):d}')


def _sen_overlay(ax, xs, ys, stats_df, period, zone='urban_core',
                 fs=3.0, ypos=0.07, xpos=0.04, ha='left'):
    # Sen slope dashed line + tiny in-panel annotation.
    if stats_df is None or stats_df.empty:
        return
    _f = stats_df.copy()
    if 'zone'   in _f.columns: _f = _f[_f['zone']   == zone]
    if 'period' in _f.columns: _f = _f[_f['period'] == period]
    if _f.empty: return
    _r   = _f.iloc[0]
    _sd  = _r.get('sen_slope_per_decade', float('nan'))
    _mk  = float(_r.get('mk_p', 1.0))
    _sig = '**' if _mk < 0.01 else ('*' if _mk < 0.05 else 'ns')
    if pd.isna(_sd) or len(xs) < 2: return
    _mx = (xs[0] + xs[-1]) / 2.0
    _my = float(np.mean(ys))
    ax.plot(xs, _my + (_sd / 10.0) * (xs - _mx),
            '--', color='#666666', lw=0.5, alpha=0.65, zorder=3)
    ax.annotate(f'{_sd:+.2f}°/dec{_sig}',
                xy=(xpos, ypos), xycoords='axes fraction',
                fontsize=fs, color='#333333', ha=ha, va='bottom')


# Multi-city clusters only (≥2 cities) for the publication composite grid
_multi_cids_pub = [
    cid for cid in _study_cluster_ids
    if len([c for c in _cluster_to_cities.get(cid, []) if c in _study_cities]) >= 2
]
_NPC_R, _NPC_C = 5, 2   # 9 clusters → 5×2 on A4 portrait

fig = plt.figure(figsize=(_FIG_W, _FIG_H))
fig.suptitle(
    'Cluster25 UC LST (Day — Night) + SUHII  |  Annual trend (left)  &  Monthly climatology (right)',
    fontsize=6.8, fontweight='bold', y=0.997)

outer = GridSpec(_NPC_R, _NPC_C, figure=fig,
                 hspace=0.55, wspace=0.30,
                 top=0.975, bottom=0.040, left=0.065, right=0.975)

for _i, _cid in enumerate(_multi_cids_pub):
    _ri = _i // _NPC_C;  _ci = _i % _NPC_C
    _slug  = f'Cluster25_{_cid:02d}'
    _color = config.CLUSTER25_COLORS.get(_cid, '#888888')
    _n_cl  = len([c for c in _cluster_to_cities.get(_cid, []) if c in _study_cities])

    inner = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[_ri, _ci], wspace=0.06)
    ax_a  = fig.add_subplot(inner[0, 0])
    ax_m  = fig.add_subplot(inner[0, 1])

    # ── Annual: UC LST day + night ────────────────────────────────────────────
    for _per, _ls, _yp, _xp, _ha in [
            ('day',   '-',  0.87, 0.04, 'left'),
            ('night', '--', 0.06, 0.96, 'right')]:
        _s = df_all_cl_annual[
            (df_all_cl_annual['city']   == _slug) &
            (df_all_cl_annual['zone']   == 'urban_core') &
            (df_all_cl_annual['period'] == _per)
        ].sort_values('year').dropna(subset=['median_lst'])
        if _s.empty: continue
        _xi = _s['year'].astype(int).values
        ax_a.plot(_xi, _s['median_lst'].values,
                  color=_color, ls=_ls, lw=0.75, marker='o', ms=1.4, zorder=4)
        _st = df_all_cl_ann_stats[
            (df_all_cl_ann_stats['city']   == _slug) &
            (df_all_cl_ann_stats['zone']   == 'urban_core') &
            (df_all_cl_ann_stats['period'] == _per)
        ]
        _sen_overlay(ax_a, _xi, _s['median_lst'].values, _st, _per,
                     fs=3.0, ypos=_yp, xpos=_xp, ha=_ha)

    # Annual SUHII on secondary y-axis (grey dotted/dash-dot)
    ax_a2 = ax_a.twinx()
    for _per, _ls in [('day', ':'), ('night', '-.')]:
        _sd = df_all_cl_delta_annual[
            (df_all_cl_delta_annual['city']   == _slug) &
            (df_all_cl_delta_annual['period'] == _per)
        ].sort_values('year').dropna(subset=['delta_lst'])
        if not _sd.empty:
            ax_a2.plot(_sd['year'].astype(int).values, _sd['delta_lst'].values,
                      color=_SUHII_COL, ls=_ls, lw=0.55, alpha=0.85, zorder=3)
    ax_a2.tick_params(axis='y', labelsize=2.5, length=1.0, pad=0.8, width=0.3)
    ax_a2.set_ylabel('SUHII', fontsize=2.5, color='#999999', labelpad=1)
    ax_a2.spines['top'].set_visible(False)

    ax_a.xaxis.set_major_formatter(_INT_FMT)
    ax_a.xaxis.set_major_locator(_mticker.MaxNLocator(integer=True, nbins=3, prune='both'))
    ax_a.tick_params(labelsize=3.5, length=1.5, width=0.4)
    ax_a.grid(alpha=0.18, lw=0.3)
    ax_a.spines['top'].set_visible(False)
    ax_a.spines['right'].set_visible(False)

    # ── Monthly: UC LST climatology day + night ───────────────────────────────
    for _per, _ls in [('day', '-'), ('night', '--')]:
        _sm = df_all_cl_monthly[
            (df_all_cl_monthly['city']   == _slug) &
            (df_all_cl_monthly['zone']   == 'urban_core') &
            (df_all_cl_monthly['period'] == _per)
        ]
        if _sm.empty: continue
        _clim = _sm.groupby('month')['median_lst'].mean()
        _std  = _sm.groupby('month')['median_lst'].std().fillna(0)
        _mos  = _clim.index.values
        ax_m.plot(_mos, _clim.values, color=_color, ls=_ls,
                  lw=0.75, marker='o', ms=1.4, zorder=4)
        ax_m.fill_between(_mos, (_clim - _std).values, (_clim + _std).values,
                          color=_color, alpha=0.13, zorder=2)

    # Monthly SUHII climatology on secondary y-axis
    ax_m2 = ax_m.twinx()
    for _per, _ls in [('day', ':'), ('night', '-.')]:
        _sdm = df_all_cl_delta_monthly[
            (df_all_cl_delta_monthly['city']   == _slug) &
            (df_all_cl_delta_monthly['period'] == _per)
        ]
        if not _sdm.empty:
            _cs = _sdm.groupby('month')['delta_lst'].mean()
            ax_m2.plot(_cs.index.values, _cs.values,
                      color=_SUHII_COL, ls=_ls, lw=0.55, alpha=0.85, zorder=3)
    ax_m2.tick_params(axis='y', labelsize=2.5, length=1.0, pad=0.8, width=0.3)
    ax_m2.set_ylabel('SUHII', fontsize=2.5, color='#999999', labelpad=1)
    ax_m2.spines['top'].set_visible(False)

    ax_m.set_xticks(range(1, 13))
    ax_m.set_xticklabels(_MONTH_TINY, fontsize=3.5)
    ax_m.tick_params(axis='y', labelsize=3.5, length=1.5, width=0.4)
    ax_m.grid(alpha=0.18, lw=0.3)
    ax_m.spines['top'].set_visible(False)
    ax_m.spines['right'].set_visible(False)

    # Title centred over both sub-panels
    ax_a.set_title(f'C{_cid}  n={_n_cl}', fontsize=5.0, fontweight='bold', pad=1.5)

    # Y-axis: label + ticks on leftmost column only; suppress elsewhere
    if _ci == 0:
        ax_a.set_ylabel('LST (°C)', fontsize=3.5, labelpad=1)
    else:
        ax_a.set_yticklabels([])
    ax_m.set_yticklabels([])

# ── Hide unused panels ────────────────────────────────────────────────────────
for _j in range(len(_multi_cids_pub), _NPC_R * _NPC_C):
    _inner_j = GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer[_j // _NPC_C, _j % _NPC_C], wspace=0.06)
    for _k in range(2):
        fig.add_subplot(_inner_j[0, _k]).set_visible(False)

# ── Figure-level legend ───────────────────────────────────────────────────────
_leg_h = [
    _mlines.Line2D([], [], color='grey',      ls='-',  lw=1.0, label='UC LST — Day'),
    _mlines.Line2D([], [], color='grey',      ls='--', lw=1.0, label='UC LST — Night'),
    _mlines.Line2D([], [], color=_SUHII_COL, ls=':',  lw=0.9, label='SUHII — Day'),
    _mlines.Line2D([], [], color=_SUHII_COL, ls='-.', lw=0.9, label='SUHII — Night'),
]
fig.legend(handles=_leg_h, loc='lower center', ncol=4,
           fontsize=5.0, framealpha=0.85, bbox_to_anchor=(0.5, 0.002))

_out = PUB_DIR / 'cluster_composite_5x2.svg'
fig.savefig(_out, bbox_inches='tight', dpi=config.FIGURE_DPI)
plt.close(fig)
print(f'  ✓  {_out.name}')
print(f'Publication composite saved → {PUB_DIR}')


---
<a id="nb03-s08"></a>
## Section 8 — National-Level Analysis

I build one Nigerian urban composite: the unweighted mean of the 56 study cities, year by year and
month by month, every city with equal weight. I run the §6a pipeline on it. It describes the typical
study city and abstracts from city-specific signals: averaging lowers the noise, so the national
trend can be significant when most cities' trends are not, and opposite city signals can cancel. I
therefore read it together with the clusters (§7).

| Sub-section | Content |
|-------------|---------|
| [§8a — Composite Series](#nb03-s08a) | Build annual + monthly UC / BRA / CR composite from all 56 cities |
| [§8b — Trend Statistics & SUHII](#nb03-s08b) | Ljung-Box → MK / Hamed-Rao → Sen's slope → OLS R²; SUHII pipeline |
| [§8c — National Composite Chart](#nb03-s08c) | Temporal trend chart (same format as §6c) |
| [§8d — Export](#nb03-s08d) | Append to Wide CSV; Markdown report |
| [§8e — National Cluster Violin](#nb03-s08e) | Distribution of city annual LST across all 16 clusters |

---

> ↑ [Back to Table of Contents](#nb03-toc)

---
<a id="nb03-s08a"></a>
### §8a — National Composite Series

I average the UC, BRA and city-region median LST of all study cities, year by year and month by
month (unweighted).

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── National Urban Composite ────────────────────────────────────────────────────────────────────────
N_CITIES       = len(sorted(df_annual['city'].unique()))
COMPOSITE_NAME = f'Nigerian Cities across {N_CITIES} City-Regions'
COMPOSITE_SLUG = 'National_Composite'

print(f'Building national composite ({N_CITIES} cities)...')

# ── Step 1: Composite annual and monthly series ────────────────────────────
_comp_ann_rows = []
for _zc in ['urban_core', 'bra', 'city_region']:
    for _per in ['day', 'night']:
        _s = df_annual[(df_annual['zone']==_zc) & (df_annual['period']==_per)]
        if _s.empty: continue
        # Build aggregation dict with percentiles if they exist
        _agg = {
            'median_lst':('median_lst', 'mean'),
            'mean_lst':  ('mean_lst',   'mean') if 'mean_lst' in _s.columns else ('median_lst','mean'),
            'min_lst':   ('min_lst',    'min')  if 'min_lst'  in _s.columns else ('median_lst','min'),
            'max_lst':   ('max_lst',    'max')  if 'max_lst'  in _s.columns else ('median_lst','max'),
            'lst_std':   ('median_lst', 'std'),
            'n_cities':  ('median_lst', 'count'),
        }
        if 'p05_lst' in _s.columns:
            _agg['p05_lst'] = ('p05_lst', 'mean')
        if 'p95_lst' in _s.columns:
            _agg['p95_lst'] = ('p95_lst', 'mean')
        _g = _s.groupby('year').agg(**_agg).reset_index()
        _g['city']   = COMPOSITE_SLUG
        _g['zone']   = _zc
        _g['period'] = _per
        _comp_ann_rows.append(_g)
df_comp_annual = pd.concat(_comp_ann_rows, ignore_index=True) if _comp_ann_rows else pd.DataFrame()

_comp_mon_rows = []
for _zc in ['urban_core', 'bra', 'city_region']:
    for _per in ['day', 'night']:
        _s = df_monthly[(df_monthly['zone']==_zc) & (df_monthly['period']==_per)]
        if _s.empty: continue
        # Build aggregation dict with percentiles if they exist
        _agg = {
            'median_lst':('median_lst', 'mean'),
            'mean_lst':  ('mean_lst',   'mean') if 'mean_lst' in _s.columns else ('median_lst','mean'),
            'min_lst':   ('min_lst',    'min')  if 'min_lst'  in _s.columns else ('median_lst','min'),
            'max_lst':   ('max_lst',    'max')  if 'max_lst'  in _s.columns else ('median_lst','max'),
            'lst_std':   ('median_lst', 'std'),
            'n_cities':  ('median_lst', 'count'),
        }
        if 'p05_lst' in _s.columns:
            _agg['p05_lst'] = ('p05_lst', 'mean')
        if 'p95_lst' in _s.columns:
            _agg['p95_lst'] = ('p95_lst', 'mean')
        _g = _s.groupby(['year','month']).agg(**_agg).reset_index()
        _g['city']   = COMPOSITE_SLUG
        _g['zone']   = _zc
        _g['period'] = _per
        _comp_mon_rows.append(_g)
df_comp_monthly = pd.concat(_comp_mon_rows, ignore_index=True) if _comp_mon_rows else pd.DataFrame()

# Delta: reuse _compute_delta from Section 7 (already in scope)
delta_comp_annual  = _compute_delta(df_comp_annual,  ['year'])
delta_comp_monthly = _compute_delta(df_comp_monthly, ['year', 'month'])

print(f'  Annual composite  : {len(df_comp_annual)} rows')
print(f'  Monthly composite : {len(df_comp_monthly)} rows')

---
<a id="nb03-s08b"></a>
### §8b — Trend Statistics & SUHII

I compute the national SUHII (UC − BRA) from the composite means and run the §6a pipeline on the
annual and monthly UC, BRA and SUHII series. The statistics come from the composite series, not from
averaging the city statistics.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── Step 2: Trend analysis on composite series ─────────────────────────────
_ca_rows, _cm_rows, _cda_rows, _cdm_rows = [], [], [], []

for _zc in ['urban_core', 'bra', 'city_region']:
    for _per in ['day', 'night']:
        _s = df_comp_annual[(df_comp_annual['zone']==_zc) & (df_comp_annual['period']==_per)]
        if _s.empty: continue
        _gs = _s.sort_values('year').set_index('year')['median_lst']
        _res = run_trend_analysis(_gs, mode='annual')
        _hi  = _gs.idxmax() if not _gs.dropna().empty else None
        _lo  = _gs.idxmin() if not _gs.dropna().empty else None
        _ca_rows.append({'city': COMPOSITE_SLUG, 'zone': _zc, 'period': _per,
                         'highest_year': _hi,
                         'highest_year_lst': round(float(_gs.max()),4) if _hi else None,
                         'lowest_year': _lo,
                         'lowest_year_lst': round(float(_gs.min()),4) if _lo else None,
                         **_res})

for _zc in ['urban_core', 'bra', 'city_region']:
    for _per in ['day', 'night']:
        _s = df_comp_monthly[(df_comp_monthly['zone']==_zc) & (df_comp_monthly['period']==_per)]
        if _s.empty: continue
        _ord = _s.sort_values(['year','month'])
        _idx = pd.to_datetime(_ord.apply(lambda r: f'{int(r["year"])}-{int(r["month"]):02d}', axis=1))
        _gs  = _ord.set_index(_idx)['median_lst']
        _res = run_trend_analysis(_gs, mode='monthly')
        _clim = _s.groupby('month')['median_lst'].mean()
        _amp  = round(float(_clim.max()-_clim.min()),4) if len(_clim)==12 else None
        _hm   = int(_clim.idxmax()) if not _clim.empty else None
        _lm   = int(_clim.idxmin()) if not _clim.empty else None
        _cm_rows.append({'city': COMPOSITE_SLUG, 'zone': _zc, 'period': _per,
                         'monthly_amplitude': _amp,
                         'highest_month': _hm, 'highest_month_lst': round(float(_clim.max()),4) if _hm else None,
                         'lowest_month':  _lm, 'lowest_month_lst':  round(float(_clim.min()),4) if _lm else None,
                         **_res})

for _per in ['day', 'night']:
    _s  = delta_comp_annual[delta_comp_annual['period']==_per]
    _gs = _s.sort_values('year').set_index('year')['delta_lst'].dropna()
    if len(_gs) < 5: continue
    _res = run_trend_analysis(_gs, mode='annual')
    _hi  = _gs.idxmax() if not _gs.empty else None
    _lo  = _gs.idxmin() if not _gs.empty else None
    _cda_rows.append({'city': COMPOSITE_SLUG, 'zone': 'delta_uc_bra', 'period': _per,
                      'highest_year': _hi,
                      'highest_year_lst': round(float(_gs.max()),4) if _hi else None,
                      'lowest_year': _lo,
                      'lowest_year_lst': round(float(_gs.min()),4) if _lo else None,
                      **_res})

for _per in ['day', 'night']:
    _s   = delta_comp_monthly[delta_comp_monthly['period']==_per]
    _ord = _s.sort_values(['year','month'])
    _idx = pd.to_datetime(_ord.apply(lambda r: f'{int(r["year"])}-{int(r["month"]):02d}', axis=1))
    _gs  = _ord.set_index(_idx)['delta_lst'].dropna()
    if len(_gs) < 5: continue
    _res  = run_trend_analysis(_gs, mode='monthly')
    _clim = _s.groupby('month')['delta_lst'].mean()
    _amp  = round(float(_clim.max()-_clim.min()),4) if len(_clim)==12 else None
    _cdm_rows.append({'city': COMPOSITE_SLUG, 'zone': 'delta_uc_bra', 'period': _per,
                      'monthly_amplitude': _amp, **_res})

df_comp_ann_stats  = pd.DataFrame(_ca_rows)
df_comp_mon_stats  = pd.DataFrame(_cm_rows)
df_comp_delta_ann  = pd.DataFrame(_cda_rows)
df_comp_delta_mon  = pd.DataFrame(_cdm_rows)

print(f'  Annual stats  : {len(df_comp_ann_stats)} rows')
print(f'  Monthly stats : {len(df_comp_mon_stats)} rows')

---
<a id="nb03-s08c"></a>
### §8c — National Composite Chart

I draw the national chart with the same `create_temporal_chart()` as the city charts (§6c).

**Output:** `Results/03_temporal_analysis/National_Composite/figures/national_composite_temporal_trend.svg`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── Step 3: Composite chart (same format as per-city) ────────────────────────
_comp_fig_dir  = RESULTS_03 / COMPOSITE_SLUG / 'figures'
_comp_fig_dir.mkdir(parents=True, exist_ok=True)
_comp_fig_path = _comp_fig_dir / 'national_composite_temporal_trend.svg'

try:
    _comp_fig = create_temporal_chart(
        city          = COMPOSITE_NAME,
        df_ann        = df_comp_annual,
        df_mon        = df_comp_monthly,
        df_del_ann    = delta_comp_annual,
        df_del_mon    = delta_comp_monthly,
        ann_stats     = df_comp_ann_stats,
        mon_stats     = df_comp_mon_stats,
        del_ann_stats = df_comp_delta_ann,
        del_mon_stats = df_comp_delta_mon,
        save_path     = _comp_fig_path,
        dpi           = config.FIGURE_DPI,
    )
    plt.close(_comp_fig)
    print(f'  ✓ Composite chart → {_comp_fig_path.name}')
except Exception as _e:
    print(f'  ⚠ Composite chart failed: {_e}')

---
<a id="nb03-s08d"></a>
### §8d — Export: Wide CSV & Markdown

I add the national row to `Stats/City_LST_Wide_Summary.csv`, in the city columns, and write
`Stats/National_Composite/National_Composite_lst_summary.md`.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── Step 4: Append composite row to wide CSV ───────────────────────────────
_nan = float('nan')
comp_row = {'City': COMPOSITE_NAME}

for _zl, _zc, _ast in [
    ('UC',    'urban_core',   df_comp_ann_stats),
    ('BRA',   'bra',          df_comp_ann_stats),
    ('SUHII', 'delta_uc_bra', df_comp_delta_ann),
]:
    for _per in ['day', 'night']:
        _pfx  = f'{_zl}_{_per.title()}'
        _sdf  = _ast[(_ast['zone']==_zc) & (_ast['period']==_per)]
        _sr   = _sdf.iloc[0].to_dict() if not _sdf.empty else None
        if _zc == 'delta_uc_bra':
            _vals = delta_comp_annual[delta_comp_annual['period']==_per]['delta_lst']
        else:
            _vals = df_comp_annual[(df_comp_annual['zone']==_zc) & (df_comp_annual['period']==_per)]['median_lst']
        _ml = round(float(_vals.mean()),3) if not _vals.empty else _nan
        _mdf = df_comp_mon_stats[(df_comp_mon_stats['zone']==_zc) & (df_comp_mon_stats['period']==_per)] \
               if _zc != 'delta_uc_bra' else pd.DataFrame()
        _mr  = _mdf.iloc[0].to_dict() if not _mdf.empty else None
        comp_row[f'{_pfx}_MeanLST']     = _ml
        comp_row[f'{_pfx}_SenSlope']    = _sr.get('sen_slope_per_year',   _nan) if _sr else _nan
        comp_row[f'{_pfx}_SlopeDec']    = _sr.get('sen_slope_per_decade', _nan) if _sr else _nan
        comp_row[f'{_pfx}_OLS_R2']      = _sr.get('ols_r_squared',        _nan) if _sr else _nan
        comp_row[f'{_pfx}_OLS_p']       = _sr.get('ols_p_value',          _nan) if _sr else _nan
        comp_row[f'{_pfx}_MKTrend']     = _sr.get('mk_trend', '')              if _sr else ''
        comp_row[f'{_pfx}_MKp']         = _sr.get('mk_p',     _nan)             if _sr else _nan
        comp_row[f'{_pfx}_HRcorrected'] = bool(_sr.get('autocorr_corrected', False)) if _sr else False
        comp_row[f'{_pfx}_WarmingRate'] = _fmt_rate(_sr)
        if _mr:
            _hi_mo = _mr.get('highest_month'); _amp_v = _mr.get('monthly_amplitude')
            try:    comp_row[f'{_pfx}_HottestMonth'] = _MONTH_NAMES.get(int(_hi_mo), str(_hi_mo)) \
                                                        if _hi_mo and not _np.isnan(float(_hi_mo)) else 'N/A'
            except: comp_row[f'{_pfx}_HottestMonth'] = 'N/A'
            try:    comp_row[f'{_pfx}_SeasonalAmp'] = round(float(_amp_v),3) \
                                                       if _amp_v and not _np.isnan(float(_amp_v)) else _nan
            except: comp_row[f'{_pfx}_SeasonalAmp'] = _nan
        else:
            comp_row[f'{_pfx}_HottestMonth'] = 'N/A'
            comp_row[f'{_pfx}_SeasonalAmp']  = _nan

_wide_path = STATS_DIR / 'City_LST_Wide_Summary.csv'
_wdf = pd.read_csv(_wide_path) if _wide_path.exists() else wide_df.copy()
if COMPOSITE_NAME not in _wdf['City'].values:
    _wdf = pd.concat([_wdf, pd.DataFrame([comp_row])], ignore_index=True)
else:
    _idx = _wdf.index[_wdf['City']==COMPOSITE_NAME][0]
    for _k, _v in comp_row.items(): _wdf.at[_idx, _k] = _v
_wdf.to_csv(_wide_path, index=False)
print(f'  ✓ Composite row appended to {_wide_path.name}')

# ── Step 5: Composite Markdown ────────────────────────────────────────────────────────
_cdir_comp = STATS_DIR / COMPOSITE_SLUG
_cdir_comp.mkdir(parents=True, exist_ok=True)

_cl = [
    f'# {COMPOSITE_NAME} — LST Temporal Analysis Summary', '',
    f'**Generated:** {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M UTC")}  ',
    f'**Source version:** `{SOURCE_VERSION}`  ',
    f'**Period:** MODIS MYD11A1, {_yr_rng} (daily acquisitions)  ',
    f'**Cities included:** {N_CITIES}  ',
    '**Aggregation:** Unweighted mean across city UC / BRA / CR annual series  ',
    '', '---', '',
    '> **Note:** MK trend, Sen slope, and OLS R² are computed on the composite mean series',
    '> directly — not averaged from per-city statistics. A significant composite trend means',
    '> the average Nigerian urban signal is trending; individual cities may diverge.', '',
]
for _zn, _zk in _ZONE_LABELS:
    _cl += [f'## {_zn}', '']
    for _pl, _pk in [('Day-time (≥13:30 local)', 'Day'), ('Night-time (≥01:30 local)', 'Night')]:
        _pfx  = f'{_zk}_{_pk}'
        _sym  = '↑' if comp_row.get(f'{_pfx}_MKTrend')=='increasing' \
                else ('↓' if comp_row.get(f'{_pfx}_MKTrend')=='decreasing' else '—')
        _hrnote = '  *(Hamed-Rao autocorrelation correction)*' \
                  if comp_row.get(f'{_pfx}_HRcorrected') else ''
        _cl += [
            f'### {_pl}', '',
            '| Metric | Value |',
            '|--------|-------|',
            f'| Mean composite LST  | {_safe(comp_row.get(f"{_pfx}_MeanLST"))} °C |',
            f'| Warming rate        | {comp_row.get(f"{_pfx}_WarmingRate", "N/A")} |',
            f'| Decadal rate        | {_safe(comp_row.get(f"{_pfx}_SlopeDec"), "+.3f")} °C/decade |',
            f'| MK trend            | {_sym} {comp_row.get(f"{_pfx}_MKTrend","")} '
            f'(p = {_safe(comp_row.get(f"{_pfx}_MKp"), ".4f")}){_hrnote} |',
        ]
        if _zk != 'SUHII':
            _cl += [
                f'| Hottest month       | {comp_row.get(f"{_pfx}_HottestMonth", "N/A")} |',
                f'| Seasonal amplitude  | {_safe(comp_row.get(f"{_pfx}_SeasonalAmp"))} °C |',
            ]
        _cl += ['']

_cl += [
    '---', '', '## Statistical Notes',
    '- Trend analysis applied directly to composite mean series (not averaged from city-level stats).',
    '- **Composite SUHII** = composite UC median LST − composite BRA median LST, computed year-wise before trend analysis.',
    '- **H&R (Hamed & Rao, 1998)**: when Ljung-Box p<0.05 detects serial autocorrelation, standard MK variance is inflated. H&R adjusts the effective sample size for a more conservative (lower) p-value. Marked [H&R] in results.',
    '',
]

(_cdir_comp / f'{COMPOSITE_SLUG}_lst_summary.md').write_text('\n'.join(_cl), encoding='utf-8')
print(f'  ✓ Composite Markdown → {COMPOSITE_SLUG}/{COMPOSITE_SLUG}_lst_summary.md')
print(f'\nNational composite complete → {RESULTS_03 / COMPOSITE_SLUG}')

---
<a id="nb03-s08e"></a>
### §8e — National Cluster Comparison Violin

I compare the spread of the cities' annual mean urban-core LST across all 16 Cluster25 classes, all
years pooled, day and night side by side — showing which classes vary more or less between cities.

**Output:** `Results/03_temporal_analysis/publication/violins/national_cluster_violin.svg`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §8e — National Cluster Comparison Violin ─────────────────────────────────
# One violin per cluster (all years pooled) — distribution of city annual UC LST.
# Clusters sorted in decreasing order of mean LST (independently per period).

import numpy as np

PUB_VIOLIN_DIR = RESULTS_03 / 'publication' / 'violins'
PUB_VIOLIN_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

# Multi-city filter for national violin
_multi_cids_nat = [
    cid for cid in _study_cluster_ids
    if len([c for c in _cluster_to_cities.get(cid, []) if c in _study_cities]) >= 3
]

for _pi, period in enumerate(['day', 'night']):
    ax = axes[_pi]
    _df_uc = (df_annual[(df_annual['zone'] == 'urban_core') &
                        (df_annual['period'] == period)].copy())
    _df_uc['cluster'] = _df_uc['city'].map(_city_to_cluster)

    # Compute mean LST per multi-city cluster and sort descending
    _cl_means = {}
    for _cid in _multi_cids_nat:
        _v = _df_uc[_df_uc['cluster'] == _cid]['median_lst'].dropna().values
        _cl_means[_cid] = float(np.mean(_v)) if len(_v) > 0 else 0.0
    _sorted_cids = sorted(_multi_cids_nat,
                          key=lambda c: _cl_means[c], reverse=True)

    _x_labels = []
    for _xi, _cid in enumerate(_sorted_cids, start=1):
        _vals  = _df_uc[_df_uc['cluster'] == _cid]['median_lst'].dropna().values
        _color = config.CLUSTER25_COLORS.get(_cid, '#888888')
        _n_c   = len([c for c in _cluster_to_cities.get(_cid, [])
                      if c in _study_cities])
        _sd_xe  = float(np.std(_vals)) if len(_vals) >= 2 else float('nan')
        _sd_xe_s = f'\nσ={_sd_xe:.1f}°C' if not np.isnan(_sd_xe) else ''
        _x_labels.append(f'C{_cid}\n(n={_n_c}){_sd_xe_s}')

        if len(_vals) >= 3 and np.std(_vals) > 0:
            try:
                parts = ax.violinplot([_vals], positions=[_xi],
                                      widths=0.65,
                                      showmeans=True, showmedians=False,
                                      showextrema=True)
                for pc in parts['bodies']:
                    pc.set_facecolor(_color); pc.set_alpha(0.75)
                for key in ['cmeans', 'cmins', 'cmaxes', 'cbars']:
                    if key in parts:
                        parts[key].set_color(_color)
                        parts[key].set_linewidth(0.9)
            except Exception:
                ax.scatter([_xi] * len(_vals), _vals,
                           c=_color, s=15, zorder=5, alpha=0.8)
        elif len(_vals) > 0:
            ax.scatter([_xi] * len(_vals), _vals,
                       c=_color, s=15, zorder=5, alpha=0.8)

    ax.set_xticks(range(1, len(_sorted_cids) + 1))
    ax.set_xticklabels(_x_labels, fontsize=6.5)
    ax.set_ylabel('UC LST (°C)', fontsize=9)
    ax.set_title(f'{period.capitalize()}time — City Annual Mean UC LST\n'
                 f'(clusters sorted by decreasing mean)', fontsize=9.5)
    ax.grid(axis='y', alpha=0.25, linewidth=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0.3, len(_sorted_cids) + 0.7)

fig.suptitle(
    'National Cluster Comparison — UC Annual LST Distribution (All Years Pooled)',
    fontsize=11, fontweight='bold')
plt.tight_layout()

_out = PUB_VIOLIN_DIR / 'national_cluster_violin.svg'
fig.savefig(_out, bbox_inches='tight', dpi=config.FIGURE_DPI)
plt.close(fig)
print(f'  ✓  {_out.name}')
print(f'National cluster violin saved → {_out}')


---
<a id="nb03-s09"></a>
## Section 9 — Cross-City Comparison Heatmaps

I draw the heatmaps on the same city → cluster → national levels, in eight configurations each
(UC or SUHII × yearly or monthly × day or night):

| Sub-section | Rows | Content |
|-------------|------|---------|
| [§9a — City-Level Heatmaps](#nb03-s09a) | 56 cities (standard) / 21 rows (publication) | City rows labelled with `↑/↓ CityName *** [C{N}]` — MK trend + Cluster25 class |
| [§9b — Cluster-Level Heatmaps](#nb03-s09b) | One row per Cluster25 composite (≥ 2 cities) + National Composite | Cluster rows; no MK arrows (composite MK reported in §7a) |
The publication heatmaps (§9a) have 21 rows: the ten hottest cities, the national composite as the
reference in the middle, and the ten coolest.

---

> ↑ [Back to Table of Contents](#nb03-toc)

---
<a id="nb03-s09a"></a>
### §9a — City-Level Heatmaps

I sort all 56 cities by mean LST. An annual row reads `↑ CityName *** [C13]`: the Mann–Kendall
direction and stars (§6a) and the Cluster25 class (NB01). A monthly row reads `CityName [C13]`,
without the trend marks, since monthly heatmaps show the seasons. The publication version (quarter A4)
keeps the most contrasting cities:

| Row group | Rows | Sort |
|-----------|------|------|
| 10 hottest cities | 1–10 | Hottest → 10th hottest |
| **National Composite** | 11 | Unweighted mean across all 56 cities |
| 10 coolest cities | 12–21 | Warmest-of-cool → coldest |
**Output:** `heatmaps/LST_{config}_Heatmap_MK.svg` (all cities) and
`heatmaps/publication/LST_{config}_Pub_Heatmap_MK.svg` (21 rows), in `Results/03_temporal_analysis/`.

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── Load Cluster25 city lookup (built in NB01 Section 8) ─────────────────────
_cluster_lookup_path = RESULTS_03.parent / '01_aoi_delineation' / 'city_cluster25_lookup.csv'
if _cluster_lookup_path.exists():
    _cl_lkp = pd.read_csv(_cluster_lookup_path)
    CITY_CLUSTER_LOOKUP = dict(zip(_cl_lkp['city_name'], _cl_lkp['Cluster25'].astype(int)))
    print(f'  Cluster25 lookup loaded: {len(CITY_CLUSTER_LOOKUP)} cities')
else:
    print(f'  [WARN] Cluster lookup not found; heatmap labels will omit [C{{N}}]')
    CITY_CLUSTER_LOOKUP = {}

from src.visualization import create_single_page_heatmap

HM_DIR = RESULTS_03 / 'heatmaps'
HM_DIR.mkdir(parents=True, exist_ok=True)

# A4 dimensions: annual = half portrait, monthly = half landscape
A4_W, A4_H = 8.27, 11.69   # inches
YR_W, YR_H = A4_W,    A4_H / 2   # ~8.27 x 5.85
MO_W, MO_H = A4_H / 2, A4_W     # ~5.85 x 8.27


def _pivot_uc(period, time_dim):
    # Cities x time matrix of UC median LST
    col = 'year' if time_dim == 'year' else 'month'
    src = df_annual if time_dim == 'year' else df_monthly
    sub = src[(src['zone'] == 'urban_core') & (src['period'] == period)]
    mat = sub.pivot_table(index='city', columns=col,
                          values='median_lst', aggfunc='median')
    return mat.dropna(how='all')


def _pivot_delta(period, time_dim):
    # Cities x time matrix of Delta (UC-BRA) LST
    col = 'year' if time_dim == 'year' else 'month'
    src = delta_annual if time_dim == 'year' else delta_monthly
    sub = src[src['period'] == period]
    mat = sub.pivot_table(index='city', columns=col,
                          values='delta_lst', aggfunc='median')
    return mat.dropna(how='all')


def _mk_dict_from_stats(stats_df, zone_filter, period):
    # Build {city: (trend_str, p_val)} dict from pre-computed corrected MK stats.
    # trend_str is 'increasing' or 'decreasing' when p<0.05, else ''.
    if 'zone' in stats_df.columns:
        sub = stats_df[(stats_df['zone'] == zone_filter) &
                       (stats_df['period'] == period)]
    else:
        sub = stats_df[stats_df['period'] == period]
    result = {}
    for _, row in sub.iterrows():
        trend = str(row.get('mk_trend', ''))
        p_val = float(row.get('mk_p', 1.0))
        direction = trend if (p_val < 0.05 and trend in ('increasing', 'decreasing')) else ''
        result[row['city']] = (direction, p_val)
    return result


# 8 heatmap configurations
hm_configs = [
    # (file_suffix,           pivot_fn,     time_dim, period, fw,   fh,   mk_zone)
    ('UC_Yearly_Day',        _pivot_uc,    'year',  'day',   YR_W, YR_H, 'urban_core'),
    ('UC_Yearly_Night',      _pivot_uc,    'year',  'night', YR_W, YR_H, 'urban_core'),
    ('UC_Monthly_Day',       _pivot_uc,    'month', 'day',   MO_W, MO_H, None),
    ('UC_Monthly_Night',     _pivot_uc,    'month', 'night', MO_W, MO_H, None),
    ('Delta_Yearly_Day',     _pivot_delta, 'year',  'day',   YR_W, YR_H, 'delta_uc_bra'),
    ('Delta_Yearly_Night',   _pivot_delta, 'year',  'night', YR_W, YR_H, 'delta_uc_bra'),
    ('Delta_Monthly_Day',    _pivot_delta, 'month', 'day',   MO_W, MO_H, None),
    ('Delta_Monthly_Night',  _pivot_delta, 'month', 'night', MO_W, MO_H, None),
]

print('Generating cross-city heatmaps...')
for fname, pivot_fn, time_dim, period, fw, fh, mk_zone in hm_configs:
    out_path = str(HM_DIR / f'LST_{fname}_Heatmap_MK.svg')

    mat = pivot_fn(period, time_dim)
    if mat.empty:
        print(f'  [SKIP] {fname} -- no data')
        continue

    # MK significance: annual panels use pre-computed corrected stats; monthly = none
    if time_dim == 'year' and mk_zone is not None:
        stats_src = (df_annual_stats if mk_zone == 'urban_core'
                     else df_delta_annual_stats)
        mk_sig = _mk_dict_from_stats(stats_src, mk_zone, period)
    else:
        mk_sig = {}

    # lst_time label shown in the chart title
    grp_label = 'UC' if 'uc' in fname.lower() else 'SUHII (UC\u2212BRA)'
    lst_label  = f'{grp_label} {period}'

    create_single_page_heatmap(
        data=mat, mk_sig=mk_sig,
        lst_time=lst_label, time_dim=time_dim,
        fig_w=fw, fig_h=fh,
        output_path=out_path,
        cluster_lookup=CITY_CLUSTER_LOOKUP,
    )

print(f'\nAll 8 heatmaps saved to: {HM_DIR}')

In [ ]:
# ── Publication heatmaps (top-10 hottest + top-10 coolest + composite) ───────
PUB_HM_DIR = RESULTS_03 / 'heatmaps' / 'publication'
PUB_HM_DIR.mkdir(parents=True, exist_ok=True)

# Sized to fit 4 per A4 page (~quarter A4 each)
PUB_YR_W, PUB_YR_H = 4.5, 6.0   # yearly  (20 year-columns × 21 rows)
PUB_MO_W, PUB_MO_H = 4.2, 6.0   # monthly (12 month-columns × 21 rows)

COMP_LABEL = 'National Composite'   # row label in publication heatmap


def _comp_pivot_uc(period, time_dim):
    """Composite UC LST as a single-row DataFrame (index = COMP_LABEL)."""
    col = 'year' if time_dim == 'year' else 'month'
    src = df_comp_annual if time_dim == 'year' else df_comp_monthly
    sub = src[(src['zone'] == 'urban_core') & (src['period'] == period)]
    row = sub.pivot_table(index='city', columns=col,
                          values='median_lst', aggfunc='median')
    row.index = [COMP_LABEL]
    return row


def _comp_pivot_delta(period, time_dim):
    """Composite Delta LST as a single-row DataFrame (index = COMP_LABEL)."""
    col = 'year' if time_dim == 'year' else 'month'
    src = delta_comp_annual if time_dim == 'year' else delta_comp_monthly
    sub = src[src['period'] == period]
    row = sub.pivot_table(index='city', columns=col,
                          values='delta_lst', aggfunc='median')
    row.index = [COMP_LABEL]
    return row


def _pub_subset(mat, comp_row, n_top=10):
    """Return 21-row matrix:
       top-10 hottest (hottest at top)
       | National Composite (middle)
       | top-10 coolest (warmest-of-cool at top, coldest at bottom)

    Separator lines should be drawn after rows 9 and 10 (0-based).
    """
    if mat.empty:
        return mat
    means    = mat.mean(axis=1).sort_values(ascending=False)
    hot_idx  = means.head(n_top).index.tolist()   # hottest -> 10th hottest
    # cool group: tail() is already ascending (47th->56th in desc-sorted means)
    # so tolist() gives warmest-of-cool -> coldest — hottest-first within group
    cool_idx = means.tail(n_top).index.tolist()
    sub_mat  = mat.loc[hot_idx + cool_idx]
    comp_aligned = comp_row.reindex(columns=mat.columns)
    # Insert composite between hot and cool groups
    return pd.concat([mat.loc[hot_idx], comp_aligned, mat.loc[cool_idx]])


print('Generating publication heatmaps (top-10 hottest + coolest + composite)...')
for fname, pivot_fn, time_dim, period, _fw, _fh, mk_zone in hm_configs:
    mat = pivot_fn(period, time_dim)
    if mat.empty:
        print(f'  [SKIP] {fname} -- no data')
        continue

    # Build composite pivot row
    comp_row = (_comp_pivot_delta(period, time_dim)
                if 'delta' in fname.lower()
                else _comp_pivot_uc(period, time_dim))

    pub_mat = _pub_subset(mat, comp_row, n_top=10)

    # MK significance: city-level plus composite row
    if time_dim == 'year' and mk_zone is not None:
        stats_src = (df_annual_stats if mk_zone == 'urban_core'
                     else df_delta_annual_stats)
        mk_sig_pub = _mk_dict_from_stats(stats_src, mk_zone, period)
        # Add composite MK significance
        _c_src = (df_comp_ann_stats if mk_zone == 'urban_core' else df_comp_delta_ann)
        _cs = _c_src[(_c_src['zone'] == mk_zone) & (_c_src['period'] == period)]
        if not _cs.empty:
            _cr = _cs.iloc[0]
            _ct = str(_cr.get('mk_trend', ''))
            _cp = float(_cr.get('mk_p', 1.0))
            _cd = _ct if (_cp < 0.05 and _ct in ('increasing', 'decreasing')) else ''
            mk_sig_pub[COMP_LABEL] = (_cd, _cp)
    else:
        mk_sig_pub = {}

    fw = PUB_YR_W if time_dim == 'year' else PUB_MO_W
    fh = PUB_YR_H if time_dim == 'year' else PUB_MO_H
    grp_label = 'UC' if 'uc' in fname.lower() else 'SUHII (UC−BRA)'
    lst_label  = f'{grp_label} {period}'

    out_path = str(PUB_HM_DIR / f'LST_{fname}_Pub_Heatmap_MK.svg')
    create_single_page_heatmap(
        data=pub_mat, mk_sig=mk_sig_pub,
        lst_time=lst_label, time_dim=time_dim,
        fig_w=fw, fig_h=fh,
        output_path=out_path,
        pre_sorted=True,
        separator_after_rows=[9, 10],   # after hot group (row 9) and composite (row 10)
        cluster_lookup=CITY_CLUSTER_LOOKUP,
    )

print()
print("All 8 publication heatmaps saved to:", PUB_HM_DIR)

---
<a id="nb03-s09b"></a>
### §9b — Cluster-Level Heatmaps

I draw one row per Cluster25 composite with at least two cities (`C{NN} [N cities]`) and the national
composite as the last row, after a separator. The cluster rows carry no trend arrows; their
Mann–Kendall and Sen statistics are in §7a.

**Output:** `Results/03_temporal_analysis/heatmaps/cluster_level/LST_{config}_Cluster_Heatmap.svg`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §9b — Cluster-Level Heatmaps ─────────────────────────────────────────────
# One row per Cluster25 composite (≥ 2 cities).
# National Composite appended as the final row.
# Same 8 configurations as §9a; same cluster labels on row axis.

CL_HM_DIR = RESULTS_03 / 'heatmaps' / 'cluster_level'
CL_HM_DIR.mkdir(parents=True, exist_ok=True)

# ── Build cluster composite pivot matrices ─────────────────────────────────────
def _cl_pivot(zone_col, series_col, period, time_dim, cluster_ids, cluster_to_cities):
    """Average per-cluster across member cities, return DataFrame rows = cluster labels."""
    col = 'year' if time_dim == 'year' else 'month'
    src = (df_annual if time_dim == 'year' else df_monthly) if zone_col != 'delta'           else (delta_annual if time_dim == 'year' else delta_monthly)
    rows = {}
    for cid in sorted(cluster_ids):
        avail = [c for c in cluster_to_cities.get(cid, []) if c in _study_cities]
        if len(avail) < 1:
            continue
        if zone_col == 'delta':
            sub = src[(src['city'].isin(avail)) & (src['period'] == period)]
            grp = sub.groupby(col)[series_col].mean()
        else:
            sub = src[(src['city'].isin(avail)) &
                      (src['zone'] == zone_col) &
                      (src['period'] == period)]
            grp = sub.groupby(col)[series_col].mean()
        if grp.empty:
            continue
        n = len(avail)
        label = f"C{cid:02d} [{n} {'city' if n == 1 else 'cities'}]"
        rows[label] = grp
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).T

_study_cities = set(df_annual['city'].unique())

# Reuse cluster lookup built in §7a
_cl_ids   = sorted(_cluster_to_cities.keys())

# Cluster MK stats: build from per-city delta_annual_stats (aggregate isn't pre-stored)
# We use the visual comparison only — no MK arrows on cluster rows (composite MK is in §7a)
# National composite row appended at bottom
def _nat_pivot_row(zone_col, series_col, period, time_dim):
    col = 'year' if time_dim == 'year' else 'month'
    if zone_col == 'delta':
        src = delta_comp_annual if time_dim == 'year' else delta_comp_monthly
        sub = src[src['period'] == period]
    else:
        src = df_comp_annual if time_dim == 'year' else df_comp_monthly
        sub = src[(src['zone'] == zone_col) & (src['period'] == period)]
    row = sub.groupby(col)[series_col].mean()
    df_row = pd.DataFrame([row])
    df_row.index = ['National Composite']
    return df_row

cl_hm_configs = [
    # (suffix,               zone_col,      series_col,  time_dim, period)
    ('UC_Yearly_Day',       'urban_core',  'median_lst', 'year',  'day'),
    ('UC_Yearly_Night',     'urban_core',  'median_lst', 'year',  'night'),
    ('UC_Monthly_Day',      'urban_core',  'median_lst', 'month', 'day'),
    ('UC_Monthly_Night',    'urban_core',  'median_lst', 'month', 'night'),
    ('Delta_Yearly_Day',    'delta',       'delta_lst',  'year',  'day'),
    ('Delta_Yearly_Night',  'delta',       'delta_lst',  'year',  'night'),
    ('Delta_Monthly_Day',   'delta',       'delta_lst',  'month', 'day'),
    ('Delta_Monthly_Night', 'delta',       'delta_lst',  'month', 'night'),
]

print('Generating cluster-level heatmaps...')
for fname, zone_col, series_col, time_dim, period in cl_hm_configs:
    cl_mat = _cl_pivot(zone_col, series_col, period, time_dim, _cl_ids, _cluster_to_cities)
    nat_row = _nat_pivot_row(zone_col, series_col, period, time_dim)
    if cl_mat.empty:
        print(f'  [SKIP] {fname} -- no cluster data')
        continue

    # Sort clusters hottest → coolest (mean across all time steps), national composite stays last
    cl_mat = cl_mat.loc[cl_mat.mean(axis=1).sort_values(ascending=False).index]

    # Append national composite as final row
    nat_aligned = nat_row.reindex(columns=cl_mat.columns)
    full_mat = pd.concat([cl_mat, nat_aligned])

    grp_label = 'UC' if 'uc' in fname.lower() else 'SUHII (UC−BRA)'
    lst_label  = f'{grp_label} {period} — Cluster Composites'
    fw = YR_W if time_dim == 'year' else MO_W
    fh = max(len(full_mat) * 0.38 + 1.2, 4.0)  # auto height by row count

    # ── Build MK significance dict from §7a cluster stats ────────────────────
    # Annual heatmaps → df_all_cl_ann_stats  (MK / Hamed-Rao)
    # Monthly heatmaps → df_all_cl_mon_stats (Seasonal MK)
    # Cluster slug in stats: 'Cluster25_01'  →  heatmap label: 'C01 [3 cities]'
    _is_uc    = 'uc' in fname.lower()
    _is_year  = time_dim == 'year'
    _zone_flt = 'urban_core' if _is_uc else 'delta_uc_bra'
    if _is_year:
        _stats_src = df_all_cl_ann_stats   if _is_uc else df_all_cl_delta_stats
    else:
        _stats_src = df_all_cl_mon_stats   if _is_uc else df_all_cl_delta_mstats

    _cl_mk_sig = {}
    if not _stats_src.empty and 'mk_trend' in _stats_src.columns:
        _sub = _stats_src[
            (_stats_src['zone']   == _zone_flt) &
            (_stats_src['period'] == period)
        ]
        for _, _sr in _sub.iterrows():
            _num   = int(str(_sr['city']).replace('Cluster25_', '').lstrip('0') or '0')
            _match = [l for l in cl_mat.index if l.startswith(f'C{_num:02d} [')]
            if _match:
                _trend = str(_sr.get('mk_trend', ''))
                _pval  = float(_sr.get('mk_p', 1.0))
                _dir   = _trend if (_pval < 0.05 and _trend in ('increasing', 'decreasing')) else ''
                _cl_mk_sig[_match[0]] = (_dir, _pval)

    out_path = str(CL_HM_DIR / f'LST_{fname}_Cluster_Heatmap.svg')
    create_single_page_heatmap(
        data=full_mat,
        mk_sig=_cl_mk_sig,
        lst_time=lst_label,
        time_dim=time_dim,
        fig_w=fw, fig_h=fh,
        output_path=out_path,
        pre_sorted=True,
        separator_after_rows=[len(cl_mat) - 1],   # line before national composite
        cluster_lookup={},                 # rows are already cluster labels
    )
    print(f'  ✓ {fname}: {len(cl_mat)} cluster rows + National Composite')

print(f'\nAll cluster heatmaps → {CL_HM_DIR}')


---
<a id="nb03-s10"></a>
## Section 10 — Stakeholder Validation

I set the MODIS trends against what the workshop participants reported, on the same city → cluster
→ national levels:

| Sub-section | Unit | Content |
|-------------|------|---------|
| [§10a — City-Level Validation](#nb03-s10a) | Per city | Q61 (10-yr trend) + Q58 (peak month) individual agreement vs UC MK trend (compute) |
| [§10b — Cluster-Level Validation](#nb03-s10b) | Per Cluster25 | Raw individual participant counts: agreed / disagreed per cluster (compute) |
| [§10c — Comprehensive Table](#nb03-s10c) | City · Cluster · National | One stacked table (Q61 · Q60 · Q58) → `Stakeholder_Validation_Comprehensive.csv` |
| [§10d — Perception Panels](#nb03-s10d) | City · Cluster · National | Bar + donut figures (Warming · Diurnal · Peak season) |
I use two questions for the check: **Q61**, *"Have you perceived temperature increases in the last 10
years?"*, against the annual Mann–Kendall trend; and **Q58**, *"During which month or season is the
temperature usually at its peak?"*, against the urban-core hottest month.

> **Agreement.** A participant agrees on Q61 when the city has a significant MODIS day warming trend
> and they answered "Yes", or no significant trend and they answered "No" or "Unsure". They agree on
> Q58 when their season (three months) holds the city's day or night hottest month. Consensus is
> High when both exceed 70 %, Medium when their average reaches 60 %, and Low otherwise.

#### Survey Limitations (apply to all sub-sections)

| Caveat | Impact |
|--------|--------|
| Workshop selection bias | Attendees are climate-aware; interpret as “engaged stakeholder perception” |
| Recency bias | Reliable for last 5–10 years; Q61 more reliable than Q58 for long trends |
| Spatial mismatch | UC LST is city-scale; personal perception is microclimate-specific |
| Temporal lag | Climate change may outpace perception adaptation |

---

> ↑ [Back to Table of Contents](#nb03-toc)

---
<a id="nb03-s10a"></a>
### §10a — City-Level Validation

I compare each city's Q61 and Q58 answers with its urban-core trend and hottest month, and take its
majority Q60 answer (reported, not checked against LST). The results feed the table in
[§10c](#nb03-s10c); the charts are in [§10d](#nb03-s10d).

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
import math

# §10a compute only — feeds the §10c comprehensive table; charts are in §10d.

# ── Load clean workshop attendee data from NB02 §8 ────────────────────────────
# NB02 handles all filtering: off-target cities removed, absorbed city name remaps,
# spelling fixes, and multi-city attendance resolution.
# DO NOT reload from raw survey here — use the NB02-verified output.
_ws_clean_path = (project_root / 'Results' / '02_stakeholder_analysis'
                  / 'Data' / 'workshop_attendees_clean.csv')
if not _ws_clean_path.exists():
    raise FileNotFoundError(
        f'NB02 workshop data not found:\n  {_ws_clean_path}\n'
        'Run NB02 §8 (Export) first to generate this file.'
    )
workshop_study = pd.read_csv(_ws_clean_path)
study_cities  = sorted(workshop_study['matched_city_y'].dropna().unique().tolist())
total_wk_pids = workshop_study['pid'].nunique()

print(f'Workshop attendees (from NB02)  : {total_wk_pids}')
print(f'Cities with survey coverage     : {len(study_cities)}')

uc_ann_stats = df_annual_stats[df_annual_stats['zone'] == 'urban_core'].copy()
uc_mon_stats = df_monthly_stats[df_monthly_stats['zone'] == 'urban_core'].copy()

def _uc_trend(city, period):
    sub = uc_ann_stats[(uc_ann_stats['city'] == city) &
                       (uc_ann_stats['period'] == period)]
    if sub.empty:
        return 'No data', False
    trend = str(sub.iloc[0].get('mk_trend', ''))
    p     = float(sub.iloc[0].get('mk_p', 1.0))
    sig   = (trend == 'increasing' and p < 0.05)
    return ('Significant increase' if sig else 'No significant trend'), sig

def _uc_hot_month(city, period):
    sub = uc_mon_stats[(uc_mon_stats['city'] == city) &
                       (uc_mon_stats['period'] == period)]
    if sub.empty or pd.isna(sub.iloc[0].get('highest_month')):
        return None
    return int(sub.iloc[0]['highest_month'])

_MON_FULL = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
             7:'July',8:'August',9:'September',10:'October',
             11:'November',12:'December'}

_QUARTER_RANGES = {
    'March \u2013 May':           ['March','April','May'],
    'March - May':                 ['March','April','May'],
    'June \u2013 August':         ['June','July','August'],
    'June - August':               ['June','July','August'],
    'September \u2013 November':  ['September','October','November'],
    'September - November':        ['September','October','November'],
    'December \u2013 February':   ['December','January','February'],
    'December - February':         ['December','January','February'],
}

# ── Annual validation ─────────────────────────────────────────────────────────
print('\n' + '='*70)
print('ANNUAL VALIDATION: Q61 vs UC MK trend')
print('='*70)

ann_rows = []
for city in tqdm(study_cities, desc='Annual validation'):
    city_resp = workshop_study[workshop_study['matched_city_y'] == city]
    n_stake   = city_resp['pid'].nunique()
    if n_stake == 0:
        continue
    city_uniq  = city_resp.groupby('pid').first().reset_index()
    n_yes      = int((city_uniq['61_temp_increase_10yrs'] == 'Yes').sum())
    pct_yes    = n_yes / n_stake * 100
    dur_counts = (city_uniq['56_recency_with_city'].value_counts().to_dict()
                  if '56_recency_with_city' in city_uniq.columns else {})

    day_txt,   day_sig   = _uc_trend(city, 'day')
    night_txt, night_sig = _uc_trend(city, 'night')
    overall_agr = round(pct_yes if day_sig else 100 - pct_yes, 1)

    ann_rows.append({
        'city': city, 'n_stakeholders': n_stake,
        'n_perceived_warming': n_yes,
        'pct_perceived_warming': round(pct_yes, 1),
        'overall_agreement': overall_agr,
        'duration_counts': dur_counts,
        'day_significance': day_txt,
        'night_significance': night_txt,
    })

validation_df = pd.DataFrame(ann_rows)
_tot_ann = validation_df['n_stakeholders'].sum()
validation_df['pct_of_total'] = (
    validation_df['n_stakeholders'] / _tot_ann * 100).round(1)
# ── Monthly validation ────────────────────────────────────────────────────────
print('\n' + '='*70)
print('MONTHLY VALIDATION: Q58 vs UC hottest month')
print('='*70)

mon_rows = []
for city in tqdm(study_cities, desc='Monthly validation'):
    city_resp = workshop_study[workshop_study['matched_city_y'] == city]
    n_stake   = city_resp['pid'].nunique()
    if n_stake == 0:
        continue
    city_uniq  = city_resp.groupby('pid').first().reset_index()
    dur_counts = (city_uniq['56_recency_with_city'].value_counts().to_dict()
                  if '56_recency_with_city' in city_uniq.columns else {})

    hot_day   = _MON_FULL.get(_uc_hot_month(city, 'day'),   None)
    hot_night = _MON_FULL.get(_uc_hot_month(city, 'night'), None)

    peak_col = '58_temp_peak_city'
    peak_responses = (city_uniq[peak_col].dropna()
                      if peak_col in city_uniq.columns else pd.Series(dtype=str))
    agree = 0
    most_common = (peak_responses.value_counts().idxmax()
                   if len(peak_responses) > 0 else None)
    for resp in peak_responses:
        mq = _QUARTER_RANGES.get(str(resp).strip(), [])
        if (hot_day in mq if hot_day else False) or \
           (hot_night in mq if hot_night else False):
            agree += 1

    n_resp = len(peak_responses)
    mon_rows.append({
        'city': city, 'n_stakeholders': n_stake,
        'lst_day_hottest':   hot_day   or 'N/A',
        'lst_night_hottest': hot_night or 'N/A',
        'stakeholder_peak_response': most_common or 'N/A',
        'overall_agreement': round(agree / n_resp * 100, 1) if n_resp > 0 else float('nan'),
        'duration_counts': dur_counts,
    })

monthly_validation_df = pd.DataFrame(mon_rows)
_tot_mon = monthly_validation_df['n_stakeholders'].sum()
monthly_validation_df['pct_of_total'] = (
    monthly_validation_df['n_stakeholders'] / _tot_mon * 100).round(1)
# ── Unified summary ───────────────────────────────────────────────────────────
print('\n' + '='*70)
print('UNIFIED VALIDATION SUMMARY')
print('='*70)

unified = validation_df[
    ['city','n_stakeholders','pct_of_total',
     'n_perceived_warming','pct_perceived_warming',
     'overall_agreement','day_significance','night_significance']
].copy().rename(columns={
    'city': 'City', 'n_stakeholders': 'N_Stakeholders',
    'pct_of_total': 'Pct_of_Total',
    'n_perceived_warming': 'N_Said_Yes_Warming',
    'pct_perceived_warming': 'Annual_Warming_Pct',
    'overall_agreement': 'Annual_Agreement(%)',
    'day_significance': 'Day_Trend',
    'night_significance': 'Night_Trend',
})
unified['Stakeholder_Warming_Input'] = (
    unified['N_Said_Yes_Warming'].astype(int).astype(str) + '/' +
    unified['N_Stakeholders'].astype(int).astype(str)
)
mon_merge = monthly_validation_df[
    ['city','overall_agreement','lst_day_hottest',
     'lst_night_hottest','stakeholder_peak_response']
].rename(columns={
    'city': 'City', 'overall_agreement': 'Monthly_Peak_Agreement(%)',
    'lst_day_hottest': 'Day_Hottest_Month',
    'lst_night_hottest': 'Night_Hottest_Month',
    'stakeholder_peak_response': 'Stakeholder_Peak_Response',
})
unified = unified.merge(mon_merge, on='City', how='left')

def _consensus(row):
    a = row.get('Annual_Agreement(%)', float('nan'))
    m = row.get('Monthly_Peak_Agreement(%)', float('nan'))
    if pd.isna(a) or pd.isna(m): return 'Unknown'
    avg = (a + m) / 2
    if a > 70 and m > 70: return 'High'
    if a > 50 and m > 50 and avg > 60: return 'Medium'
    if avg >= 60: return 'Medium'
    return 'Low'

unified['Consensus_Level'] = unified.apply(_consensus, axis=1)
unified = unified[unified['N_Stakeholders'] > 0]
# Diurnal (Q60) majority per city — reported perception (not LST-validated)
_ws_u = workshop_study.groupby('pid').first().reset_index()
_city_diurnal_maj = (_ws_u.dropna(subset=['60_diurnal_severity_heat'])
                     .groupby('matched_city_y')['60_diurnal_severity_heat']
                     .agg(lambda s: s.value_counts().idxmax()))
unified['Diurnal_Majority'] = unified['City'].map(_city_diurnal_maj).fillna('N/A')

# Keep every column in memory (consumed by §10c comprehensive table + §11c master)
unified = unified[[
    'City','N_Stakeholders','Pct_of_Total','N_Said_Yes_Warming',
    'Annual_Warming_Pct','Stakeholder_Warming_Input','Annual_Agreement(%)',
    'Day_Trend','Night_Trend','Diurnal_Majority',
    'Stakeholder_Peak_Response','Monthly_Peak_Agreement(%)',
    'Day_Hottest_Month','Night_Hottest_Month','Consensus_Level',
]]
print('\nConsensus distribution:')
for lvl in ['High','Medium','Low','Unknown']:
    cnt = (unified['Consensus_Level'] == lvl).sum()
    if cnt:
        print(f'  {lvl:8s}: {cnt:2d} ({cnt/len(unified)*100:.1f}%)')
avg_a = unified['Annual_Agreement(%)'].mean()
avg_m = unified['Monthly_Peak_Agreement(%)'].mean()
print(f'\n  Avg annual agreement  : {avg_a:.1f}%')
print(f'  Avg monthly agreement : {avg_m:.1f}%')


---
<a id="nb03-s10b"></a>
### §10b — Cluster-Level Validation

I count agreement per participant, by Cluster25 class — not as a mean of city scores — with the
rules of §10. The seasons are Mar–May, Jun–Aug, Sep–Nov and Dec–Feb, and any month in the chosen
season counts. Per cluster I report the participants who agree and disagree and the share who agree;
the Mann–Kendall reference is the cluster's composite day trend (§7a). The results, with the Q60
majority, feed the table in [§10c](#nb03-s10c); the charts are the cluster panels in [§10d](#nb03-s10d).

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §10b — Cluster-Level Stakeholder Validation ──────────────────────────────
# Raw participant-level aggregation by Cluster25.
# Agreement is per individual, then summed — NOT a weighted mean.
#
# Q61: participant agrees when city has significant MODIS day warming AND they
#      said "Yes"; OR city has no significant trend AND they said "No"/"Unsure"
# Q58: participant agrees when their chosen quarter contains the MODIS hottest month
#
# Sunburst rings (same layout as city-level §10a):
#   Inner  (r=0.50): cluster share of all surveyed participants
#   Middle (r=0.70): agreed (blue) vs disagreed (red) — raw counts
#   Outer  (r=0.90): duration of residence distribution

# ── Step 1: participant-level data with cluster labels ────────────────────────
# workshop_study, _uc_trend, _uc_hot_month, _MON_FULL, _QUARTER_RANGES
# all defined in §10a — available in the same kernel session.

_city_to_cnum_val = CITY_CLUSTER_LOOKUP   # dict: city_name -> Cluster25 int  (from §9a)

_ws_cl = workshop_study.copy()
_ws_cl['cluster_num']   = _ws_cl['matched_city_y'].map(_city_to_cnum_val)
_ws_cl['cluster_label'] = _ws_cl['cluster_num'].apply(
    lambda x: f'C{int(x):02d}' if pd.notna(x) else 'C*')

_ws_uniq = (
    _ws_cl.groupby('pid')
    .agg(
        city          = ('matched_city_y',         'first'),
        cluster_label = ('cluster_label',           'first'),
        warming       = ('61_temp_increase_10yrs',  'first'),
        peak_season   = ('58_temp_peak_city',       'first'),
        diurnal       = ('60_diurnal_severity_heat', 'first'),
        duration      = ('56_recency_with_city',    'first'),
    )
    .reset_index()
)
_ws_uniq['warming']     = _ws_uniq['warming'].fillna("I'd rather not say")
_ws_uniq['peak_season'] = _ws_uniq['peak_season'].fillna("I'd rather not say")
_ws_uniq['diurnal']     = _ws_uniq['diurnal'].fillna("I'd rather not say")

# Pre-compute city-level MODIS signals (functions from §10a)
_cities_ws = _ws_uniq['city'].unique()
_city_day_sig   = {c: _uc_trend(c, 'day')[1]                         for c in _cities_ws}
_city_hot_day   = {c: _MON_FULL.get(_uc_hot_month(c, 'day'),   None) for c in _cities_ws}
_city_hot_night = {c: _MON_FULL.get(_uc_hot_month(c, 'night'), None) for c in _cities_ws}

# ── Step 2: individual Q61 agreement ─────────────────────────────────────────
def _q61_agree(row):
    sig = _city_day_sig.get(row['city'], False)
    w   = row['warming']
    return (w == 'Yes') if sig else (w in ('No', "I'm not sure"))

_ws_uniq['q61_agreed'] = _ws_uniq.apply(_q61_agree, axis=1)

# ── Step 3: individual Q58 agreement ─────────────────────────────────────────
def _q58_agree(row):
    hot_d = _city_hot_day.get(row['city'])
    hot_n = _city_hot_night.get(row['city'])
    mq    = _QUARTER_RANGES.get(str(row['peak_season']).strip(), [])
    return bool((hot_d in mq if hot_d else False) or
                (hot_n in mq if hot_n else False))

_ws_uniq['q58_agreed'] = _ws_uniq.apply(_q58_agree, axis=1)

# ── Step 4: cluster totals ────────────────────────────────────────────────────
_DUR_CATS_CL = [
    'Within the last 5 years', 'Between 5 - 10 years',
    'Between 11 - 15 years',   'Over 15 years',
]

def _agg_cluster(g):
    n_tot  = len(g)
    n_q61  = int(g['q61_agreed'].sum())
    n_q58  = int(g['q58_agreed'].sum())
    dur    = {d: int((g['duration'] == d).sum()) for d in _DUR_CATS_CL}
    rc     = g['peak_season'].value_counts()
    maj_q58 = rc.index[0] if len(rc) > 0 else 'N/A'
    return pd.Series({
        'n_stakeholders':        n_tot,
        'n_q61_agreed':          n_q61,
        'n_q61_disagreed':       n_tot - n_q61,
        'pct_q61_agreed':        round(n_q61 / n_tot * 100, 1) if n_tot > 0 else float('nan'),
        'n_q58_agreed':          n_q58,
        'n_q58_disagreed':       n_tot - n_q58,
        'pct_q58_agreed':        round(n_q58 / n_tot * 100, 1) if n_tot > 0 else float('nan'),
        'duration_counts':       dur,
        'majority_q58_response': maj_q58,
    })

_cl_raw = (
    _ws_uniq.groupby('cluster_label')
    .apply(_agg_cluster)
    .reset_index()
)

# ── Step 5: join cluster MK trend from §7a ────────────────────────────────────
def _get_cl_mk(period):
    if df_all_cl_ann_stats.empty:
        return pd.DataFrame()
    sub = df_all_cl_ann_stats[
        (df_all_cl_ann_stats['zone']   == 'urban_core') &
        (df_all_cl_ann_stats['period'] == period)
    ].copy()
    sub['cluster_label'] = (
        sub['city']
        .str.replace('Cluster25_0', 'C', regex=False)
        .str.replace('Cluster25_',  'C', regex=False)
    )
    return sub[['cluster_label', 'mk_trend', 'mk_p', 'sen_slope_per_decade']]

_mk_day   = _get_cl_mk('day').rename(columns={
    'mk_trend': 'cl_mk_trend_day',   'mk_p': 'cl_mk_p_day',
    'sen_slope_per_decade': 'cl_sen_dec_day'})
_mk_night = _get_cl_mk('night').rename(columns={
    'mk_trend': 'cl_mk_trend_night', 'mk_p': 'cl_mk_p_night',
    'sen_slope_per_decade': 'cl_sen_dec_night'})

_cl_val = _cl_raw.copy()
if not _mk_day.empty:
    _cl_val = _cl_val.merge(_mk_day,   on='cluster_label', how='left')
if not _mk_night.empty:
    _cl_val = _cl_val.merge(_mk_night, on='cluster_label', how='left')

# ── Step 6: consensus level ───────────────────────────────────────────────────
def _cl_consensus(row):
    a = row.get('pct_q61_agreed', float('nan'))
    m = row.get('pct_q58_agreed', float('nan'))
    if pd.isna(a) or pd.isna(m): return 'Unknown'
    if a > 70 and m > 70:        return 'High'
    if (a + m) / 2 >= 60:        return 'Medium'
    return 'Low'

_cl_val['consensus_level'] = _cl_val.apply(_cl_consensus, axis=1)

_QR58 = {
    'March – May':          ['March', 'April', 'May'],
    'March - May':               ['March', 'April', 'May'],
    'June – August':        ['June', 'July', 'August'],
    'June - August':             ['June', 'July', 'August'],
    'September – November': ['September', 'October', 'November'],
    'September - November':      ['September', 'October', 'November'],
    'December – February':  ['December', 'January', 'February'],
    'December - February':       ['December', 'January', 'February'],
}

def _cl_hot_month_from_series(label, zone, period):
    _MNAMES_FL = {
        1: 'January',  2: 'February', 3: 'March',    4: 'April',
        5: 'May',      6: 'June',     7: 'July',      8: 'August',
        9: 'September',10: 'October', 11: 'November', 12: 'December',
    }
    try:
        num = int(label.replace('C', '').replace('*', ''))
    except (ValueError, AttributeError):
        return 'N/A'
    for slug in [f'Cluster25_{num:02d}', f'Cluster25_{num}']:
        if df_all_cl_monthly.empty:
            break
        sub = df_all_cl_monthly[
            (df_all_cl_monthly['city']   == slug) &
            (df_all_cl_monthly['zone']   == zone) &
            (df_all_cl_monthly['period'] == period)
        ]
        if not sub.empty and 'month' in sub.columns and 'median_lst' in sub.columns:
            grp = sub.groupby('month')['median_lst'].mean()
            if not grp.empty:
                return _MNAMES_FL.get(int(grp.idxmax()), 'N/A')
    return 'N/A'

# ── Enrich _cl_val for the comprehensive table (§10c) ─────────────────────────
_cl_val['lst_day_hottest']   = _cl_val['cluster_label'].apply(
    lambda c: _cl_hot_month_from_series(c, 'urban_core', 'day'))
_cl_val['lst_night_hottest'] = _cl_val['cluster_label'].apply(
    lambda c: _cl_hot_month_from_series(c, 'urban_core', 'night'))
_cl_diur_maj = (_ws_uniq.dropna(subset=['diurnal'])
                .groupby('cluster_label')['diurnal']
                .agg(lambda s: s.value_counts().idxmax()))
_cl_val['diurnal_majority'] = _cl_val['cluster_label'].map(_cl_diur_maj).fillna('N/A')
_cl_yes = (_ws_uniq.assign(_yes=(_ws_uniq['warming'] == 'Yes'))
           .groupby('cluster_label')['_yes'].sum())
_cl_val['n_warming_yes'] = _cl_val['cluster_label'].map(_cl_yes).fillna(0).astype(int)
print(f"  ✓ Cluster-level validation computed: {len(_cl_val)} clusters "
      f"(annual + monthly + diurnal). Comprehensive table in §10c.")



---
<a id="nb03-s10c"></a>
### §10c — Comprehensive Validation Table (City · Cluster · National)

I bring the city (§10a) and cluster (§10b) results and a national row into one table, cities first;
the charts are in [§10d](#nb03-s10d).

| Column group | Fields |
|--------------|--------|
| Identity | `Level`, `Unit`, `Cluster25`, `N`, `Pct_of_Total` |
| **Q61 — Annual warming** *(LST-validated)* | `Q61_Warming_Input`, `Q61_Annual_Agreement_%`, `LST_Day_Trend`, `LST_Night_Trend` |
| **Q60 — Diurnal** *(reported perception)* | `Q60_Diurnal_Majority` |
| **Q58 — Peak season** *(LST-validated)* | `Q58_Peak_Response`, `Q58_Monthly_Agreement_%`, `LST_Day_Hottest_Month`, `LST_Night_Hottest_Month` |
| Consensus | `Consensus_Level` |
Q61 and Q58 are scored against urban-core LST; Q60 is the majority answer, not checked against LST.

**Output:** `Stakeholder_Validation_Comprehensive.csv`

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §10c — Comprehensive stakeholder validation table (city · cluster · national)
# One tidy table replacing the former per-level summaries and sunbursts.
# Rows = units (cities → clusters → national). Q61 (annual warming) and Q58
# (peak month) are validated against UC LST; Q60 (diurnal) is REPORTED
# perception, not LST-validated. Charts live only in §10d.
_COLS = ['Level', 'Unit', 'Cluster25', 'N', 'Pct_of_Total',
         'Q61_Warming_Input', 'Q61_Annual_Agreement_%', 'LST_Day_Trend', 'LST_Night_Trend',
         'Q60_Diurnal_Majority',
         'Q58_Peak_Response', 'Q58_Monthly_Agreement_%',
         'LST_Day_Hottest_Month', 'LST_Night_Hottest_Month', 'Consensus_Level']

def _trend_lbl(v):
    return 'Significant increase' if 'increas' in str(v).lower() else 'No significant trend'

_rows = []

# ---- City rows (from unified + monthly_validation_df) ----
for _, r in unified.iterrows():
    _city = r['City']
    _cnum = CITY_CLUSTER_LOOKUP.get(_city)
    _rows.append({
        'Level': 'City', 'Unit': _city,
        'Cluster25': (f'C{int(_cnum):02d}' if pd.notna(_cnum) else 'C*'),
        'N': int(r['N_Stakeholders']), 'Pct_of_Total': r['Pct_of_Total'],
        'Q61_Warming_Input': r['Stakeholder_Warming_Input'],
        'Q61_Annual_Agreement_%': r['Annual_Agreement(%)'],
        'LST_Day_Trend': r['Day_Trend'], 'LST_Night_Trend': r['Night_Trend'],
        'Q60_Diurnal_Majority': r['Diurnal_Majority'],
        'Q58_Peak_Response': r['Stakeholder_Peak_Response'],
        'Q58_Monthly_Agreement_%': r['Monthly_Peak_Agreement(%)'],
        'LST_Day_Hottest_Month': r['Day_Hottest_Month'],
        'LST_Night_Hottest_Month': r['Night_Hottest_Month'],
        'Consensus_Level': r['Consensus_Level'],
    })

# ---- Cluster rows (from _cl_val) ----
_cl_tot = _cl_val['n_stakeholders'].sum()
for _, r in _cl_val.iterrows():
    _rows.append({
        'Level': 'Cluster', 'Unit': r['cluster_label'], 'Cluster25': r['cluster_label'],
        'N': int(r['n_stakeholders']),
        'Pct_of_Total': round(r['n_stakeholders'] / _cl_tot * 100, 1),
        'Q61_Warming_Input': f"{int(r['n_warming_yes'])}/{int(r['n_stakeholders'])}",
        'Q61_Annual_Agreement_%': r['pct_q61_agreed'],
        'LST_Day_Trend': _trend_lbl(r.get('cl_mk_trend_day')),
        'LST_Night_Trend': _trend_lbl(r.get('cl_mk_trend_night')),
        'Q60_Diurnal_Majority': r['diurnal_majority'],
        'Q58_Peak_Response': r['majority_q58_response'],
        'Q58_Monthly_Agreement_%': r['pct_q58_agreed'],
        'LST_Day_Hottest_Month': r['lst_day_hottest'],
        'LST_Night_Hottest_Month': r['lst_night_hottest'],
        'Consensus_Level': r['consensus_level'],
    })

# ---- National row (whole pool) ----
_ws_n = workshop_study.groupby('pid').first().reset_index()
def _maj(col):
    s = _ws_n[col].dropna()
    return s.value_counts().idxmax() if len(s) else 'N/A'
_n_tot   = int(unified['N_Stakeholders'].sum())
_yes_tot = (int(unified['N_Said_Yes_Warming'].sum()) if 'N_Said_Yes_Warming' in unified.columns
            else int((_ws_n['61_temp_increase_10yrs'] == 'Yes').sum()))
_ann_w   = round(float((unified['Annual_Agreement(%)'] * unified['N_Stakeholders']).sum() / _n_tot), 1)
_mon_w   = round(float((monthly_validation_df['overall_agreement'].fillna(0)
                        * monthly_validation_df['n_stakeholders']).sum()
                       / monthly_validation_df['n_stakeholders'].sum()), 1)
_ncity   = len(unified)
_nsig_d  = int((unified['Day_Trend']   == 'Significant increase').sum())
_nsig_n  = int((unified['Night_Trend'] == 'Significant increase').sum())
_dhot    = monthly_validation_df['lst_day_hottest'].dropna()
_nhot    = monthly_validation_df['lst_night_hottest'].dropna()
_rows.append({
    'Level': 'National', 'Unit': 'National (all cities)', 'Cluster25': 'All',
    'N': _n_tot, 'Pct_of_Total': 100.0,
    'Q61_Warming_Input': f'{_yes_tot}/{_n_tot}', 'Q61_Annual_Agreement_%': _ann_w,
    'LST_Day_Trend': f'{_nsig_d}/{_ncity} cities sig.',
    'LST_Night_Trend': f'{_nsig_n}/{_ncity} cities sig.',
    'Q60_Diurnal_Majority': _maj('60_diurnal_severity_heat'),
    'Q58_Peak_Response': _maj('58_temp_peak_city'), 'Q58_Monthly_Agreement_%': _mon_w,
    'LST_Day_Hottest_Month': (_dhot.mode().iloc[0] if len(_dhot) else 'N/A'),
    'LST_Night_Hottest_Month': (_nhot.mode().iloc[0] if len(_nhot) else 'N/A'),
    'Consensus_Level': 'aggregate',
})

validation_comprehensive = pd.DataFrame(_rows)[_COLS]
_city_part = (validation_comprehensive[validation_comprehensive['Level'] == 'City']
              .sort_values(['Cluster25', 'N'], ascending=[True, False]))
_rest_part = validation_comprehensive[validation_comprehensive['Level'] != 'City']
validation_comprehensive = pd.concat([_city_part, _rest_part], ignore_index=True)

_out_csv = RESULTS_03 / 'Stakeholder_Validation_Comprehensive.csv'
validation_comprehensive.to_csv(_out_csv, index=False)
print(f'Comprehensive validation table: {len(validation_comprehensive)} rows '
      f'({(validation_comprehensive["Level"] == "City").sum()} cities + '
      f'{(validation_comprehensive["Level"] == "Cluster").sum()} clusters + 1 national)')
print(f'  \u2713 {_out_csv.name}')
print('  Q61 (annual) & Q58 (monthly) are LST-validated; Q60 (diurnal) is reported perception.')
try:
    from IPython.display import display
    display(validation_comprehensive)
except Exception:
    print(validation_comprehensive.to_string(index=False))


---
<a id="nb03-s10d"></a>
### §10d — Consolidated Perception Panels (Warming · Diurnal · Peak Season)

I draw one figure of three bar-and-donut panels, at city, cluster and national level, all on the same
105 participants:

| Panel | Survey column | Measures |
|-------|---------------|----------|
| **A — Warming** | `61_temp_increase_10yrs` | perceived decadal warming |
| **B — Diurnal severity** | `60_diurnal_severity_heat` | most intense *time of day* |
| **C — Peak month/season** | `58_temp_peak_city` | most intense *time of year* |
I order the seasons Dec–Feb, Mar–May, Jun–Aug, Sep–Nov, and show "I am not sure" and "I'd rather not
say" in grey, as in panels A and B. The sunburst charts of §10a–§10c show the annual and monthly
answers separately.

> ↑ [Back to §10](#nb03-s10)

In [ ]:
# ── §10d — Consolidated perception panels (city / cluster / national) ─────────
# create_perception_chart_with_donuts / create_cluster_perception_chart /
# create_national_perception_donuts imported in §1.

# ── Identical 105-participant base for all three panels ───────────────────────
_perc = (
    workshop_study.groupby('pid').agg(
        city    = ('matched_city_y',          'first'),
        warming = ('61_temp_increase_10yrs',   'first'),
        diurnal = ('60_diurnal_severity_heat', 'first'),
        peak    = ('58_temp_peak_city',        'first'),
    ).reset_index()
)
_perc['warming'] = _perc['warming'].fillna("I'd rather not say")
_perc['diurnal'] = _perc['diurnal'].fillna("I'd rather not say")
_perc['peak']    = (_perc['peak'].astype('object')
                    .str.replace(' - ', ' \u2013 ', regex=False)   # normalise to en-dash
                    .fillna("I'd rather not say"))
_perc['cluster_num'] = _perc['city'].map(CITY_CLUSTER_LOOKUP)

# Helper-expected survey column names
_perc['matched_city_y']           = _perc['city']
_perc['61_temp_increase_10yrs']   = _perc['warming']
_perc['60_diurnal_severity_heat'] = _perc['diurnal']
_perc['58_temp_peak_city']        = _perc['peak']

_warm_total = _perc['warming'].value_counts()
_diur_total = _perc['diurnal'].value_counts()
_peak_total = _perc['peak'].value_counts()

print(f"Consolidated perception base : {len(_perc)} participants, "
      f"{_perc['city'].nunique()} cities, "
      f"{int(_perc['cluster_num'].notna().sum())} on-target participants, "
      f"{_perc['cluster_num'].dropna().nunique()} clusters")

_perc_dir = RESULTS_03 / 'Perception_Panels'
_perc_dir.mkdir(parents=True, exist_ok=True)

# ── CITY level (rows = cities, >=2 participants) ──────────────────────────────
_fig_city, _fig_city_donuts = create_perception_chart_with_donuts(
    unique_resp   = _perc,
    warming_total = _warm_total,
    diurnal_total = _diur_total,
    peak_total    = _peak_total,
    min_participants = 2,
    figsize       = (20, 5.5),
    legend_scale  = 1.1,
)
_fig_city.savefig(_perc_dir / 'Perception_City_3panel.svg', bbox_inches='tight')
_fig_city_donuts.savefig(_perc_dir / 'Perception_City_donuts.svg', bbox_inches='tight')
plt.show()

# ── CLUSTER level (rows = Cluster25) ──────────────────────────────────────────
_pc = _perc[_perc['cluster_num'].notna()].copy()
_pc['cluster_num'] = _pc['cluster_num'].astype(int)
_cl_order = _pc['cluster_num'].value_counts().index.tolist()
_w_ct = pd.crosstab(_pc['cluster_num'], _pc['warming']).reindex(_cl_order)
_d_ct = pd.crosstab(_pc['cluster_num'], _pc['diurnal']).reindex(_cl_order)
_p_ct = pd.crosstab(_pc['cluster_num'], _pc['peak']).reindex(_cl_order)

_fig_cl, _fig_cl_donuts = create_cluster_perception_chart(
    warming_by_cluster = _w_ct,
    diurnal_by_cluster = _d_ct,
    warming_total      = _pc['warming'].value_counts(),
    diurnal_total      = _pc['diurnal'].value_counts(),
    peak_by_cluster    = _p_ct,
    peak_total         = _pc['peak'].value_counts(),
    cluster25_colors   = config.CLUSTER25_COLORS,
    figsize            = (20, 5.5),
    legend_scale       = 1.0,
)
_fig_cl.savefig(_perc_dir / 'Perception_Cluster_3panel.svg', bbox_inches='tight')
_fig_cl_donuts.savefig(_perc_dir / 'Perception_Cluster_donuts.svg', bbox_inches='tight')
plt.show()

# ── NATIONAL level (three large donuts, whole pool) ───────────────────────────
_fig_nat = create_national_perception_donuts(
    warming_total = _warm_total,
    diurnal_total = _diur_total,
    peak_total    = _peak_total,
    legend_scale  = 1.2,
)
_fig_nat.savefig(_perc_dir / 'Perception_National_donuts.svg', bbox_inches='tight')
plt.show()

print(f"  \u2713 Consolidated perception panels (city / cluster / national) \u2192 {_perc_dir}")

---
<a id="nb03-s11"></a>
## Section 11 — Composite Summary Tables

I summarise the whole analysis in three tables (§11a is computed in this section's first cell):

| Sub-section | File | Contents |
|-------------|------|----------|
| [§11a — Hottest Periods](#nb03-s11) | `Stats/Hottest_Periods_Summary.csv` | Per city + cluster: count of years/months above 90th-pct UC LST; most frequent hot calendar months |
| [§11b — SUHII Intensity](#nb03-s11b) | `Stats/SUHII_Intensity_Summary.csv` | Per city + cluster: annual SUHII max / mean / min / std × day and night; trend direction |
| [§11c — Master Composite Table](#nb03-s11c) | `Stats/Master_Composite_Summary.csv` | Full cross-notebook summary: UC / BRA / CR trends + SUHII + hottest periods + stakeholder validation (Q61 + Q58), grouped by Cluster25; single-city clusters shown as merged rows; national aggregate at end |

---

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §11a — Hottest Periods (>90th pct UC LST) per city and cluster ───────────
# Counts years/months where median UC LST exceeds the 90th percentile of that
# entity's own annual / monthly distribution.  Identifies which calendar months
# most frequently appear among the hottest records.

import numpy as np

_MON_A = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
          7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

_hot_rows = []

# ── City level ────────────────────────────────────────────────────────────────
_uc_ann = df_annual[df_annual['zone'] == 'urban_core']
_uc_mon = df_monthly[df_monthly['zone'] == 'urban_core']

for city in tqdm(sorted(_uc_ann['city'].unique()), desc='Hottest periods — cities'):
    row = {'Entity': city, 'Type': 'City',
           'Cluster25': _city_to_cluster.get(city, 'N/A')}
    for period in ['day', 'night']:
        pfx = f'UC_{period.title()}'
        # Annual
        _av = _uc_ann[(_uc_ann['city']==city) & (_uc_ann['period']==period)]['median_lst'].dropna().values
        if len(_av) > 0:
            p90a = np.percentile(_av, 90)
            hot_yr_mask = _uc_ann[(_uc_ann['city']==city) & (_uc_ann['period']==period) &
                                  (_uc_ann['median_lst'] >= p90a)]
            hot_yrs = sorted(hot_yr_mask['year'].astype(int).tolist())
            row[f'{pfx}_Ann_90pct_LST'] = round(float(p90a), 2)
            row[f'{pfx}_N_Hot_Yrs']     = len(hot_yrs)
            row[f'{pfx}_Hot_Yrs']       = ','.join(str(y) for y in hot_yrs)
        # Monthly
        _mv = _uc_mon[(_uc_mon['city']==city) & (_uc_mon['period']==period)]['median_lst'].dropna().values
        if len(_mv) > 0:
            p90m = np.percentile(_mv, 90)
            hot_mon_df = _uc_mon[(_uc_mon['city']==city) & (_uc_mon['period']==period) &
                                  (_uc_mon['median_lst'] >= p90m)]
            freq = hot_mon_df['month'].value_counts().sort_index()
            most_common = freq.idxmax() if not freq.empty else None
            freq_mos = [_MON_A[m] for m, cnt in freq.items() if cnt >= 2]
            row[f'{pfx}_Mon_90pct_LST']      = round(float(p90m), 2)
            row[f'{pfx}_N_Hot_Mon_Records']  = int(len(hot_mon_df))
            row[f'{pfx}_Hottest_Cal_Month']  = _MON_A.get(most_common, 'N/A') if most_common else 'N/A'
            row[f'{pfx}_Freq_Hot_Cal_Months'] = ','.join(freq_mos) if freq_mos else _MON_A.get(most_common, 'N/A')
    _hot_rows.append(row)

# ── Cluster level ─────────────────────────────────────────────────────────────
_uc_cl_ann = df_all_cl_annual[df_all_cl_annual['zone'] == 'urban_core']
_uc_cl_mon = df_all_cl_monthly[df_all_cl_monthly['zone'] == 'urban_core']

for _cid in tqdm(_study_cluster_ids, desc='Hottest periods — clusters'):
    _slug = f'Cluster25_{_cid:02d}'
    row = {'Entity': _slug, 'Type': 'Cluster', 'Cluster25': _cid}
    for period in ['day', 'night']:
        pfx = f'UC_{period.title()}'
        _av = _uc_cl_ann[(_uc_cl_ann['city']==_slug) & (_uc_cl_ann['period']==period)]['median_lst'].dropna().values
        if len(_av) > 0:
            p90a = np.percentile(_av, 90)
            hot_yr_mask = _uc_cl_ann[(_uc_cl_ann['city']==_slug) & (_uc_cl_ann['period']==period) &
                                     (_uc_cl_ann['median_lst'] >= p90a)]
            hot_yrs = sorted(hot_yr_mask['year'].astype(int).tolist())
            row[f'{pfx}_Ann_90pct_LST'] = round(float(p90a), 2)
            row[f'{pfx}_N_Hot_Yrs']     = len(hot_yrs)
            row[f'{pfx}_Hot_Yrs']       = ','.join(str(y) for y in hot_yrs)
        _mv = _uc_cl_mon[(_uc_cl_mon['city']==_slug) & (_uc_cl_mon['period']==period)]['median_lst'].dropna().values
        if len(_mv) > 0:
            p90m = np.percentile(_mv, 90)
            hot_mon_df = _uc_cl_mon[(_uc_cl_mon['city']==_slug) & (_uc_cl_mon['period']==period) &
                                     (_uc_cl_mon['median_lst'] >= p90m)]
            freq = hot_mon_df['month'].value_counts().sort_index()
            most_common = freq.idxmax() if not freq.empty else None
            freq_mos = [_MON_A[m] for m, cnt in freq.items() if cnt >= 2]
            row[f'{pfx}_Mon_90pct_LST']       = round(float(p90m), 2)
            row[f'{pfx}_N_Hot_Mon_Records']   = int(len(hot_mon_df))
            row[f'{pfx}_Hottest_Cal_Month']   = _MON_A.get(most_common, 'N/A') if most_common else 'N/A'
            row[f'{pfx}_Freq_Hot_Cal_Months'] = ','.join(freq_mos) if freq_mos else _MON_A.get(most_common, 'N/A')
    _hot_rows.append(row)

hot_periods_df = pd.DataFrame(_hot_rows)
hot_periods_df.to_csv(STATS_DIR / 'Hottest_Periods_Summary.csv', index=False)
print(f'\nHottest_Periods_Summary.csv  ({len(hot_periods_df)} rows, {len(hot_periods_df.columns)} cols)')
print(hot_periods_df[hot_periods_df['Type']=='City'][
    ['Entity','UC_Day_N_Hot_Yrs','UC_Day_Hottest_Cal_Month',
     'UC_Night_N_Hot_Yrs','UC_Night_Hottest_Cal_Month']
].to_string(index=False))

---
<a id="nb03-s11b"></a>
### §11b — SUHII Intensity

I summarise each city's and cluster's annual SUHII (maximum, mean, minimum, spread) by day and night,
with its trend.

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §11b — SUHII Intensity per city and cluster (annual) ─────────────────────
# Max, mean, min, std of annual SUHII (UC - BRA) × day and night.
# Trend direction and Sen slope from df_delta_annual_stats / df_all_cl_delta_stats.

import numpy as np

_si_rows = []

def _suhii_stat_row(entity, entity_type, cluster_id, delta_df, stats_df):
    row = {'Entity': entity, 'Type': entity_type, 'Cluster25': cluster_id}
    for period in ['day', 'night']:
        pfx = f'SUHII_{period.title()}'
        vals = delta_df[(delta_df['city']==entity) &
                        (delta_df['period']==period)]['delta_lst'].dropna().values
        if len(vals) > 0:
            row[f'{pfx}_Mean'] = round(float(np.mean(vals)), 3)
            row[f'{pfx}_Max']  = round(float(np.max(vals)), 3)
            row[f'{pfx}_Min']  = round(float(np.min(vals)), 3)
            row[f'{pfx}_Std']  = round(float(np.std(vals)), 3)
        sub = stats_df[(stats_df['city']==entity) & (stats_df['period']==period)]
        if not sub.empty:
            r    = sub.iloc[0]
            _mk  = r.get('mk_trend', 'N/A')
            _sen = r.get('sen_slope_per_decade', float('nan'))
            _p   = float(r.get('mk_p', 1.0))
            _sig = '*' if _p < 0.05 else ''
            row[f'{pfx}_Trend']      = _mk
            row[f'{pfx}_Sen(deg/dec)'] = (f'{float(_sen):+.3f}{_sig}'
                                           if not pd.isna(float(_sen)) else 'N/A')
    return row

# Cities
for city in tqdm(sorted(delta_annual['city'].unique()), desc='SUHII intensity — cities'):
    _si_rows.append(_suhii_stat_row(
        city, 'City', _city_to_cluster.get(city, 'N/A'),
        delta_annual, df_delta_annual_stats))

# Clusters
for _cid in tqdm(_study_cluster_ids, desc='SUHII intensity — clusters'):
    _slug = f'Cluster25_{_cid:02d}'
    _si_rows.append(_suhii_stat_row(
        _slug, 'Cluster', _cid,
        df_all_cl_delta_annual, df_all_cl_delta_stats))

# National aggregate (all cities pooled)
_nat_row = {'Entity': 'NATIONAL (all cities)', 'Type': 'National', 'Cluster25': 'All'}
for period in ['day', 'night']:
    pfx  = f'SUHII_{period.title()}'
    vals = delta_annual[delta_annual['period']==period]['delta_lst'].dropna().values
    if len(vals) > 0:
        _nat_row[f'{pfx}_Mean'] = round(float(np.mean(vals)), 3)
        _nat_row[f'{pfx}_Max']  = round(float(np.max(vals)), 3)
        _nat_row[f'{pfx}_Min']  = round(float(np.min(vals)), 3)
        _nat_row[f'{pfx}_Std']  = round(float(np.std(vals)), 3)
    _nat_row[f'{pfx}_Trend']       = 'see §8'
    _nat_row[f'{pfx}_Sen(deg/dec)'] = 'see §8'
_si_rows.append(_nat_row)

suhii_intensity_df = pd.DataFrame(_si_rows)
suhii_intensity_df.to_csv(STATS_DIR / 'SUHII_Intensity_Summary.csv', index=False)
print(f'SUHII_Intensity_Summary.csv  ({len(suhii_intensity_df)} rows, {len(suhii_intensity_df.columns)} cols)')
print(suhii_intensity_df[suhii_intensity_df['Type']=='City'][
    ['Entity','SUHII_Day_Mean','SUHII_Day_Max','SUHII_Day_Trend',
     'SUHII_Night_Mean','SUHII_Night_Max','SUHII_Night_Trend']
].head(12).to_string(index=False))

---
<a id="nb03-s11c"></a>
### §11c — Master Composite Table

I list every city under its Cluster25 class, with cluster rows (`▶`), single-city clusters merged
(`◆`) and the national row at the end. `Stats/Master_Composite_Summary.html` shows it in colour.

> ↑ [Back to Table of Contents](#nb03-toc)

In [ ]:
# ── §11c — Master Composite Summary Table ────────────────────────────────────
# One comprehensive table covering the entire notebook:
#   • UC / BRA / CR trend (MK direction + Sen slope) × day & night
#   • SUHII mean, max, trend × day & night
#   • Hottest calendar month & year (UC, day & night)
#   • Count of years above 90th-pct UC LST
#   • Stakeholder Q61 (annual warming) + Q58 (peak season) + consensus level
#
# Row structure:
#   — City rows grouped by Cluster25
#   — Cluster composite row after each multi-city group  (label: "▶ Cluster N")
#   — Single-city clusters: one merged row              (label: "◆ City / Cluster N")
#   — National aggregate at end                         (label: "■ NATIONAL")
#
# hot_periods_df and suhii_intensity_df must be computed first (§11a / §11b).

import numpy as np

_MON_A2 = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
           7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

# ── Pre-compute look-up dicts ─────────────────────────────────────────────────
def _ann_stat(stats_df, entity, zone, period, col):
    sub = stats_df[(stats_df['city']==entity) &
                   (stats_df['zone']==zone) &
                   (stats_df['period']==period)]
    if sub.empty: return float('nan')
    v = sub.iloc[0].get(col)
    return v if v is not None else float('nan')

def _fmt_mk(stats_df, entity, zone, period):
    sub = stats_df[(stats_df['city']==entity) &
                   (stats_df['zone']==zone) &
                   (stats_df['period']==period)]
    if sub.empty: return 'N/A'
    r  = sub.iloc[0]
    mk = r.get('mk_trend', 'N/A')
    p  = float(r.get('mk_p', 1.0))
    s  = '*' if p < 0.05 else ''
    return f'{mk}{s}' if mk not in ('N/A', None, '') else 'N/A'

def _fmt_sen2(stats_df, entity, zone, period):
    sub = stats_df[(stats_df['city']==entity) &
                   (stats_df['zone']==zone) &
                   (stats_df['period']==period)]
    if sub.empty: return 'N/A'
    r   = sub.iloc[0]
    s   = r.get('sen_slope_per_decade', float('nan'))
    try:
        if pd.isna(float(s)): return 'N/A'
        p   = float(r.get('mk_p', 1.0))
        sig = '*' if p < 0.05 else ''
        return f'{float(s):+.3f}{sig}'
    except: return 'N/A'

def _hot_mo(entity, period, source='city'):
    _sdf = df_monthly_stats if source == 'city' else df_all_cl_mon_stats
    v = _ann_stat(_sdf, entity, 'urban_core', period, 'highest_month')
    try:
        return _MON_A2.get(int(v), 'N/A') if not (isinstance(v, float) and pd.isna(v)) else 'N/A'
    except: return 'N/A'

def _hot_yr(entity, period, source='city'):
    _sdf = df_annual_stats if source == 'city' else df_all_cl_ann_stats
    v = _ann_stat(_sdf, entity, 'urban_core', period, 'highest_year')
    try:
        return int(v) if not (isinstance(v, float) and pd.isna(v)) else 'N/A'
    except: return 'N/A'

def _n_hot_yrs(entity, period):
    if not hot_periods_df.empty and entity in hot_periods_df['Entity'].values:
        r = hot_periods_df[hot_periods_df['Entity']==entity].iloc[0]
        v = r.get(f'UC_{period.title()}_N_Hot_Yrs')
        return int(v) if not (isinstance(v, float) and pd.isna(v)) else 'N/A'
    return 'N/A'

def _suhii_val(entity, period, col):
    if not suhii_intensity_df.empty and entity in suhii_intensity_df['Entity'].values:
        r = suhii_intensity_df[suhii_intensity_df['Entity']==entity].iloc[0]
        v = r.get(f'SUHII_{period.title()}_{col}')
        return v if not (isinstance(v, float) and pd.isna(v)) else 'N/A'
    return 'N/A'

def _delta_mk(entity, period, source='city'):
    _sdf = df_delta_annual_stats if source == 'city' else df_all_cl_delta_stats
    sub = _sdf[(_sdf['city']==entity) & (_sdf['period']==period)]
    if sub.empty: return 'N/A'
    r  = sub.iloc[0]
    mk = r.get('mk_trend', 'N/A')
    p  = float(r.get('mk_p', 1.0))
    s  = '*' if p < 0.05 else ''
    return f'{mk}{s}' if mk not in ('N/A', None, '') else 'N/A'

def _delta_sen(entity, period, source='city'):
    _sdf = df_delta_annual_stats if source == 'city' else df_all_cl_delta_stats
    sub = _sdf[(_sdf['city']==entity) & (_sdf['period']==period)]
    if sub.empty: return 'N/A'
    r = sub.iloc[0]
    s = r.get('sen_slope_per_decade', float('nan'))
    try:
        if pd.isna(float(s)): return 'N/A'
        p   = float(r.get('mk_p', 1.0))
        sig = '*' if p < 0.05 else ''
        return f'{float(s):+.3f}{sig}'
    except: return 'N/A'

# Stakeholder lookup  (city-level only; clusters get 'cluster' in Type)
try:
    _val_lkp = unified.set_index('City') if 'City' in unified.columns else pd.DataFrame()
except NameError:
    _val_lkp = pd.DataFrame()
try:
    _mon_lkp_col = 'city' if 'city' in monthly_validation_df.columns else 'City'
    _mon_lkp = monthly_validation_df.set_index(_mon_lkp_col)
except NameError:
    _mon_lkp = pd.DataFrame()

# Stakeholder counts per city and per cluster (drives the N_Stakeholders column,
# populated for every row type incl. multi-city cluster composites).
try:
    _city_nstk = dict(zip(unified['City'], unified['N_Stakeholders'].astype(int)))
except Exception:
    _city_nstk = {}
_cluster_nstk = {}
for _c, _n in _city_nstk.items():
    _cc = CITY_CLUSTER_LOOKUP.get(_c)
    if _cc is not None:
        _cluster_nstk[int(_cc)] = _cluster_nstk.get(int(_cc), 0) + int(_n)

def _nstk_for(row_type, cluster_id, entity):
    if row_type == 'Cluster':
        try:
            return int(_cluster_nstk.get(int(cluster_id), 0))
        except Exception:
            return 0
    if row_type in ('City', 'City/Cluster'):
        return int(_city_nstk.get(entity, 0))
    return 0

def _stk(entity):
    out = {k: 'N/A' for k in ['Q61_N','Q61_%Yes','Ann_Agr%','Q58_Peak','Mon_Agr%','Consensus']}
    if not _val_lkp.empty and entity in _val_lkp.index:
        r = _val_lkp.loc[entity]
        out['Q61_N']      = int(r.get('N_Stakeholders', 0))
        out['Q61_%Yes']   = round(float(r.get('Annual_Warming_Pct',   float('nan'))), 1)
        out['Ann_Agr%']   = round(float(r.get('Annual_Agreement(%)',  float('nan'))), 1)
        out['Consensus']  = r.get('Consensus_Level', 'N/A')
    if not _mon_lkp.empty and entity in _mon_lkp.index:
        mr = _mon_lkp.loc[entity]
        out['Q58_Peak']   = mr.get('stakeholder_peak_response', 'N/A')
        out['Mon_Agr%']   = round(float(mr.get('overall_agreement', float('nan'))), 1)
    return out


# Cluster-level stakeholder aggregates → populate multi-city COMPOSITE rows
# (from §10b _cl_val), so agreement no longer lives only on the child city rows.
_cl_stk = {}
try:
    for _, _cr in _cl_val.iterrows():
        try:
            _cnum = int(str(_cr['cluster_label']).replace('C', '').replace('*', ''))
        except (ValueError, AttributeError):
            continue
        _nn  = int(_cr['n_stakeholders'])
        _yes = int(_cr.get('n_warming_yes', 0))
        _cl_stk[_cnum] = {
            'Q61_N':     _nn,
            'Q61_%Yes':  round(_yes / _nn * 100, 1) if _nn else 'N/A',
            'Ann_Agr%':  round(float(_cr.get('pct_q61_agreed', float('nan'))), 1),
            'Q58_Peak':  _cr.get('majority_q58_response', 'N/A'),
            'Mon_Agr%':  round(float(_cr.get('pct_q58_agreed', float('nan'))), 1),
            'Consensus': _cr.get('consensus_level', 'N/A'),
        }
except NameError:
    _cl_stk = {}

def _stk_cluster(cluster_id):
    _blank = {k: 'N/A' for k in ['Q61_N', 'Q61_%Yes', 'Ann_Agr%', 'Q58_Peak', 'Mon_Agr%', 'Consensus']}
    try:
        return _cl_stk.get(int(cluster_id), _blank)
    except Exception:
        return _blank


# ── Row builder ───────────────────────────────────────────────────────────────
def _build_row(label, row_type, cluster_id, n_cl, climate, lc,
               entity, stats_src='city'):
    r = {'Entity': label, 'Type': row_type,
         'Cluster25': cluster_id, 'N_Cl_Cities': n_cl,
         'N_Stakeholders': _nstk_for(row_type, cluster_id, entity),
         'Climate_Class': climate, 'Landcover': lc}
    for zone, zl in [('urban_core','UC'), ('bra','BRA'), ('city_region','CR')]:
        for per, pl in [('day','Day'), ('night','Night')]:
            _sdf  = df_annual_stats if stats_src=='city' else df_all_cl_ann_stats
            _msdf = df_monthly_stats if stats_src=='city' else df_all_cl_mon_stats
            r[f'{zl}_{pl}_MK']      = _fmt_mk(  _sdf, entity, zone, per)
            r[f'{zl}_{pl}_Sen(d/d)'] = _fmt_sen2(_sdf, entity, zone, per)
    for per, pl in [('day','Day'), ('night','Night')]:
        r[f'UC_{pl}_Hottest_Mo']     = _hot_mo(entity, per, stats_src)
        r[f'UC_{pl}_Hottest_Yr']     = _hot_yr(entity, per, stats_src)
        r[f'UC_{pl}_N_HotYrs(90p)']  = _n_hot_yrs(entity, per)
        r[f'SUHII_{pl}_Mean']        = _suhii_val(entity, per, 'Mean')
        r[f'SUHII_{pl}_Max']         = _suhii_val(entity, per, 'Max')
        r[f'SUHII_{pl}_Trend']       = _delta_mk(entity, per, stats_src)
        r[f'SUHII_{pl}_Sen(d/d)']    = _delta_sen(entity, per, stats_src)
    sv = _stk(entity) if stats_src == 'city' else _stk_cluster(cluster_id)
    r.update(sv)
    return r


# ── Build rows ────────────────────────────────────────────────────────────────
_master_rows   = []
_cluster_bands = {}   # for HTML: cluster_id → list of row indices in this group

for _cid in _study_cluster_ids:
    _slug     = f'Cluster25_{_cid:02d}'
    _avail    = [c for c in _cluster_to_cities.get(_cid, []) if c in _study_cities]
    _n        = len(_avail)
    try:
        _cm_row   = _cl_meta[_cl_meta['Cluster25'] == _cid].iloc[0]
        _climate  = _cm_row['Climate_Class']
        _lc       = _cm_row['Landcover_Class']
    except Exception:
        _climate = _lc = 'N/A'

    _band_start = len(_master_rows)

    if _n == 1:
        _city = _avail[0]
        _label = f'◆ {_city}  [Cluster {_cid}]'
        _master_rows.append(_build_row(_label, 'City/Cluster', _cid, 1, _climate, _lc,
                                       _city, stats_src='city'))
    else:
        for _city in sorted(_avail):
            _master_rows.append(_build_row(_city, 'City', _cid, _n, _climate, _lc,
                                           _city, stats_src='city'))
        # Cluster composite row
        _cl_label = f'▶ Cluster {_cid} composite  (n={_n})'
        _master_rows.append(_build_row(_cl_label, 'Cluster', _cid, _n, _climate, _lc,
                                       _slug, stats_src='cluster'))

    _cluster_bands[_cid] = (_band_start, len(_master_rows) - 1)

# National aggregate row
_nat_label = '■ NATIONAL  (all study cities)'
_nat_row = {'Entity': _nat_label, 'Type': 'National',
            'Cluster25': 'All', 'N_Cl_Cities': len(_study_cities),
            'N_Stakeholders': int(sum(_city_nstk.values())),
            'Climate_Class': '—', 'Landcover': '—'}
for zone, zl in [('urban_core','UC'), ('bra','BRA'), ('city_region','CR')]:
    for per, pl in [('day','Day'), ('night','Night')]:
        _sdf = df_all_cl_ann_stats  # re-use cluster stats for brevity; national §8 data is in df_comp_ann_stats
        try:
            _mk_vals  = [_fmt_mk(_sdf, f'Cluster25_{c:02d}', zone, per)  for c in _study_cluster_ids]
            _inc = sum(1 for v in _mk_vals if 'increasing' in str(v).lower())
            _nat_row[f'{zl}_{pl}_MK'] = f'{_inc}/{len(_study_cluster_ids)} increasing'
        except Exception:
            _nat_row[f'{zl}_{pl}_MK'] = 'see §8'
        _nat_row[f'{zl}_{pl}_Sen(d/d)'] = 'see §8'
# SUHII national
for per, pl in [('day','Day'), ('night','Night')]:
    _nat_suhii = delta_annual[delta_annual['period']==per]['delta_lst'].dropna()
    if len(_nat_suhii) > 0:
        _nat_row[f'SUHII_{pl}_Mean'] = round(float(_nat_suhii.mean()), 3)
        _nat_row[f'SUHII_{pl}_Max']  = round(float(_nat_suhii.max()),  3)
    _nat_row[f'SUHII_{pl}_Trend']   = 'see §8'
    _nat_row[f'SUHII_{pl}_Sen(d/d)'] = 'see §8'
    _nat_row[f'UC_{pl}_Hottest_Mo']  = 'varies'
    _nat_row[f'UC_{pl}_Hottest_Yr']  = 'varies'
    _nat_row[f'UC_{pl}_N_HotYrs(90p)'] = 'varies'
# Stakeholder national aggregate
try:
    _n_stk_tot = int(unified['N_Stakeholders'].sum())
    _pct_yes   = round(float((unified['Annual_Warming_Pct'] * unified['N_Stakeholders']).sum()
                             / _n_stk_tot), 1)
    _ann_agr   = round(float((unified['Annual_Agreement(%)'] * unified['N_Stakeholders']).sum()
                             / _n_stk_tot), 1)
    _nat_row['Q61_N']     = _n_stk_tot
    _nat_row['Q61_%Yes']  = _pct_yes
    _nat_row['Ann_Agr%']  = _ann_agr
    _nat_row['Q58_Peak']  = 'varies by city'
    _nat_row['Mon_Agr%']  = round(float((monthly_validation_df['overall_agreement']
                                         .fillna(0) * monthly_validation_df['n_stakeholders']).sum()
                                        / monthly_validation_df['n_stakeholders'].sum()), 1)
    _nat_row['Consensus'] = 'aggregate'
except Exception:
    _nat_row.update({k:'N/A' for k in ['Q61_N','Q61_%Yes','Ann_Agr%','Q58_Peak','Mon_Agr%','Consensus']})
_master_rows.append(_nat_row)

master_df = pd.DataFrame(_master_rows)
master_df.to_csv(STATS_DIR / 'Master_Composite_Summary.csv', index=False)

# ── Styled HTML ───────────────────────────────────────────────────────────────
_BAND_COLORS = ['#f8f9fa', '#eef3fb']
_css = '''<style>
  .master-tbl { border-collapse: collapse; font-family: monospace; font-size: 9px; }
  .master-tbl th { background:#2c3e50; color:#fff; padding:3px 6px; border:1px solid #aaa; position:sticky; top:0; }
  .master-tbl td { padding:2px 5px; border:1px solid #ddd; white-space:nowrap; }
  .row-cluster  { background:#d5e8f7; font-weight:bold; }
  .row-single   { background:#e8f4e8; font-weight:bold; }
  .row-national { background:#ffd9a0; font-weight:bold; border-top:2px solid #c0392b; }
  .row-city0    { background:#f8f9fa; }
  .row-city1    { background:#edf1f7; }
</style>'''

_html_rows = []
_band_idx  = 0
for idx, row_data in master_df.iterrows():
    rtype  = row_data.get('Type','')
    if rtype == 'National':
        css_cls = 'row-national'
    elif rtype == 'Cluster':
        css_cls = 'row-cluster'
    elif rtype == 'City/Cluster':
        css_cls = 'row-single'
    else:
        css_cls = f'row-city{_band_idx % 2}'
    cells = ''.join(f'<td>{v}</td>' for v in row_data.values)
    _html_rows.append(f'<tr class="{css_cls}">{cells}</tr>')
    if rtype == 'Cluster':
        _band_idx += 1

_hdr = '<tr>' + ''.join(f'<th>{c}</th>' for c in master_df.columns) + '</tr>'
_html = (f'{_css}<div style="overflow:auto; max-height:90vh;">'
         f'<table class="master-tbl">{_hdr}{"".join(_html_rows)}</table></div>')
_html_path = STATS_DIR / 'Master_Composite_Summary.html'
with open(_html_path, 'w', encoding='utf-8') as _fh:
    _fh.write(_html)

print(f'\nMaster_Composite_Summary.csv   ({len(master_df)} rows, {len(master_df.columns)} cols)')
print(f'Master_Composite_Summary.html  → {_html_path.name}')
print()
_type_counts = master_df['Type'].value_counts()
for t, n in _type_counts.items():
    print(f'  {t:15s}: {n:3d} rows')

---
<a id="nb03-s12"></a>
## Section 12 — Summary & Next Steps

### Outputs

```
Results/03_temporal_analysis/
├── all_cities_temporal_annual.csv          city × zone × period × year + MK + Sen slope + OLS R²
├── all_cities_temporal_monthly.csv         city × zone × period × year × month
├── all_cities_delta_annual.csv             SUHII annual series + MK + Sen slope + OLS R²
├── all_cities_delta_monthly.csv            SUHII monthly series
├── Stakeholder_Validation_Comprehensive.csv §10c: city+cluster+national validation (Q61·Q60·Q58)
├── Perception_Panels/                        §10d: bar+donut figures (Warming·Diurnal·Peak)
│   ├── Perception_City_3panel.svg
│   ├── Perception_Cluster_3panel.svg
│   └── Perception_National_donuts.svg
├── Stats/
│   ├── Hottest_Periods_Summary.csv          per city + cluster: hottest months/years above 90th-pct UC LST
│   ├── Hottest_Periods_Summary.html         styled HTML
│   ├── SUHII_Intensity_Summary.csv          per city + cluster: SUHII max/mean/min/std + trend
│   ├── SUHII_Intensity_Summary.html         styled HTML
│   ├── Master_Composite_Summary.csv         cross-notebook composite: UC/BRA/CR trends, SUHII, hottest, stakeholder validation
│   ├── Master_Composite_Summary.html        styled HTML (cluster-grouped, national row at end)
│   ├── Stats/
│   ├── City_LST_Wide_Summary.csv           one row per city; all zones, periods, metrics
│   ├── City_LST_Wide_Summary.html          gradient-styled HTML for co-author review
│   ├── City_Comparison_Table.html          transposed HTML (cities as columns; for visual scanning only)
│   ├── Cluster25_LST_Wide_Summary.csv      one row per cluster; all zones, periods, metrics
│   ├── Cluster25_LST_Wide_Summary.html     gradient-styled HTML (cluster level)
│   ├── National_Composite/
│   │   └── National_Composite_lst_summary.md
│   └── {city_slug}/
│       └── {city_slug}_lst_summary.md
├── heatmaps/
│   ├── LST_{metric}_{period}_Heatmap_MK.svg          8 standard heatmaps (56 cities, annual)
│   ├── LST_{metric}_{period}_Monthly_Heatmap.svg     8 monthly heatmaps (56 cities)
│   ├── cluster/
│   │   ├── LST_{metric}_{period}_Cluster_Heatmap_MK.svg    cluster annual heatmaps
│   │   └── LST_{metric}_{period}_Cluster_Monthly_Heatmap.svg cluster monthly heatmaps
│   └── publication/
│       └── LST_{metric}_{period}_Pub_Heatmap_MK.svg    8 publication heatmaps (21 rows)
├── National_Composite/
│   └── figures/
│       └── national_composite_temporal_trend.svg
├── Cluster25_Composites/
│   ├── Cluster25_{NN}/
│   │   └── figures/
│   │       └── cluster25_{nn}_{slug}_temporal_trend.svg
│   ├── cluster_annual_lst.csv              cluster × zone × period × year
│   ├── cluster_monthly_lst.csv             cluster × zone × period × month
│   ├── cluster_delta_annual.csv            cluster SUHII annual
│   ├── cluster_delta_monthly.csv           cluster SUHII monthly
│   └── cluster_annual_stats.csv            MK + Sen slope + OLS R² per cluster
└── {city_slug}/
    ├── {city_slug}_monthly_lst_threezone.csv
    ├── {city_slug}_annual_lst.csv
    ├── {city_slug}_monthly_lst.csv
    ├── {city_slug}_delta_annual.csv
    ├── {city_slug}_delta_monthly.csv
    └── figures/
        ├── {city_slug}_temporal_trend.svg
        └── {city_slug}_temporal_trend.pdf
```

---

> ↑ [Back to Table of Contents](#nb03-toc)

---

## 2024–2025 LST Anomaly — Observations & Climate Context

From 2024 to 2025, daytime LST fell in about 85 % of the study cities (most in Benin City, −5.07 °C)
while night-time LST rose in about 70 %. This opposite day and night movement points to a real
change at the surface or in the atmosphere, not to a sensor error: calibration drift would move day
and night the same way.

| Evidence | Satellite calibration drift? | Real atmospheric change? |
|----------|------------------------------|--------------------------|
| Daytime LST: widespread cooling | No — drift affects both periods in the same direction | ✓ Yes |
| Nighttime LST: widespread warming | No — opposite sign to day | ✓ Yes |
| Asymmetric day/night pattern | Rules out calibration error | ✓ Yes |
| `n_images` stable (335–365/yr) | No satellite availability collapse | ✓ Data intact |
| `pct_valid_obs` constant (~0.41–0.46) | No QA filtering regression | ✓ Data intact |
| > 85% of cities show same pattern | Not a localised artifact | ✓ National signal |
| Region | Daytime LST | Nighttime LST | Interpretation |
|--------|-------------|---------------|----------------|
| South / Southwest (Benin, Lagos, Ibadan, Oyo) | ↓↓ strong | ↑ moderate | Robust atmospheric/vegetation forcing |
| Sahel / North (Kano, Katsina, Gamboru) | Variable | Variable | Less coherent signal |

The likely drivers, probably combined, are more cloud (less daytime sunshine, more night-time
longwave trapping, as with a stronger West African Monsoon flow), greener land (more evaporative
cooling by day, canopy warmth at night) and wetter soils (higher thermal inertia, a smaller daily
range). I therefore treat it as a climate observation, not a data problem.

---

## Next → Notebook 04 — Daily LST & Persistent Heat Events

Open **`Notebook/04_temporal_analysis_daily.ipynb`**. Before I open it, I check that:
- [ ] `Results/03_temporal_analysis/all_cities_temporal_annual.csv` exists
- [ ] `Results/03_temporal_analysis/all_cities_delta_annual.csv` exists
- [ ] the per-city `*_temporal_trend.svg` charts exist (§6c)
- [ ] NB01's Earth Engine assets are still valid (source version `NB01_v5_SMOD`)